In [1]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 1 — LOAD ROUND 5 DATA
# ════════════════════════════════════════════════════════════════════════════

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROUND = 5
DAYS = [2, 3, 4]

DATA_DIR = Path("Data/round5")

ALGO_PRODUCTS = [
    # Galaxy Sounds Recorders
    "GALAXY_SOUNDS_DARK_MATTER",
    "GALAXY_SOUNDS_BLACK_HOLES",
    "GALAXY_SOUNDS_PLANETARY_RINGS",
    "GALAXY_SOUNDS_SOLAR_WINDS",
    "GALAXY_SOUNDS_SOLAR_FLAMES",

    # Vertical Sleeping Pods
    "SLEEP_POD_SUEDE",
    "SLEEP_POD_LAMB_WOOL",
    "SLEEP_POD_POLYESTER",
    "SLEEP_POD_NYLON",
    "SLEEP_POD_COTTON",

    # Organic Microchips
    "MICROCHIP_CIRCLE",
    "MICROCHIP_OVAL",
    "MICROCHIP_SQUARE",
    "MICROCHIP_RECTANGLE",
    "MICROCHIP_TRIANGLE",

    # Purification Pebbles
    "PEBBLES_XS",
    "PEBBLES_S",
    "PEBBLES_M",
    "PEBBLES_L",
    "PEBBLES_XL",

    # Domestic Robots
    "ROBOT_VACUUMING",
    "ROBOT_MOPPING",
    "ROBOT_DISHES",
    "ROBOT_LAUNDRY",
    "ROBOT_IRONING",

    # UV-Visors
    "UV_VISOR_YELLOW",
    "UV_VISOR_AMBER",
    "UV_VISOR_ORANGE",
    "UV_VISOR_RED",
    "UV_VISOR_MAGENTA",

    # Instant Translators
    "TRANSLATOR_SPACE_GRAY",
    "TRANSLATOR_ASTRO_BLACK",
    "TRANSLATOR_ECLIPSE_CHARCOAL",
    "TRANSLATOR_GRAPHITE_MIST",
    "TRANSLATOR_VOID_BLUE",

    # Construction Panels
    "PANEL_1X2",
    "PANEL_2X2",
    "PANEL_1X4",
    "PANEL_2X4",
    "PANEL_4X4",

    # Liquid Breath Oxygen Shakes
    "OXYGEN_SHAKE_MORNING_BREATH",
    "OXYGEN_SHAKE_EVENING_BREATH",
    "OXYGEN_SHAKE_MINT",
    "OXYGEN_SHAKE_CHOCOLATE",
    "OXYGEN_SHAKE_GARLIC",

    # Protein Snack Packs
    "SNACKPACK_CHOCOLATE",
    "SNACKPACK_VANILLA",
    "SNACKPACK_PISTACHIO",
    "SNACKPACK_STRAWBERRY",
    "SNACKPACK_RASPBERRY",
]

POSITION_LIMITS = {product: 10 for product in ALGO_PRODUCTS}

prices_parts = []
trades_parts = []

for day in DAYS:
    price_path = DATA_DIR / f"prices_round_{ROUND}_day_{day}.csv"
    trade_path = DATA_DIR / f"trades_round_{ROUND}_day_{day}.csv"

    p = pd.read_csv(price_path, sep=";")
    t = pd.read_csv(trade_path, sep=";")

    p["file_day"] = day
    t["file_day"] = day

    prices_parts.append(p)
    trades_parts.append(t)

prices = pd.concat(prices_parts, ignore_index=True)
trades = pd.concat(trades_parts, ignore_index=True)

# Standardise trade product column name.
if "symbol" in trades.columns and "product" not in trades.columns:
    trades = trades.rename(columns={"symbol": "product"})

# Keep only valid Round 5 algorithmic products.
prices = prices[prices["product"].isin(ALGO_PRODUCTS)].copy()
trades = trades[trades["product"].isin(ALGO_PRODUCTS)].copy()

# Useful global time index across days.
# Assumes timestamp resets each day.
min_day = min(DAYS)
prices["global_ts"] = (prices["file_day"] - min_day) * 1_000_000 + prices["timestamp"]
trades["global_ts"] = (trades["file_day"] - min_day) * 1_000_000 + trades["timestamp"]

prices = prices.sort_values(["product", "global_ts"]).reset_index(drop=True)
trades = trades.sort_values(["product", "global_ts"]).reset_index(drop=True)

# Basic sanity checks.
price_products = sorted(prices["product"].unique())
trade_products = sorted(trades["product"].unique())

missing_in_prices = sorted(set(ALGO_PRODUCTS) - set(price_products))
missing_in_trades = sorted(set(ALGO_PRODUCTS) - set(trade_products))

print("prices shape:", prices.shape)
print("trades shape:", trades.shape)
print()
print("Price products:", price_products)
print("Trade products:", trade_products)
print()
print("Missing in prices:", missing_in_prices)
print("Missing in trades:", missing_in_trades)
print()
print("Price days:", sorted(prices["file_day"].unique()))
print("Trade days:", sorted(trades["file_day"].unique()))
print()
print("Position limits:", POSITION_LIMITS)

display(prices.head())
display(trades.head())

prices shape: (1500000, 19)
trades shape: (35385, 9)

Price products: ['GALAXY_SOUNDS_BLACK_HOLES', 'GALAXY_SOUNDS_DARK_MATTER', 'GALAXY_SOUNDS_PLANETARY_RINGS', 'GALAXY_SOUNDS_SOLAR_FLAMES', 'GALAXY_SOUNDS_SOLAR_WINDS', 'MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_RECTANGLE', 'MICROCHIP_SQUARE', 'MICROCHIP_TRIANGLE', 'OXYGEN_SHAKE_CHOCOLATE', 'OXYGEN_SHAKE_EVENING_BREATH', 'OXYGEN_SHAKE_GARLIC', 'OXYGEN_SHAKE_MINT', 'OXYGEN_SHAKE_MORNING_BREATH', 'PANEL_1X2', 'PANEL_1X4', 'PANEL_2X2', 'PANEL_2X4', 'PANEL_4X4', 'PEBBLES_L', 'PEBBLES_M', 'PEBBLES_S', 'PEBBLES_XL', 'PEBBLES_XS', 'ROBOT_DISHES', 'ROBOT_IRONING', 'ROBOT_LAUNDRY', 'ROBOT_MOPPING', 'ROBOT_VACUUMING', 'SLEEP_POD_COTTON', 'SLEEP_POD_LAMB_WOOL', 'SLEEP_POD_NYLON', 'SLEEP_POD_POLYESTER', 'SLEEP_POD_SUEDE', 'SNACKPACK_CHOCOLATE', 'SNACKPACK_PISTACHIO', 'SNACKPACK_RASPBERRY', 'SNACKPACK_STRAWBERRY', 'SNACKPACK_VANILLA', 'TRANSLATOR_ASTRO_BLACK', 'TRANSLATOR_ECLIPSE_CHARCOAL', 'TRANSLATOR_GRAPHITE_MIST', 'TRANSLATOR_SPACE_GRAY'

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,file_day,global_ts
0,2,0,GALAXY_SOUNDS_BLACK_HOLES,9994,22,9992.0,26.0,NaN,NaN,10006,22,10008.0,26.0,NaN,NaN,10000.0,0.0,2,0
1,2,100,GALAXY_SOUNDS_BLACK_HOLES,10001,18,10000.0,25.0,NaN,NaN,10014,18,10016.0,25.0,NaN,NaN,10007.5,0.0,2,100
2,2,200,GALAXY_SOUNDS_BLACK_HOLES,9996,19,9995.0,31.0,NaN,NaN,10009,19,10011.0,31.0,NaN,NaN,10002.5,0.0,2,200
3,2,300,GALAXY_SOUNDS_BLACK_HOLES,9994,25,9993.0,33.0,NaN,NaN,10007,25,10009.0,33.0,NaN,NaN,10000.5,0.0,2,300
4,2,400,GALAXY_SOUNDS_BLACK_HOLES,9999,14,9997.0,32.0,NaN,NaN,10012,14,10013.0,32.0,NaN,NaN,10005.5,0.0,2,400


,timestamp,buyer,seller,product,currency,price,quantity,file_day,global_ts
0,1700,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9969.0,4,2,1700
1,14500,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9749.0,1,2,14500
2,15100,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9764.0,2,2,15100
3,26500,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9656.0,4,2,26500
4,36400,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9675.0,4,2,36400


In [2]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 2 — PANELS CONFIG + FEATURES
# ════════════════════════════════════════════════════════════════════════════

from pathlib import Path
from itertools import combinations, product as iter_product
import math
import warnings

warnings.filterwarnings("ignore")

OUT_DIR = Path("outputs_panels")
OUT_DIR.mkdir(exist_ok=True)

PANEL_PRODUCTS = [
    "PANEL_1X2",
    "PANEL_2X2",
    "PANEL_1X4",
    "PANEL_2X4",
    "PANEL_4X4",
]

PANEL_META = pd.DataFrame([
    {"product": "PANEL_1X2", "w": 1, "h": 2},
    {"product": "PANEL_2X2", "w": 2, "h": 2},
    {"product": "PANEL_1X4", "w": 1, "h": 4},
    {"product": "PANEL_2X4", "w": 2, "h": 4},
    {"product": "PANEL_4X4", "w": 4, "h": 4},
])

PANEL_META["area"] = PANEL_META["w"] * PANEL_META["h"]
PANEL_META["perimeter"] = 2 * (PANEL_META["w"] + PANEL_META["h"])
PANEL_META["aspect"] = PANEL_META["h"] / PANEL_META["w"]
PANEL_META["is_square"] = (PANEL_META["w"] == PANEL_META["h"]).astype(int)
PANEL_META["log_area"] = np.log(PANEL_META["area"])

display(PANEL_META)

PAST_HORIZONS = [1, 2, 5, 10, 25, 50, 100, 250, 500]
FUTURE_HORIZONS = [1, 2, 5, 10, 25, 50, 100, 250, 500]
ROLLING_WINDOWS = [100, 250, 500, 1000, 2500]
FLOW_WINDOWS = [500, 1000, 2500, 5000, 10000]

ENTRY_ZS = [0.75, 1.0, 1.5, 2.0]
POS_LIMIT = 10


def add_book_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["mid"] = df["mid_price"]
    df["spread"] = df["ask_price_1"] - df["bid_price_1"]

    for col in [
        "bid_volume_1", "bid_volume_2", "bid_volume_3",
        "ask_volume_1", "ask_volume_2", "ask_volume_3",
    ]:
        df[col] = df[col].fillna(0)

    df["bid_depth_l1"] = df["bid_volume_1"]
    df["ask_depth_l1"] = df["ask_volume_1"]

    df["bid_depth_total"] = df["bid_volume_1"] + df["bid_volume_2"] + df["bid_volume_3"]
    df["ask_depth_total"] = df["ask_volume_1"] + df["ask_volume_2"] + df["ask_volume_3"]
    df["depth_total"] = df["bid_depth_total"] + df["ask_depth_total"]

    df["imbalance_l1"] = (
        (df["bid_depth_l1"] - df["ask_depth_l1"])
        / (df["bid_depth_l1"] + df["ask_depth_l1"]).replace(0, np.nan)
    )

    df["imbalance_total"] = (
        (df["bid_depth_total"] - df["ask_depth_total"])
        / (df["bid_depth_total"] + df["ask_depth_total"]).replace(0, np.nan)
    )

    df["microprice_l1"] = (
        (df["ask_price_1"] * df["bid_depth_l1"] + df["bid_price_1"] * df["ask_depth_l1"])
        / (df["bid_depth_l1"] + df["ask_depth_l1"]).replace(0, np.nan)
    )

    df["microprice_edge"] = df["microprice_l1"] - df["mid"]
    df["microprice_edge_norm"] = df["microprice_edge"] / df["spread"].replace(0, np.nan)

    return df


prices_f = add_book_features(prices)

panel_prices = (
    prices_f[prices_f["product"].isin(PANEL_PRODUCTS)]
    .sort_values(["file_day", "timestamp", "product"])
    .reset_index(drop=True)
)

panel_trades = (
    trades[trades["product"].isin(PANEL_PRODUCTS)]
    .sort_values(["file_day", "timestamp", "product"])
    .reset_index(drop=True)
)

print("panel_prices:", panel_prices.shape)
print("panel_trades:", panel_trades.shape)
print("products:", sorted(panel_prices["product"].unique()))

,product,w,h,area,perimeter,aspect,is_square,log_area
0,PANEL_1X2,1,2,2,6,2.0,0,0.693147
1,PANEL_2X2,2,2,4,8,1.0,1,1.386294
2,PANEL_1X4,1,4,4,10,4.0,0,1.386294
3,PANEL_2X4,2,4,8,12,2.0,0,2.079442
4,PANEL_4X4,4,4,16,16,1.0,1,2.772589


panel_prices: (150000, 31)
panel_trades: (3665, 9)
products: ['PANEL_1X2', 'PANEL_1X4', 'PANEL_2X2', 'PANEL_2X4', 'PANEL_4X4']


In [3]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 3 — WIDE MATRICES + BASIC DIAGNOSTICS
# ════════════════════════════════════════════════════════════════════════════

def make_wide(value_col: str) -> pd.DataFrame:
    return (
        panel_prices
        .pivot_table(
            index=["file_day", "timestamp"],
            columns="product",
            values=value_col,
            aggfunc="last",
        )
        .sort_index()
        .reindex(columns=PANEL_PRODUCTS)
    )

wide = {
    "mid": make_wide("mid"),
    "bid1": make_wide("bid_price_1"),
    "ask1": make_wide("ask_price_1"),
    "bidvol1": make_wide("bid_volume_1"),
    "askvol1": make_wide("ask_volume_1"),
    "spread": make_wide("spread"),
    "imbalance_l1": make_wide("imbalance_l1"),
    "imbalance_total": make_wide("imbalance_total"),
    "microprice_edge": make_wide("microprice_edge"),
    "microprice_edge_norm": make_wide("microprice_edge_norm"),
}

mid_wide = wide["mid"]

diag = (
    panel_prices
    .groupby(["file_day", "product"])
    .agg(
        rows=("timestamp", "count"),
        first_mid=("mid", "first"),
        last_mid=("mid", "last"),
        min_mid=("mid", "min"),
        max_mid=("mid", "max"),
        mid_range=("mid", lambda x: x.max() - x.min()),
        mean_spread=("spread", "mean"),
        median_spread=("spread", "median"),
        max_spread=("spread", "max"),
        mean_depth=("depth_total", "mean"),
        median_depth=("depth_total", "median"),
        std_l1_imb=("imbalance_l1", "std"),
    )
    .reset_index()
)

trade_diag = (
    panel_trades
    .groupby(["file_day", "product"])
    .agg(
        trades=("price", "count"),
        total_qty=("quantity", "sum"),
        mean_trade_price=("price", "mean"),
        min_trade_price=("price", "min"),
        max_trade_price=("price", "max"),
    )
    .reset_index()
)

display(diag)
display(trade_diag)

diag.to_csv(OUT_DIR / "panel_diagnostics.csv", index=False)
trade_diag.to_csv(OUT_DIR / "panel_trade_diagnostics.csv", index=False)

,file_day,product,rows,first_mid,last_mid,min_mid,max_mid,mid_range,mean_spread,median_spread,max_spread,mean_depth,median_depth,std_l1_imb
0,2,PANEL_1X2,10000,10000.0,8431.5,8411.5,10203.0,1791.5,11.8891,12.0,14,99.1411,100.0,0.114716
1,2,PANEL_1X4,10000,10000.0,11065.0,9196.0,11226.0,2030.0,8.8769,9.0,10,59.1043,60.0,0.119055
2,2,PANEL_2X2,10000,10000.0,10111.5,9201.0,10325.5,1124.5,8.7079,9.0,10,59.1043,60.0,0.131468
3,2,PANEL_2X4,10000,10000.0,10738.0,9982.5,11806.5,1824.0,9.4145,10.0,11,59.1043,60.0,0.086386
4,2,PANEL_4X4,10000,10000.0,9678.5,9528.5,10548.0,1019.5,8.8933,9.0,10,59.1043,60.0,0.103444
5,3,PANEL_1X2,10000,8428.5,8853.5,7667.0,8945.0,1278.0,10.7469,11.0,12,99.3298,100.0,0.118394
6,3,PANEL_1X4,10000,11089.0,8717.0,8344.0,11111.0,2767.0,8.5579,9.0,10,59.1236,60.0,0.174003
7,3,PANEL_2X2,10000,10089.5,9206.0,9114.0,10865.5,1751.5,8.9785,9.0,10,59.1236,60.0,0.115175
8,3,PANEL_2X4,10000,10721.0,11459.0,10543.5,11848.0,1304.5,9.7905,10.0,11,59.1236,60.0,0.089290
9,3,PANEL_4X4,10000,9678.0,10251.5,8997.0,10477.5,1480.5,8.4874,9.0,10,59.1236,60.0,0.167562


,file_day,product,trades,total_qty,mean_trade_price,min_trade_price,max_trade_price
0,2,PANEL_1X2,229,562,9225.148472,8522.0,10102.0
1,2,PANEL_1X4,229,562,10061.816594,9266.0,11213.0
2,2,PANEL_2X2,229,562,9823.982533,9222.0,10245.0
3,2,PANEL_2X4,229,562,10671.021834,10013.0,11777.0
4,2,PANEL_4X4,229,562,10039.017467,9605.0,10495.0
5,3,PANEL_1X2,255,649,8299.129412,7662.0,8899.0
6,3,PANEL_1X4,255,649,9589.250980,8365.0,11092.0
7,3,PANEL_2X2,255,649,10155.619608,9139.0,10799.0
8,3,PANEL_2X4,255,649,11210.278431,10589.0,11839.0
9,3,PANEL_4X4,255,649,9559.337255,9015.0,10466.0


In [4]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 4 — UTILITY FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════

def zscore_series(s: pd.Series, window=500, min_periods=100):
    mean = s.rolling(window, min_periods=min_periods).mean()
    std = s.rolling(window, min_periods=min_periods).std()
    return (s - mean) / std.replace(0, np.nan)


def safe_corr(x: pd.Series, y: pd.Series):
    valid = x.notna() & y.notna()
    if valid.sum() < 200:
        return np.nan

    xv = x[valid]
    yv = y[valid]

    if xv.std() == 0 or yv.std() == 0:
        return np.nan

    return xv.corr(yv)


def directional_hit(signal: pd.Series, future_move: pd.Series):
    valid = signal.notna() & future_move.notna()
    if valid.sum() < 200:
        return np.nan

    x = signal[valid]
    y = future_move[valid]

    nonzero = (x != 0) & (y != 0)
    if nonzero.sum() < 50:
        return np.nan

    return (np.sign(x[nonzero]) == np.sign(y[nonzero])).mean()


def directional_edge(signal: pd.Series, future_move: pd.Series):
    valid = signal.notna() & future_move.notna()
    if valid.sum() < 200:
        return np.nan

    x = signal[valid]
    y = future_move[valid]

    s = np.sign(x)
    active = s != 0

    if active.sum() < 50:
        return np.nan

    return (s[active] * y[active]).mean()


def fit_ols_y_on_x(y: pd.Series, x: pd.Series):
    valid = y.notna() & x.notna()
    yv = y[valid].values
    xv = x[valid].values

    if len(yv) < 200 or np.var(xv) == 0:
        return np.nan, np.nan

    beta = np.cov(yv, xv, ddof=0)[0, 1] / np.var(xv)
    alpha = np.mean(yv) - beta * np.mean(xv)

    return alpha, beta


def mean_reversion_stats(s: pd.Series):
    s = s.dropna()

    if len(s) < 300:
        return np.nan, np.nan, np.nan

    lagged = s.shift(1)
    delta = s.diff()
    valid = lagged.notna() & delta.notna()

    x = lagged[valid].values
    y = delta[valid].values

    if len(x) < 300 or np.var(x) == 0:
        return s.autocorr(1), np.nan, np.nan

    slope = np.cov(y, x, ddof=0)[0, 1] / np.var(x)

    if slope < 0:
        half_life = -np.log(2) / slope
    else:
        half_life = np.inf

    return s.autocorr(1), slope, half_life


def infer_trade_flow(panel_prices, panel_trades):
    quote_cols = [
        "file_day", "timestamp", "product",
        "bid_price_1", "ask_price_1", "mid", "spread",
    ]

    tq = panel_trades.merge(
        panel_prices[quote_cols],
        on=["file_day", "timestamp", "product"],
        how="left",
    )

    tq["inferred_side"] = 0
    tq.loc[tq["price"] >= tq["ask_price_1"], "inferred_side"] = 1
    tq.loc[tq["price"] <= tq["bid_price_1"], "inferred_side"] = -1

    tq["signed_qty"] = tq["inferred_side"] * tq["quantity"]

    return tq


trade_q = infer_trade_flow(panel_prices, panel_trades)
display(trade_q.head())

,timestamp,buyer,seller,product,currency,price,quantity,file_day,global_ts,bid_price_1,ask_price_1,mid,spread,inferred_side,signed_qty
0,1700,NaN,NaN,PANEL_1X2,XIRECS,10018.0,4,2,1700,10018,10031,10024.5,13,-1,-4
1,1700,NaN,NaN,PANEL_1X4,XIRECS,9926.0,4,2,1700,9926,9935,9930.5,9,-1,-4
2,1700,NaN,NaN,PANEL_2X2,XIRECS,9934.0,4,2,1700,9934,9943,9938.5,9,-1,-4
3,1700,NaN,NaN,PANEL_2X4,XIRECS,10013.0,4,2,1700,10013,10022,10017.5,9,-1,-4
4,1700,NaN,NaN,PANEL_4X4,XIRECS,10003.0,4,2,1700,10003,10012,10007.5,9,-1,-4


In [5]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 5 — STRUCTURAL GEOMETRY SCANS
# ════════════════════════════════════════════════════════════════════════════

# 5A — area-normalised price behaviour
area_map = PANEL_META.set_index("product")["area"].to_dict()
aspect_map = PANEL_META.set_index("product")["aspect"].to_dict()
is_square_map = PANEL_META.set_index("product")["is_square"].to_dict()

area_norm_rows = []

for day in DAYS:
    dm = mid_wide.loc[day].copy()

    for p in PANEL_PRODUCTS:
        area = area_map[p]
        s = dm[p] / area

        lag1, slope, hl = mean_reversion_stats(s)

        area_norm_rows.append({
            "day": day,
            "product": p,
            "area": area,
            "area_norm_mean": s.mean(),
            "area_norm_std": s.std(),
            "area_norm_range": s.max() - s.min(),
            "lag1": lag1,
            "mr_slope": slope,
            "half_life": hl,
        })

area_norm_df = pd.DataFrame(area_norm_rows)
display(area_norm_df)
area_norm_df.to_csv(OUT_DIR / "area_normalized_scan.csv", index=False)


# 5B — explicit geometry formula combos
FORMULAS = [
    # same-area
    {
        "name": "same_area_2x2_minus_1x4",
        "coefs": {"PANEL_2X2": 1, "PANEL_1X4": -1},
    },

    # same aspect
    {
        "name": "same_aspect_2x4_minus_1x2",
        "coefs": {"PANEL_2X4": 1, "PANEL_1X2": -1},
    },
    {
        "name": "same_aspect_4x4_minus_2x2",
        "coefs": {"PANEL_4X4": 1, "PANEL_2X2": -1},
    },

    # additive / tiling ideas
    {
        "name": "2x_1x2_minus_2x2",
        "coefs": {"PANEL_1X2": 2, "PANEL_2X2": -1},
    },
    {
        "name": "2x_1x2_minus_1x4",
        "coefs": {"PANEL_1X2": 2, "PANEL_1X4": -1},
    },
    {
        "name": "2x_2x2_minus_2x4",
        "coefs": {"PANEL_2X2": 2, "PANEL_2X4": -1},
    },
    {
        "name": "2x_1x4_minus_2x4",
        "coefs": {"PANEL_1X4": 2, "PANEL_2X4": -1},
    },
    {
        "name": "2x_2x4_minus_4x4",
        "coefs": {"PANEL_2X4": 2, "PANEL_4X4": -1},
    },
    {
        "name": "4x_2x2_minus_4x4",
        "coefs": {"PANEL_2X2": 4, "PANEL_4X4": -1},
    },
    {
        "name": "4x_1x4_minus_4x4",
        "coefs": {"PANEL_1X4": 4, "PANEL_4X4": -1},
    },
    {
        "name": "2x4_minus_2x2_plus_1x4",
        "coefs": {"PANEL_2X4": 1, "PANEL_2X2": -1, "PANEL_1X4": -1},
    },
]

formula_rows = []

for day in DAYS:
    dm = mid_wide.loc[day].copy()

    for formula in FORMULAS:
        coefs = formula["coefs"]
        combo = pd.Series(0.0, index=dm.index)

        for p, c in coefs.items():
            combo += c * dm[p]

        lag1, slope, hl = mean_reversion_stats(combo)

        for w in ROLLING_WINDOWS:
            z = zscore_series(combo, window=w, min_periods=max(50, w // 5))

            formula_rows.append({
                "day": day,
                "formula": formula["name"],
                "coefs": str(coefs),
                "window": w,
                "combo_mean": combo.mean(),
                "combo_std": combo.std(),
                "combo_range": combo.max() - combo.min(),
                "lag1": lag1,
                "mr_slope": slope,
                "half_life": hl,
                "z_abs_gt_2": int((z.abs() > 2).sum()),
                "z_abs_gt_3": int((z.abs() > 3).sum()),
            })

formula_scan = pd.DataFrame(formula_rows)

formula_agg = (
    formula_scan
    .groupby(["formula", "coefs", "window"])
    .agg(
        days=("day", "count"),
        avg_std=("combo_std", "mean"),
        avg_range=("combo_range", "mean"),
        avg_lag1=("lag1", "mean"),
        avg_half_life=("half_life", "mean"),
        total_z_abs_gt_2=("z_abs_gt_2", "sum"),
        total_z_abs_gt_3=("z_abs_gt_3", "sum"),
    )
    .reset_index()
)

formula_agg["finite_half_life"] = np.isfinite(formula_agg["avg_half_life"])
formula_agg["formula_score"] = (
    formula_agg["total_z_abs_gt_2"]
    / (1 + formula_agg["avg_std"].abs())
    * formula_agg["finite_half_life"].map({True: 1.0, False: 0.25})
)

formula_agg = formula_agg.sort_values("formula_score", ascending=False)

display(formula_agg.head(50))

formula_scan.to_csv(OUT_DIR / "formula_scan_raw.csv", index=False)
formula_agg.to_csv(OUT_DIR / "formula_scan_agg.csv", index=False)

,day,product,area,area_norm_mean,area_norm_std,area_norm_range,lag1,mr_slope,half_life
0,2,PANEL_1X2,2,4615.800375,214.967964,895.75000,0.999761,-0.000226,3.067558e+03
1,2,PANEL_2X2,4,2451.723013,51.891460,281.12500,0.998871,-0.001065,6.510600e+02
2,2,PANEL_1X4,4,2505.589687,107.623333,507.50000,0.999722,0.000015,inf
3,2,PANEL_2X4,8,1339.192581,56.101055,228.00000,0.999719,-0.000407,1.702367e+03
4,2,PANEL_4X4,16,627.507584,13.797021,63.71875,0.998952,-0.000915,7.574036e+02
5,3,PANEL_1X2,2,4140.463775,146.818447,639.00000,0.999592,-0.000230,3.010311e+03
6,3,PANEL_2X2,4,2541.765438,112.225923,437.87500,0.999749,-0.000024,2.921061e+04
7,3,PANEL_1X4,4,2408.148787,200.954010,691.75000,0.999927,-0.000172,4.027846e+03
8,3,PANEL_2X4,8,1400.865569,42.666105,163.06250,0.999459,-0.000615,1.127116e+03
9,3,PANEL_4X4,16,596.752319,19.953588,92.53125,0.999542,-0.000224,3.096572e+03


,formula,coefs,window,days,avg_std,avg_range,avg_lag1,avg_half_life,total_z_abs_gt_2,total_z_abs_gt_3,finite_half_life,formula_score
46,same_aspect_2x4_minus_1x2,"{'PANEL_2X4': 1, 'PANEL_1X2': -1}",250,3,447.714387,2028.166667,0.998922,1136.295200,4526,372,True,10.086594
45,same_aspect_2x4_minus_1x2,"{'PANEL_2X4': 1, 'PANEL_1X2': -1}",100,3,447.714387,2028.166667,0.998922,1136.295200,4295,264,True,9.571790
43,same_area_2x2_minus_1x4,"{'PANEL_2X2': 1, 'PANEL_1X4': -1}",1000,3,497.287039,2218.166667,0.999092,1481.098115,4745,376,True,9.522624
42,same_area_2x2_minus_1x4,"{'PANEL_2X2': 1, 'PANEL_1X4': -1}",500,3,497.287039,2218.166667,0.999092,1481.098115,4690,383,True,9.412246
41,same_area_2x2_minus_1x4,"{'PANEL_2X2': 1, 'PANEL_1X4': -1}",250,3,497.287039,2218.166667,0.999092,1481.098115,4455,381,True,8.940630
40,same_area_2x2_minus_1x4,"{'PANEL_2X2': 1, 'PANEL_1X4': -1}",100,3,497.287039,2218.166667,0.999092,1481.098115,4333,392,True,8.695791
47,same_aspect_2x4_minus_1x2,"{'PANEL_2X4': 1, 'PANEL_1X2': -1}",500,3,447.714387,2028.166667,0.998922,1136.295200,3844,338,True,8.566697
44,same_area_2x2_minus_1x4,"{'PANEL_2X2': 1, 'PANEL_1X4': -1}",2500,3,497.287039,2218.166667,0.999092,1481.098115,4200,605,True,8.428877
49,same_aspect_2x4_minus_1x2,"{'PANEL_2X4': 1, 'PANEL_1X2': -1}",2500,3,447.714387,2028.166667,0.998922,1136.295200,3552,406,True,7.915949
48,same_aspect_2x4_minus_1x2,"{'PANEL_2X4': 1, 'PANEL_1X2': -1}",1000,3,447.714387,2028.166667,0.998922,1136.295200,3499,345,True,7.797833


In [6]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 6 — INTEGER COMBO SCAN
# This can take a little while.
# ════════════════════════════════════════════════════════════════════════════

def normalise_coefs(coefs):
    coefs = list(coefs)

    if all(c == 0 for c in coefs):
        return None

    if sum(c != 0 for c in coefs) < 2:
        return None

    gcd = 0
    for c in coefs:
        gcd = math.gcd(gcd, abs(c))

    if gcd > 1:
        coefs = [c // gcd for c in coefs]

    first_nonzero = next(c for c in coefs if c != 0)
    if first_nonzero < 0:
        coefs = [-c for c in coefs]

    return tuple(coefs)


coef_values = [-4, -3, -2, -1, 0, 1, 2, 3, 4]
seen = set()
combo_rows = []

for coefs_raw in iter_product(coef_values, repeat=len(PANEL_PRODUCTS)):
    coefs = normalise_coefs(coefs_raw)

    if coefs is None or coefs in seen:
        continue

    seen.add(coefs)

    # Limit absolute gross leg count so we don't only find untradeable huge combos.
    gross_abs = sum(abs(c) for c in coefs)
    max_abs = max(abs(c) for c in coefs)

    if gross_abs > 10:
        continue

    if max_abs > 4:
        continue

    for day in DAYS:
        dm = mid_wide.loc[day].copy()
        coef_arr = np.array(coefs)

        combo = pd.Series(dm.values @ coef_arr, index=dm.index)
        lag1, slope, hl = mean_reversion_stats(combo)

        for w in [250, 500, 1000, 2500]:
            z = zscore_series(combo, window=w, min_periods=max(50, w // 5))

            combo_rows.append({
                "day": day,
                "coefs": str(list(coefs)),
                "gross_abs": gross_abs,
                "max_abs": max_abs,
                "window": w,
                "combo_mean": combo.mean(),
                "combo_std": combo.std(),
                "combo_range": combo.max() - combo.min(),
                "lag1": lag1,
                "mr_slope": slope,
                "half_life": hl,
                "z_abs_gt_2": int((z.abs() > 2).sum()),
                "z_abs_gt_3": int((z.abs() > 3).sum()),
            })

combo_scan = pd.DataFrame(combo_rows)

combo_agg = (
    combo_scan
    .groupby(["coefs", "gross_abs", "max_abs", "window"])
    .agg(
        days=("day", "count"),
        avg_std=("combo_std", "mean"),
        avg_range=("combo_range", "mean"),
        avg_lag1=("lag1", "mean"),
        avg_half_life=("half_life", "mean"),
        total_z_abs_gt_2=("z_abs_gt_2", "sum"),
        total_z_abs_gt_3=("z_abs_gt_3", "sum"),
    )
    .reset_index()
)

combo_agg["finite_half_life"] = np.isfinite(combo_agg["avg_half_life"])
combo_agg["combo_score"] = (
    combo_agg["total_z_abs_gt_2"]
    / (1 + combo_agg["avg_std"].abs())
    * combo_agg["finite_half_life"].map({True: 1.0, False: 0.25})
)

combo_agg = combo_agg.sort_values("combo_score", ascending=False)

display(combo_agg.head(50))

combo_scan.to_csv(OUT_DIR / "integer_combo_scan_raw.csv", index=False)
combo_agg.to_csv(OUT_DIR / "integer_combo_scan_agg.csv", index=False)

,coefs,gross_abs,max_abs,window,days,avg_std,avg_range,avg_lag1,avg_half_life,total_z_abs_gt_2,total_z_abs_gt_3,finite_half_life,combo_score
2394,"[0, 1, 0, 0, 1]",2,1,1000,3,355.168787,1551.333333,0.998899,966.198556,4272,304,True,11.994313
2392,"[0, 1, 0, 0, 1]",2,1,250,3,355.168787,1551.333333,0.998899,966.198556,4222,349,True,11.853930
2393,"[0, 1, 0, 0, 1]",2,1,500,3,355.168787,1551.333333,0.998899,966.198556,4077,385,True,11.446820
18612,"[1, 1, 0, 0, 1]",3,1,250,3,425.418908,1940.333333,0.999191,908.907794,4564,367,True,10.703090
16,"[0, 0, 0, 1, 1]",2,1,250,3,438.378069,2004.333333,0.999323,1392.753453,4625,371,True,10.526242
16176,"[1, 0, 0, 0, 1]",2,1,250,3,468.704098,2050.166667,0.999441,15704.878173,4785,464,True,10.187265
2395,"[0, 1, 0, 0, 1]",2,1,2500,3,355.168787,1551.333333,0.998899,966.198556,3602,197,True,10.113183
16032,"[1, 0, 0, -1, 0]",2,1,250,3,447.714387,2028.166667,0.998922,1136.295200,4526,372,True,10.086594
16178,"[1, 0, 0, 0, 1]",2,1,1000,3,468.704098,2050.166667,0.999441,15704.878173,4653,266,True,9.906237
2428,"[0, 1, 0, 1, 1]",3,1,250,3,456.778711,2111.166667,0.999012,906.871550,4517,283,True,9.867213


In [7]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 7 — PAIR SPREAD SCAN
# ════════════════════════════════════════════════════════════════════════════

pair_rows = []

for day in DAYS:
    dm = mid_wide.loc[day].copy()

    for a, b in combinations(PANEL_PRODUCTS, 2):
        alpha, beta = fit_ols_y_on_x(dm[a], dm[b])
        spread = dm[a] - (alpha + beta * dm[b])

        lag1, slope, hl = mean_reversion_stats(spread)

        for w in ROLLING_WINDOWS:
            z = zscore_series(spread, window=w, min_periods=max(50, w // 5))

            pair_rows.append({
                "day": day,
                "a": a,
                "b": b,
                "window": w,
                "alpha": alpha,
                "beta": beta,
                "spread_std": spread.std(),
                "spread_range": spread.max() - spread.min(),
                "lag1": lag1,
                "mr_slope": slope,
                "half_life": hl,
                "z_abs_gt_2": int((z.abs() > 2).sum()),
                "z_abs_gt_3": int((z.abs() > 3).sum()),
            })

pair_scan = pd.DataFrame(pair_rows)

pair_agg = (
    pair_scan
    .groupby(["a", "b", "window"])
    .agg(
        days=("day", "count"),
        avg_beta=("beta", "mean"),
        beta_std=("beta", "std"),
        avg_spread_std=("spread_std", "mean"),
        avg_spread_range=("spread_range", "mean"),
        avg_lag1=("lag1", "mean"),
        avg_half_life=("half_life", "mean"),
        total_z_abs_gt_2=("z_abs_gt_2", "sum"),
        total_z_abs_gt_3=("z_abs_gt_3", "sum"),
    )
    .reset_index()
)

pair_agg["finite_half_life"] = np.isfinite(pair_agg["avg_half_life"])
pair_agg["pair_score"] = (
    pair_agg["total_z_abs_gt_2"]
    / (1 + pair_agg["avg_spread_std"].abs())
    * pair_agg["finite_half_life"].map({True: 1.0, False: 0.25})
)

pair_agg = pair_agg.sort_values("pair_score", ascending=False)

display(pair_agg.head(50))

pair_scan.to_csv(OUT_DIR / "pair_scan_raw.csv", index=False)
pair_agg.to_csv(OUT_DIR / "pair_scan_agg.csv", index=False)

,a,b,window,days,avg_beta,beta_std,avg_spread_std,avg_spread_range,avg_lag1,avg_half_life,total_z_abs_gt_2,total_z_abs_gt_3,finite_half_life,pair_score
41,PANEL_2X2,PANEL_4X4,250,3,-0.520771,0.505677,227.950985,1113.480871,0.998749,6.134002e+02,4416,398,True,19.287971
40,PANEL_2X2,PANEL_4X4,100,3,-0.520771,0.505677,227.950985,1113.480871,0.998749,6.134002e+02,4309,367,True,18.820622
42,PANEL_2X2,PANEL_4X4,500,3,-0.520771,0.505677,227.950985,1113.480871,0.998749,6.134002e+02,4132,409,True,18.047531
44,PANEL_2X2,PANEL_4X4,2500,3,-0.520771,0.505677,227.950985,1113.480871,0.998749,6.134002e+02,4073,69,True,17.789834
12,PANEL_1X2,PANEL_2X4,500,3,0.205034,0.709289,268.703686,1321.547888,0.998935,1.165570e+03,4766,535,True,17.671245
43,PANEL_2X2,PANEL_4X4,1000,3,-0.520771,0.505677,227.950985,1113.480871,0.998749,6.134002e+02,4022,259,True,17.567079
39,PANEL_2X2,PANEL_2X4,2500,3,0.125856,0.164870,284.838513,1286.445216,0.999222,1.610503e+04,4842,247,True,16.939635
11,PANEL_1X2,PANEL_2X4,250,3,0.205034,0.709289,268.703686,1321.547888,0.998935,1.165570e+03,4495,367,True,16.666439
16,PANEL_1X2,PANEL_4X4,250,3,0.249083,0.747198,272.039817,1343.854504,0.998960,7.610229e+02,4507,421,True,16.506750
15,PANEL_1X2,PANEL_4X4,100,3,0.249083,0.747198,272.039817,1343.854504,0.998960,7.610229e+02,4468,325,True,16.363914


In [8]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 8 — GEOMETRY RESIDUAL MODEL
# ════════════════════════════════════════════════════════════════════════════

def build_geometry_residuals():
    """
    At each timestamp, fit:
        mid ~ intercept + log_area + aspect + is_square

    Then record each product's residual.
    This is a relative-value signal across the panel family.
    """
    meta = PANEL_META.set_index("product").reindex(PANEL_PRODUCTS)

    X = np.column_stack([
        np.ones(len(PANEL_PRODUCTS)),
        meta["log_area"].values,
        meta["aspect"].values,
        meta["is_square"].values,
    ])

    residual_frames = []

    for day in DAYS:
        dm = mid_wide.loc[day].copy()

        residuals = []

        for ts, row in dm.iterrows():
            y = row.values.astype(float)

            # Least squares cross-sectional fit.
            beta, *_ = np.linalg.lstsq(X, y, rcond=None)
            fitted = X @ beta
            resid = y - fitted

            residuals.append(resid)

        resid_df = pd.DataFrame(
            residuals,
            index=dm.index,
            columns=PANEL_PRODUCTS,
        )

        resid_df["file_day"] = day
        resid_df["timestamp"] = resid_df.index

        residual_frames.append(resid_df.reset_index(drop=True))

    out = pd.concat(residual_frames, ignore_index=True)
    return out


geometry_resid = build_geometry_residuals()

display(geometry_resid.head())

geometry_rows = []

for day in DAYS:
    day_resid = geometry_resid[geometry_resid["file_day"] == day].set_index("timestamp")

    for p in PANEL_PRODUCTS:
        r = day_resid[p]
        lag1, slope, hl = mean_reversion_stats(r)

        for w in ROLLING_WINDOWS:
            z = zscore_series(r, window=w, min_periods=max(50, w // 5))

            geometry_rows.append({
                "day": day,
                "product": p,
                "window": w,
                "resid_std": r.std(),
                "resid_range": r.max() - r.min(),
                "lag1": lag1,
                "mr_slope": slope,
                "half_life": hl,
                "z_abs_gt_2": int((z.abs() > 2).sum()),
                "z_abs_gt_3": int((z.abs() > 3).sum()),
            })

geometry_scan = pd.DataFrame(geometry_rows)

geometry_agg = (
    geometry_scan
    .groupby(["product", "window"])
    .agg(
        days=("day", "count"),
        avg_resid_std=("resid_std", "mean"),
        avg_resid_range=("resid_range", "mean"),
        avg_lag1=("lag1", "mean"),
        avg_half_life=("half_life", "mean"),
        total_z_abs_gt_2=("z_abs_gt_2", "sum"),
        total_z_abs_gt_3=("z_abs_gt_3", "sum"),
    )
    .reset_index()
)

geometry_agg["finite_half_life"] = np.isfinite(geometry_agg["avg_half_life"])
geometry_agg["geometry_score"] = (
    geometry_agg["total_z_abs_gt_2"]
    / (1 + geometry_agg["avg_resid_std"].abs())
    * geometry_agg["finite_half_life"].map({True: 1.0, False: 0.25})
)

geometry_agg = geometry_agg.sort_values("geometry_score", ascending=False)

display(geometry_agg.head(50))

geometry_resid.to_csv(OUT_DIR / "geometry_residuals.csv", index=False)
geometry_scan.to_csv(OUT_DIR / "geometry_residual_scan_raw.csv", index=False)
geometry_agg.to_csv(OUT_DIR / "geometry_residual_scan_agg.csv", index=False)

,PANEL_1X2,PANEL_2X2,PANEL_1X4,PANEL_2X4,PANEL_4X4,file_day,timestamp
0,1.455192e-11,9.094947e-12,1.091394e-11,9.094947e-12,5.456968e-12,2,0
1,6.375000e+00,-6.375000e+00,7.275958e-12,-6.375000e+00,6.375000e+00,2,100
2,7.250000e+00,-7.250000e+00,1.455192e-11,-7.250000e+00,7.250000e+00,2,200
3,1.387500e+01,-1.387500e+01,7.275958e-12,-1.387500e+01,1.387500e+01,2,300
4,1.300000e+01,-1.300000e+01,7.275958e-12,-1.300000e+01,1.300000e+01,2,400


,product,window,days,avg_resid_std,avg_resid_range,avg_lag1,avg_half_life,total_z_abs_gt_2,total_z_abs_gt_3,finite_half_life,geometry_score
8,PANEL_1X4,1000,3,3.019127e-12,2.243420e-11,0.011954,0.701731,1485,82,True,1485.000000
9,PANEL_1X4,2500,3,3.019127e-12,2.243420e-11,0.011954,0.701731,1451,74,True,1451.000000
7,PANEL_1X4,500,3,3.019127e-12,2.243420e-11,0.011954,0.701731,1407,77,True,1407.000000
6,PANEL_1X4,250,3,3.019127e-12,2.243420e-11,0.011954,0.701731,1396,77,True,1396.000000
5,PANEL_1X4,100,3,3.019127e-12,2.243420e-11,0.011954,0.701731,1318,62,True,1318.000000
4,PANEL_1X2,2500,3,2.138566e+02,8.961250e+02,0.999697,9334.254952,4970,411,True,23.131712
19,PANEL_2X4,2500,3,2.138566e+02,8.961250e+02,0.999697,9334.254952,4970,411,True,23.131712
14,PANEL_2X2,2500,3,2.138566e+02,8.961250e+02,0.999697,9334.254952,4970,411,True,23.131712
24,PANEL_4X4,2500,3,2.138566e+02,8.961250e+02,0.999697,9334.254952,4970,411,True,23.131712
1,PANEL_1X2,250,3,2.138566e+02,8.961250e+02,0.999697,9334.254952,4598,407,True,21.400324


In [9]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 9 — SINGLE-PRODUCT SIGNAL SCAN
# ════════════════════════════════════════════════════════════════════════════

def get_product_day_frame(product_name, day):
    p = (
        panel_prices[
            (panel_prices["product"] == product_name)
            & (panel_prices["file_day"] == day)
        ]
        .sort_values("timestamp")
        .copy()
        .reset_index(drop=True)
    )

    tq = trade_q[
        (trade_q["product"] == product_name)
        & (trade_q["file_day"] == day)
    ].copy()

    flow = (
        tq.groupby("timestamp")
        .agg(
            signed_qty=("signed_qty", "sum"),
            gross_qty=("quantity", "sum"),
        )
        .reindex(p["timestamp"])
        .fillna(0)
        .reset_index(drop=True)
    )

    p["signed_qty"] = flow["signed_qty"]
    p["gross_qty"] = flow["gross_qty"]

    return p


def make_single_signals(product_name, day):
    p = get_product_day_frame(product_name, day)
    sigs = pd.DataFrame(index=p.index)

    # Book signals
    sigs["imbalance_l1"] = p["imbalance_l1"]
    sigs["imbalance_total"] = p["imbalance_total"]
    sigs["microprice_edge"] = p["microprice_edge"]
    sigs["microprice_edge_norm"] = p["microprice_edge_norm"]

    # Past move momentum / reversion
    for h in PAST_HORIZONS:
        move = p["mid"].diff(h)
        sigs[f"mom_ret_{h}"] = move
        sigs[f"rev_ret_{h}"] = -move

    # Rolling z-score price mean-reversion / breakout
    for w in ROLLING_WINDOWS:
        z = zscore_series(p["mid"], window=w, min_periods=max(50, w // 5))
        sigs[f"meanrev_z_{w}"] = -z
        sigs[f"breakout_z_{w}"] = z

    # Spread regime
    for w in ROLLING_WINDOWS:
        spread_z = zscore_series(p["spread"], window=w, min_periods=max(50, w // 5))
        sigs[f"spread_z_{w}"] = spread_z

    # Trade-flow
    for fw in FLOW_WINDOWS:
        grid_w = max(1, fw // 100)
        signed_roll = p["signed_qty"].rolling(grid_w, min_periods=1).sum()
        gross_roll = p["gross_qty"].rolling(grid_w, min_periods=1).sum()

        sigs[f"flow_imb_{fw}"] = (
            signed_roll / gross_roll.replace(0, np.nan)
        ).fillna(0)

    return p, sigs


single_rows = []

for day in DAYS:
    for p_name in PANEL_PRODUCTS:
        p, sigs = make_single_signals(p_name, day)

        for sig_name in sigs.columns:
            sig = sigs[sig_name]

            for fh in FUTURE_HORIZONS:
                future_move = p["mid"].shift(-fh) - p["mid"]

                single_rows.append({
                    "day": day,
                    "product": p_name,
                    "signal": sig_name,
                    "future_h": fh,
                    "corr": safe_corr(sig, future_move),
                    "hit_rate": directional_hit(sig, future_move),
                    "directional_edge": directional_edge(sig, future_move),
                    "n": int((sig.notna() & future_move.notna()).sum()),
                })

single_raw = pd.DataFrame(single_rows)

single_agg = (
    single_raw
    .groupby(["product", "signal", "future_h"])
    .agg(
        days=("day", "count"),
        avg_corr=("corr", "mean"),
        min_corr=("corr", "min"),
        max_corr=("corr", "max"),
        avg_hit=("hit_rate", "mean"),
        avg_edge=("directional_edge", "mean"),
        min_edge=("directional_edge", "min"),
        max_edge=("directional_edge", "max"),
        avg_n=("n", "mean"),
    )
    .reset_index()
)

single_agg["same_positive_corr"] = single_agg["min_corr"] > 0
single_agg["same_negative_corr"] = single_agg["max_corr"] < 0
single_agg["same_positive_edge"] = single_agg["min_edge"] > 0
single_agg["same_negative_edge"] = single_agg["max_edge"] < 0
single_agg["abs_avg_corr"] = single_agg["avg_corr"].abs()

single_agg["single_score"] = (
    single_agg["abs_avg_corr"].fillna(0) * 100
    + (single_agg["avg_hit"].fillna(0.5) - 0.5).abs() * 50
    + single_agg["avg_edge"].abs().fillna(0)
)

single_agg.loc[
    single_agg["same_positive_corr"] | single_agg["same_negative_corr"],
    "single_score"
] *= 1.5

single_agg.loc[
    single_agg["same_positive_edge"] | single_agg["same_negative_edge"],
    "single_score"
] *= 1.5

single_agg = single_agg.sort_values("single_score", ascending=False)

display(single_agg.head(80))

single_raw.to_csv(OUT_DIR / "single_signal_scan_raw.csv", index=False)
single_agg.to_csv(OUT_DIR / "single_signal_scan_agg.csv", index=False)

,product,signal,future_h,days,avg_corr,min_corr,max_corr,avg_hit,avg_edge,min_edge,max_edge,avg_n,same_positive_corr,same_negative_corr,same_positive_edge,same_negative_edge,abs_avg_corr,single_score
1169,PANEL_2X4,breakout_z_2500,500,3,-0.270496,-0.396724,-0.178947,0.459511,-23.738696,-31.429952,-15.986613,9001.0,False,True,False,True,0.270496,118.828607
1277,PANEL_2X4,meanrev_z_2500,500,3,0.270496,0.178947,0.396724,0.540489,23.738696,15.986613,31.429952,9001.0,True,False,True,False,0.270496,118.828607
503,PANEL_1X4,meanrev_z_1000,500,3,-0.093264,-0.230378,-0.017042,0.499751,-30.015758,-75.337276,-6.193602,9301.0,False,True,False,True,0.093264,88.547739
395,PANEL_1X4,breakout_z_1000,500,3,0.093264,0.017042,0.230378,0.500249,30.015758,6.193602,75.337276,9301.0,True,False,True,False,0.093264,88.547739
683,PANEL_1X4,rev_ret_250,500,3,-0.093413,-0.237819,0.025590,0.452651,-43.890826,-81.308800,-6.587037,9250.0,False,False,False,True,0.093413,83.399366
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1889,PANEL_4X4,spread_z_500,500,3,-0.002528,-0.009080,0.004315,0.472417,-22.108407,-48.417004,-0.067386,9401.0,False,False,False,True,0.002528,35.610510
1221,PANEL_2X4,flow_imb_5000,100,3,-0.059656,-0.071113,-0.051459,0.470122,-8.258013,-11.833754,-5.274697,9900.0,False,True,False,True,0.059656,35.364313
1203,PANEL_2X4,flow_imb_2500,100,3,-0.047621,-0.059881,-0.041018,0.463968,-8.972424,-12.619563,-6.024905,9900.0,False,True,False,True,0.047621,34.956209
52,PANEL_1X2,flow_imb_1000,250,3,0.035249,0.015237,0.048674,0.520735,10.965390,2.771829,17.560158,9750.0,True,False,True,False,0.035249,34.935936


In [10]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 10 — LEAD-LAG SCAN
# ════════════════════════════════════════════════════════════════════════════

leadlag_rows = []

for day in DAYS:
    dm = mid_wide.loc[day].copy()

    for leader in PANEL_PRODUCTS:
        for follower in PANEL_PRODUCTS:
            if leader == follower:
                continue

            for ph in PAST_HORIZONS:
                leader_move = dm[leader].diff(ph)

                for fh in FUTURE_HORIZONS:
                    follower_future = dm[follower].shift(-fh) - dm[follower]

                    leadlag_rows.append({
                        "day": day,
                        "leader": leader,
                        "follower": follower,
                        "past_h": ph,
                        "future_h": fh,
                        "corr": safe_corr(leader_move, follower_future),
                        "hit_rate": directional_hit(leader_move, follower_future),
                        "directional_edge": directional_edge(leader_move, follower_future),
                    })

leadlag_raw = pd.DataFrame(leadlag_rows)

leadlag_agg = (
    leadlag_raw
    .groupby(["leader", "follower", "past_h", "future_h"])
    .agg(
        days=("day", "count"),
        avg_corr=("corr", "mean"),
        min_corr=("corr", "min"),
        max_corr=("corr", "max"),
        avg_hit=("hit_rate", "mean"),
        avg_edge=("directional_edge", "mean"),
        min_edge=("directional_edge", "min"),
        max_edge=("directional_edge", "max"),
    )
    .reset_index()
)

leadlag_agg["same_positive_corr"] = leadlag_agg["min_corr"] > 0
leadlag_agg["same_negative_corr"] = leadlag_agg["max_corr"] < 0
leadlag_agg["same_positive_edge"] = leadlag_agg["min_edge"] > 0
leadlag_agg["same_negative_edge"] = leadlag_agg["max_edge"] < 0
leadlag_agg["abs_avg_corr"] = leadlag_agg["avg_corr"].abs()

leadlag_agg["leadlag_score"] = (
    leadlag_agg["abs_avg_corr"].fillna(0) * 100
    + (leadlag_agg["avg_hit"].fillna(0.5) - 0.5).abs() * 50
    + leadlag_agg["avg_edge"].abs().fillna(0)
)

leadlag_agg.loc[
    leadlag_agg["same_positive_corr"] | leadlag_agg["same_negative_corr"],
    "leadlag_score"
] *= 1.5

leadlag_agg.loc[
    leadlag_agg["same_positive_edge"] | leadlag_agg["same_negative_edge"],
    "leadlag_score"
] *= 1.5

leadlag_agg = leadlag_agg.sort_values("leadlag_score", ascending=False)

display(leadlag_agg.head(80))

leadlag_raw.to_csv(OUT_DIR / "leadlag_scan_raw.csv", index=False)
leadlag_agg.to_csv(OUT_DIR / "leadlag_scan_agg.csv", index=False)

,leader,follower,past_h,future_h,days,avg_corr,min_corr,max_corr,avg_hit,avg_edge,min_edge,max_edge,same_positive_corr,same_negative_corr,same_positive_edge,same_negative_edge,abs_avg_corr,leadlag_score
1214,PANEL_2X4,PANEL_2X2,500,500,3,0.246028,0.164503,0.343485,0.591630,56.469381,24.275245,95.706609,True,False,True,False,0.246028,192.720790
161,PANEL_1X2,PANEL_2X2,500,500,3,0.261828,0.109892,0.367490,0.582068,48.339996,18.111525,75.771271,True,False,True,False,0.261828,176.908890
1538,PANEL_4X4,PANEL_2X2,500,500,3,-0.206448,-0.314318,-0.031663,0.447462,-48.770864,-75.583584,-4.261934,False,True,False,True,0.206448,162.095715
1537,PANEL_4X4,PANEL_2X2,500,250,3,-0.180506,-0.291842,-0.040032,0.411763,-34.552660,-53.167641,-10.432337,False,True,False,True,0.180506,128.283908
152,PANEL_1X2,PANEL_2X2,250,500,3,0.185175,0.104453,0.265678,0.567531,32.892437,12.081491,52.293665,True,False,True,False,0.185175,123.269705
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1601,PANEL_4X4,PANEL_2X4,100,500,3,-0.056833,-0.164981,0.127859,0.450091,-16.885888,-37.234503,17.855719,False,False,False,False,0.056833,25.064656
134,PANEL_1X2,PANEL_2X2,50,500,3,0.061590,0.001943,0.114403,0.521159,9.470158,-1.166986,17.521130,True,False,False,False,0.061590,25.030607
1445,PANEL_4X4,PANEL_1X4,250,50,3,-0.049571,-0.092458,-0.015087,0.464626,-4.377095,-6.989368,-0.560242,False,True,False,True,0.049571,24.981598
1122,PANEL_2X4,PANEL_1X4,250,100,3,0.096453,0.023012,0.223605,0.524738,5.687682,-4.610327,15.869784,True,False,False,False,0.096453,24.854882


In [11]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 11 — EXECUTION BACKTEST HELPERS
# ════════════════════════════════════════════════════════════════════════════

def execute_target_single(p, target_series, pos_limit=10):
    cash = 0.0
    pos = 0
    turnover = 0
    fills = 0
    pnl_path = []

    target_series = target_series.shift(1).fillna(0).reset_index(drop=True)

    for i, row in p.iterrows():
        desired = int(target_series.iloc[i])
        desired = max(-pos_limit, min(pos_limit, desired))

        delta = desired - pos

        if delta > 0:
            qty = min(delta, int(row["ask_volume_1"]))
            if qty > 0:
                cash -= qty * row["ask_price_1"]
                pos += qty
                turnover += qty
                fills += 1

        elif delta < 0:
            qty = min(-delta, int(row["bid_volume_1"]))
            if qty > 0:
                cash += qty * row["bid_price_1"]
                pos -= qty
                turnover += qty
                fills += 1

        pnl_path.append(cash + pos * row["mid"])

    final = p.iloc[-1]

    marked = cash + pos * final["mid"]

    liquidated = cash
    if pos > 0:
        liquidated += pos * final["bid_price_1"]
    elif pos < 0:
        liquidated -= (-pos) * final["ask_price_1"]

    return {
        "marked_pnl": marked,
        "liquidated_pnl": liquidated,
        "final_pos": pos,
        "turnover": turnover,
        "fills": fills,
        "max_pnl": max(pnl_path) if pnl_path else 0,
        "min_pnl": min(pnl_path) if pnl_path else 0,
    }


def make_target_from_signal(signal, entry_z, exit_z=0.0, pos_limit=10):
    target = pd.Series(0, index=signal.index, dtype=float)

    current = 0

    for i, s in enumerate(signal.fillna(0)):
        if current == 0:
            if s > entry_z:
                current = pos_limit
            elif s < -entry_z:
                current = -pos_limit
        elif current > 0:
            if s < exit_z:
                current = 0
        elif current < 0:
            if s > -exit_z:
                current = 0

        target.iloc[i] = current

    return target


def backtest_single_signal(product_name, signal_name, entry_z, invert=False):
    day_rows = []

    for day in DAYS:
        p, sigs = make_single_signals(product_name, day)

        if signal_name not in sigs.columns:
            continue

        signal = sigs[signal_name].copy()

        # Signals with "_z_" are already z-like. Others are standardized.
        if "_z_" not in signal_name:
            signal = zscore_series(signal, window=500, min_periods=100).fillna(0)
        else:
            signal = signal.fillna(0)

        if invert:
            signal = -signal

        target = make_target_from_signal(signal, entry_z=entry_z, exit_z=0.0, pos_limit=POS_LIMIT)
        result = execute_target_single(p, target, pos_limit=POS_LIMIT)

        result.update({
            "day": day,
            "strategy": "single_signal",
            "product": product_name,
            "signal": signal_name,
            "entry_z": entry_z,
            "invert": invert,
        })

        day_rows.append(result)

    day_df = pd.DataFrame(day_rows)

    summary = {
        "strategy": "single_signal",
        "product": product_name,
        "signal": signal_name,
        "entry_z": entry_z,
        "invert": invert,
        "total_marked_pnl": day_df["marked_pnl"].sum(),
        "total_liquidated_pnl": day_df["liquidated_pnl"].sum(),
        "avg_day_liquidated_pnl": day_df["liquidated_pnl"].mean(),
        "worst_day_liquidated_pnl": day_df["liquidated_pnl"].min(),
        "best_day_liquidated_pnl": day_df["liquidated_pnl"].max(),
        "positive_days": int((day_df["liquidated_pnl"] > 0).sum()),
        "total_turnover": day_df["turnover"].sum(),
        "total_fills": day_df["fills"].sum(),
    }

    return summary, day_df

In [12]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 12 — BACKTEST SINGLE-PRODUCT CANDIDATES
# ════════════════════════════════════════════════════════════════════════════

single_bt_summaries = []
single_bt_days = []

top_single_candidates = single_agg.head(60).copy()

for _, row in top_single_candidates.iterrows():
    p_name = row["product"]
    sig_name = row["signal"]

    for invert in [False, True]:
        for entry_z in ENTRY_ZS:
            summary, day_df = backtest_single_signal(
                product_name=p_name,
                signal_name=sig_name,
                entry_z=entry_z,
                invert=invert,
            )

            summary["source_score"] = row["single_score"]
            single_bt_summaries.append(summary)

            day_df["source_score"] = row["single_score"]
            single_bt_days.append(day_df)

single_bt_summary = pd.DataFrame(single_bt_summaries)
single_bt_day = pd.concat(single_bt_days, ignore_index=True)

single_bt_summary = single_bt_summary.sort_values("total_liquidated_pnl", ascending=False)

display(single_bt_summary.head(80))

single_bt_summary.to_csv(OUT_DIR / "single_backtest_summary.csv", index=False)
single_bt_day.to_csv(OUT_DIR / "single_backtest_by_day.csv", index=False)

,strategy,product,signal,entry_z,invert,total_marked_pnl,total_liquidated_pnl,avg_day_liquidated_pnl,worst_day_liquidated_pnl,best_day_liquidated_pnl,positive_days,total_turnover,total_fills,source_score
63,single_signal,PANEL_1X4,meanrev_z_1000,2.0,True,30261.0,30126.0,10042.000000,4552.0,20484.0,3,630,85,77.028324
51,single_signal,PANEL_1X4,breakout_z_1000,2.0,False,30261.0,30126.0,10042.000000,4552.0,20484.0,3,630,85,77.028324
135,single_signal,PANEL_1X4,meanrev_z_1000,2.0,True,30261.0,30126.0,10042.000000,4552.0,20484.0,3,630,85,57.051341
139,single_signal,PANEL_1X4,breakout_z_1000,2.0,False,30261.0,30126.0,10042.000000,4552.0,20484.0,3,630,85,57.051341
443,single_signal,PANEL_1X4,breakout_z_1000,2.0,False,30261.0,30126.0,10042.000000,4552.0,20484.0,3,630,85,40.608742
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230,single_signal,PANEL_1X4,rev_ret_500,1.5,True,21497.0,21362.0,7120.666667,215.0,20875.0,3,1750,234,52.142762
469,single_signal,PANEL_1X4,meanrev_z_500,1.0,True,20835.0,20740.0,6913.333333,-263.0,18765.0,2,2400,322,39.226469
277,single_signal,PANEL_1X4,meanrev_z_500,1.0,True,20835.0,20740.0,6913.333333,-263.0,18765.0,2,2400,322,49.110921
241,single_signal,PANEL_1X4,breakout_z_500,1.0,False,20835.0,20740.0,6913.333333,-263.0,18765.0,2,2400,322,50.253266


In [13]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 13 — LEAD-LAG BACKTEST
# ════════════════════════════════════════════════════════════════════════════

def backtest_leadlag(leader, follower, past_h, entry_z, invert=False):
    day_rows = []

    for day in DAYS:
        dm = mid_wide.loc[day].copy()

        leader_move = dm[leader].diff(past_h)
        signal = zscore_series(leader_move, window=500, min_periods=100).fillna(0)

        if invert:
            signal = -signal

        p = get_product_day_frame(follower, day)

        signal = signal.reindex(p["timestamp"]).fillna(0).reset_index(drop=True)
        target = make_target_from_signal(signal, entry_z=entry_z, exit_z=0.0, pos_limit=POS_LIMIT)

        result = execute_target_single(p, target, pos_limit=POS_LIMIT)

        result.update({
            "day": day,
            "strategy": "leadlag",
            "leader": leader,
            "follower": follower,
            "past_h": past_h,
            "entry_z": entry_z,
            "invert": invert,
        })

        day_rows.append(result)

    day_df = pd.DataFrame(day_rows)

    summary = {
        "strategy": "leadlag",
        "leader": leader,
        "follower": follower,
        "past_h": past_h,
        "entry_z": entry_z,
        "invert": invert,
        "total_marked_pnl": day_df["marked_pnl"].sum(),
        "total_liquidated_pnl": day_df["liquidated_pnl"].sum(),
        "avg_day_liquidated_pnl": day_df["liquidated_pnl"].mean(),
        "worst_day_liquidated_pnl": day_df["liquidated_pnl"].min(),
        "best_day_liquidated_pnl": day_df["liquidated_pnl"].max(),
        "positive_days": int((day_df["liquidated_pnl"] > 0).sum()),
        "total_turnover": day_df["turnover"].sum(),
        "total_fills": day_df["fills"].sum(),
    }

    return summary, day_df


leadlag_bt_summaries = []
leadlag_bt_days = []

top_leadlag_candidates = leadlag_agg.head(50).copy()

for _, row in top_leadlag_candidates.iterrows():
    leader = row["leader"]
    follower = row["follower"]
    past_h = int(row["past_h"])

    for invert in [False, True]:
        for entry_z in ENTRY_ZS:
            summary, day_df = backtest_leadlag(
                leader=leader,
                follower=follower,
                past_h=past_h,
                entry_z=entry_z,
                invert=invert,
            )

            summary["source_score"] = row["leadlag_score"]
            leadlag_bt_summaries.append(summary)

            day_df["source_score"] = row["leadlag_score"]
            leadlag_bt_days.append(day_df)

leadlag_bt_summary = pd.DataFrame(leadlag_bt_summaries)
leadlag_bt_day = pd.concat(leadlag_bt_days, ignore_index=True)

leadlag_bt_summary = leadlag_bt_summary.sort_values("total_liquidated_pnl", ascending=False)

display(leadlag_bt_summary.head(80))

leadlag_bt_summary.to_csv(OUT_DIR / "leadlag_backtest_summary.csv", index=False)
leadlag_bt_day.to_csv(OUT_DIR / "leadlag_backtest_by_day.csv", index=False)

,strategy,leader,follower,past_h,entry_z,invert,total_marked_pnl,total_liquidated_pnl,avg_day_liquidated_pnl,worst_day_liquidated_pnl,best_day_liquidated_pnl,positive_days,total_turnover,total_fills,source_score
307,leadlag,PANEL_4X4,PANEL_2X2,50,2.0,False,11279.0,11279.0,3759.666667,2057.0,5936.0,3,2560,341,44.551203
391,leadlag,PANEL_2X2,PANEL_4X4,500,2.0,True,10538.0,10448.0,3482.666667,-1013.0,10876.0,2,1360,189,38.285822
103,leadlag,PANEL_2X2,PANEL_4X4,500,2.0,True,10538.0,10448.0,3482.666667,-1013.0,10876.0,2,1360,189,88.868586
159,leadlag,PANEL_2X2,PANEL_4X4,500,2.0,True,10538.0,10448.0,3482.666667,-1013.0,10876.0,2,1360,189,68.837202
102,leadlag,PANEL_2X2,PANEL_4X4,500,1.5,True,10200.0,10110.0,3370.000000,-1400.0,11631.0,1,1880,266,88.868586
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,leadlag,PANEL_2X2,PANEL_4X4,500,1.0,True,-2889.0,-2979.0,-993.000000,-5748.0,6984.0,1,2720,368,68.837202
389,leadlag,PANEL_2X2,PANEL_4X4,500,1.0,True,-2889.0,-2979.0,-993.000000,-5748.0,6984.0,1,2720,368,38.285822
179,leadlag,PANEL_1X2,PANEL_2X4,500,2.0,False,-2953.0,-3108.0,-1036.000000,-8226.0,7061.0,1,1310,175,62.547863
163,leadlag,PANEL_1X2,PANEL_2X4,500,2.0,False,-2953.0,-3108.0,-1036.000000,-8226.0,7061.0,1,1310,175,66.626103


In [14]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 14 — PAIR SPREAD BACKTEST
# ════════════════════════════════════════════════════════════════════════════

def execute_pair_targets(pa, pb, target_a, target_b):
    cash = 0.0
    pos_a = 0
    pos_b = 0
    turnover = 0
    fills = 0
    pnl_path = []

    target_a = target_a.shift(1).fillna(0).reset_index(drop=True)
    target_b = target_b.shift(1).fillna(0).reset_index(drop=True)

    for i in range(len(pa)):
        ra = pa.iloc[i]
        rb = pb.iloc[i]

        desired_a = int(max(-POS_LIMIT, min(POS_LIMIT, target_a.iloc[i])))
        desired_b = int(max(-POS_LIMIT, min(POS_LIMIT, target_b.iloc[i])))

        # Execute A
        delta_a = desired_a - pos_a
        if delta_a > 0:
            qty = min(delta_a, int(ra["ask_volume_1"]))
            if qty > 0:
                cash -= qty * ra["ask_price_1"]
                pos_a += qty
                turnover += qty
                fills += 1
        elif delta_a < 0:
            qty = min(-delta_a, int(ra["bid_volume_1"]))
            if qty > 0:
                cash += qty * ra["bid_price_1"]
                pos_a -= qty
                turnover += qty
                fills += 1

        # Execute B
        delta_b = desired_b - pos_b
        if delta_b > 0:
            qty = min(delta_b, int(rb["ask_volume_1"]))
            if qty > 0:
                cash -= qty * rb["ask_price_1"]
                pos_b += qty
                turnover += qty
                fills += 1
        elif delta_b < 0:
            qty = min(-delta_b, int(rb["bid_volume_1"]))
            if qty > 0:
                cash += qty * rb["bid_price_1"]
                pos_b -= qty
                turnover += qty
                fills += 1

        pnl_path.append(cash + pos_a * ra["mid"] + pos_b * rb["mid"])

    final_a = pa.iloc[-1]
    final_b = pb.iloc[-1]

    marked = cash + pos_a * final_a["mid"] + pos_b * final_b["mid"]

    liquidated = cash

    if pos_a > 0:
        liquidated += pos_a * final_a["bid_price_1"]
    elif pos_a < 0:
        liquidated -= (-pos_a) * final_a["ask_price_1"]

    if pos_b > 0:
        liquidated += pos_b * final_b["bid_price_1"]
    elif pos_b < 0:
        liquidated -= (-pos_b) * final_b["ask_price_1"]

    return {
        "marked_pnl": marked,
        "liquidated_pnl": liquidated,
        "final_pos_a": pos_a,
        "final_pos_b": pos_b,
        "turnover": turnover,
        "fills": fills,
        "max_pnl": max(pnl_path) if pnl_path else 0,
        "min_pnl": min(pnl_path) if pnl_path else 0,
    }


def backtest_pair_spread(a, b, window, entry_z):
    day_rows = []

    for day in DAYS:
        dm = mid_wide.loc[day].copy()

        alpha, beta = fit_ols_y_on_x(dm[a], dm[b])
        spread = dm[a] - (alpha + beta * dm[b])

        z = zscore_series(spread, window=window, min_periods=max(50, window // 5)).fillna(0)

        # If spread high: A expensive vs B => sell A, buy B.
        target_a = pd.Series(0, index=z.index, dtype=float)
        target_b = pd.Series(0, index=z.index, dtype=float)

        state = 0

        for i, val in enumerate(z):
            if state == 0:
                if val > entry_z:
                    state = -1
                elif val < -entry_z:
                    state = 1
            else:
                if abs(val) < 0.0:
                    state = 0

            if state == -1:
                target_a.iloc[i] = -POS_LIMIT
                target_b.iloc[i] = POS_LIMIT
            elif state == 1:
                target_a.iloc[i] = POS_LIMIT
                target_b.iloc[i] = -POS_LIMIT
            else:
                target_a.iloc[i] = 0
                target_b.iloc[i] = 0

        pa = get_product_day_frame(a, day)
        pb = get_product_day_frame(b, day)

        result = execute_pair_targets(pa, pb, target_a, target_b)

        result.update({
            "day": day,
            "strategy": "pair_spread",
            "a": a,
            "b": b,
            "window": window,
            "entry_z": entry_z,
            "alpha": alpha,
            "beta": beta,
        })

        day_rows.append(result)

    day_df = pd.DataFrame(day_rows)

    summary = {
        "strategy": "pair_spread",
        "a": a,
        "b": b,
        "window": window,
        "entry_z": entry_z,
        "total_marked_pnl": day_df["marked_pnl"].sum(),
        "total_liquidated_pnl": day_df["liquidated_pnl"].sum(),
        "avg_day_liquidated_pnl": day_df["liquidated_pnl"].mean(),
        "worst_day_liquidated_pnl": day_df["liquidated_pnl"].min(),
        "best_day_liquidated_pnl": day_df["liquidated_pnl"].max(),
        "positive_days": int((day_df["liquidated_pnl"] > 0).sum()),
        "total_turnover": day_df["turnover"].sum(),
        "total_fills": day_df["fills"].sum(),
    }

    return summary, day_df


pair_bt_summaries = []
pair_bt_days = []

top_pair_candidates = pair_agg.head(30).copy()

for _, row in top_pair_candidates.iterrows():
    a = row["a"]
    b = row["b"]
    window = int(row["window"])

    for entry_z in ENTRY_ZS:
        summary, day_df = backtest_pair_spread(
            a=a,
            b=b,
            window=window,
            entry_z=entry_z,
        )

        summary["source_score"] = row["pair_score"]
        pair_bt_summaries.append(summary)

        day_df["source_score"] = row["pair_score"]
        pair_bt_days.append(day_df)

pair_bt_summary = pd.DataFrame(pair_bt_summaries)
pair_bt_day = pd.concat(pair_bt_days, ignore_index=True)

pair_bt_summary = pair_bt_summary.sort_values("total_liquidated_pnl", ascending=False)

display(pair_bt_summary.head(80))

pair_bt_summary.to_csv(OUT_DIR / "pair_backtest_summary.csv", index=False)
pair_bt_day.to_csv(OUT_DIR / "pair_backtest_by_day.csv", index=False)

,strategy,a,b,window,entry_z,total_marked_pnl,total_liquidated_pnl,avg_day_liquidated_pnl,worst_day_liquidated_pnl,best_day_liquidated_pnl,positive_days,total_turnover,total_fills,source_score
99,pair_spread,PANEL_1X2,PANEL_1X4,500,2.00,59216.0,58911.0,19637.000000,4351.0,28470.0,3,60,7,14.770684
98,pair_spread,PANEL_1X2,PANEL_1X4,500,1.50,59005.0,58700.0,19566.666667,4170.0,28440.0,3,60,6,14.770684
119,pair_spread,PANEL_1X2,PANEL_1X4,1000,2.00,53965.0,53660.0,17886.666667,-2370.0,28470.0,2,60,6,13.518394
118,pair_spread,PANEL_1X2,PANEL_1X4,1000,1.50,53439.0,53134.0,17711.333333,-2816.0,28390.0,2,60,7,13.518394
117,pair_spread,PANEL_1X2,PANEL_1X4,1000,1.00,52955.0,52650.0,17550.000000,-3300.0,28390.0,2,60,7,13.518394
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34,pair_spread,PANEL_1X2,PANEL_4X4,250,1.50,-8015.0,-8320.0,-2773.333333,-19620.0,12940.0,1,60,7,16.506750
42,pair_spread,PANEL_1X2,PANEL_4X4,1000,1.50,-9285.0,-9590.0,-3196.666667,-21140.0,13010.0,1,60,6,16.221077
41,pair_spread,PANEL_1X2,PANEL_4X4,1000,1.00,-9285.0,-9590.0,-3196.666667,-21140.0,13010.0,1,60,6,16.221077
40,pair_spread,PANEL_1X2,PANEL_4X4,1000,0.75,-9285.0,-9590.0,-3196.666667,-21140.0,13010.0,1,60,6,16.221077


In [15]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 15 — INTEGER / FORMULA BASKET BACKTEST
# ════════════════════════════════════════════════════════════════════════════

def parse_coef_list(coef_str):
    return [int(x.strip()) for x in coef_str.strip("[]").split(",")]


def execute_multi_targets(day_frames, target_positions):
    """
    day_frames: dict product -> frame
    target_positions: DataFrame columns product, index timestamp row number
    """
    cash = 0.0
    positions = {p: 0 for p in PANEL_PRODUCTS}
    turnover = 0
    fills = 0
    pnl_path = []

    target_positions = target_positions.shift(1).fillna(0)

    n = len(next(iter(day_frames.values())))

    for i in range(n):
        for p in PANEL_PRODUCTS:
            frame = day_frames[p]
            row = frame.iloc[i]

            desired = int(max(-POS_LIMIT, min(POS_LIMIT, target_positions[p].iloc[i])))
            delta = desired - positions[p]

            if delta > 0:
                qty = min(delta, int(row["ask_volume_1"]))
                if qty > 0:
                    cash -= qty * row["ask_price_1"]
                    positions[p] += qty
                    turnover += qty
                    fills += 1
            elif delta < 0:
                qty = min(-delta, int(row["bid_volume_1"]))
                if qty > 0:
                    cash += qty * row["bid_price_1"]
                    positions[p] -= qty
                    turnover += qty
                    fills += 1

        mark = cash
        for p in PANEL_PRODUCTS:
            mark += positions[p] * day_frames[p].iloc[i]["mid"]

        pnl_path.append(mark)

    marked = cash
    liquidated = cash

    for p in PANEL_PRODUCTS:
        final = day_frames[p].iloc[-1]
        pos = positions[p]

        marked += pos * final["mid"]

        if pos > 0:
            liquidated += pos * final["bid_price_1"]
        elif pos < 0:
            liquidated -= (-pos) * final["ask_price_1"]

    out = {
        "marked_pnl": marked,
        "liquidated_pnl": liquidated,
        "turnover": turnover,
        "fills": fills,
        "max_pnl": max(pnl_path) if pnl_path else 0,
        "min_pnl": min(pnl_path) if pnl_path else 0,
    }

    for p in PANEL_PRODUCTS:
        out[f"final_pos_{p}"] = positions[p]

    return out


def backtest_integer_combo(coefs, window, entry_z):
    coefs = list(coefs)
    day_rows = []

    max_units = min(
        POS_LIMIT // abs(c)
        for c in coefs
        if c != 0
    )

    max_units = max(1, max_units)

    for day in DAYS:
        dm = mid_wide.loc[day].copy()
        coef_arr = np.array(coefs)

        combo = pd.Series(dm.values @ coef_arr, index=dm.index)
        z = zscore_series(combo, window=window, min_periods=max(50, window // 5)).fillna(0)

        target_positions = pd.DataFrame(0, index=dm.index, columns=PANEL_PRODUCTS, dtype=float)

        state = 0

        for i, val in enumerate(z):
            if state == 0:
                if val > entry_z:
                    state = -1
                elif val < -entry_z:
                    state = 1
            else:
                if abs(val) < 0.0:
                    state = 0

            for p, c in zip(PANEL_PRODUCTS, coefs):
                # If combo high, sell the combo: target = -coef.
                target_positions.iloc[i, target_positions.columns.get_loc(p)] = state * c * max_units

        day_frames = {
            p: get_product_day_frame(p, day)
            for p in PANEL_PRODUCTS
        }

        result = execute_multi_targets(day_frames, target_positions)

        result.update({
            "day": day,
            "strategy": "integer_combo",
            "coefs": str(coefs),
            "window": window,
            "entry_z": entry_z,
            "max_units": max_units,
        })

        day_rows.append(result)

    day_df = pd.DataFrame(day_rows)

    summary = {
        "strategy": "integer_combo",
        "coefs": str(coefs),
        "window": window,
        "entry_z": entry_z,
        "max_units": max_units,
        "total_marked_pnl": day_df["marked_pnl"].sum(),
        "total_liquidated_pnl": day_df["liquidated_pnl"].sum(),
        "avg_day_liquidated_pnl": day_df["liquidated_pnl"].mean(),
        "worst_day_liquidated_pnl": day_df["liquidated_pnl"].min(),
        "best_day_liquidated_pnl": day_df["liquidated_pnl"].max(),
        "positive_days": int((day_df["liquidated_pnl"] > 0).sum()),
        "total_turnover": day_df["turnover"].sum(),
        "total_fills": day_df["fills"].sum(),
    }

    return summary, day_df


combo_bt_summaries = []
combo_bt_days = []

top_combo_candidates = combo_agg.head(40).copy()

for _, row in top_combo_candidates.iterrows():
    coefs = parse_coef_list(row["coefs"])
    window = int(row["window"])

    for entry_z in [1.0, 1.5, 2.0]:
        summary, day_df = backtest_integer_combo(
            coefs=coefs,
            window=window,
            entry_z=entry_z,
        )

        summary["source_score"] = row["combo_score"]
        combo_bt_summaries.append(summary)

        day_df["source_score"] = row["combo_score"]
        combo_bt_days.append(day_df)

combo_bt_summary = pd.DataFrame(combo_bt_summaries)
combo_bt_day = pd.concat(combo_bt_days, ignore_index=True)

combo_bt_summary = combo_bt_summary.sort_values("total_liquidated_pnl", ascending=False)

display(combo_bt_summary.head(80))

combo_bt_summary.to_csv(OUT_DIR / "combo_backtest_summary.csv", index=False)
combo_bt_day.to_csv(OUT_DIR / "combo_backtest_by_day.csv", index=False)

,strategy,coefs,window,entry_z,max_units,total_marked_pnl,total_liquidated_pnl,avg_day_liquidated_pnl,worst_day_liquidated_pnl,best_day_liquidated_pnl,positive_days,total_turnover,total_fills,source_score
82,integer_combo,"[0, 1, 1, 0, 0]",1000,1.5,10,55274.0,55014.0,18338.000000,7170.0,32720.0,3,60,8,9.267030
26,integer_combo,"[1, 0, 0, 0, 1]",1000,2.0,10,41095.0,40790.0,13596.666667,5870.0,21430.0,3,60,6,9.906237
25,integer_combo,"[1, 0, 0, 0, 1]",1000,1.5,10,40005.0,39700.0,13233.333333,5740.0,21430.0,3,60,6,9.906237
35,integer_combo,"[1, 0, 0, 0, 1]",500,2.0,10,39725.0,39420.0,13140.000000,5670.0,20260.0,3,60,6,9.855141
24,integer_combo,"[1, 0, 0, 0, 1]",1000,1.0,10,39534.0,39229.0,13076.333333,5740.0,21430.0,3,60,7,9.906237
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36,integer_combo,"[1, 1, 0, 0, 1]",500,1.0,10,-11020.0,-11450.0,-3816.666667,-16650.0,2750.0,2,90,9,9.851815
55,integer_combo,"[0, 1, 0, -1, 0]",250,1.5,10,-11430.0,-11710.0,-3903.333333,-15450.0,8710.0,1,60,6,9.614861
78,integer_combo,"[0, 0, 0, 1, 1]",500,1.0,10,-11430.0,-11720.0,-3906.666667,-12130.0,2690.0,1,60,6,9.372339
79,integer_combo,"[0, 0, 0, 1, 1]",500,1.5,10,-11430.0,-11720.0,-3906.666667,-12130.0,2690.0,1,60,6,9.372339


In [16]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 16 — GEOMETRY RESIDUAL BACKTEST
# ════════════════════════════════════════════════════════════════════════════

def backtest_geometry_residual(product_name, window, entry_z, invert=False):
    day_rows = []

    for day in DAYS:
        p = get_product_day_frame(product_name, day)

        day_resid = (
            geometry_resid[geometry_resid["file_day"] == day]
            .set_index("timestamp")
        )

        resid = day_resid[product_name].reindex(p["timestamp"]).reset_index(drop=True)

        # If residual high, product expensive vs geometry model.
        # Default signal = -resid z-score, so high residual => sell.
        z = zscore_series(resid, window=window, min_periods=max(50, window // 5)).fillna(0)
        signal = -z

        if invert:
            signal = -signal

        target = make_target_from_signal(signal, entry_z=entry_z, exit_z=0.0, pos_limit=POS_LIMIT)
        result = execute_target_single(p, target, pos_limit=POS_LIMIT)

        result.update({
            "day": day,
            "strategy": "geometry_residual",
            "product": product_name,
            "window": window,
            "entry_z": entry_z,
            "invert": invert,
        })

        day_rows.append(result)

    day_df = pd.DataFrame(day_rows)

    summary = {
        "strategy": "geometry_residual",
        "product": product_name,
        "window": window,
        "entry_z": entry_z,
        "invert": invert,
        "total_marked_pnl": day_df["marked_pnl"].sum(),
        "total_liquidated_pnl": day_df["liquidated_pnl"].sum(),
        "avg_day_liquidated_pnl": day_df["liquidated_pnl"].mean(),
        "worst_day_liquidated_pnl": day_df["liquidated_pnl"].min(),
        "best_day_liquidated_pnl": day_df["liquidated_pnl"].max(),
        "positive_days": int((day_df["liquidated_pnl"] > 0).sum()),
        "total_turnover": day_df["turnover"].sum(),
        "total_fills": day_df["fills"].sum(),
    }

    return summary, day_df


geometry_bt_summaries = []
geometry_bt_days = []

top_geometry_candidates = geometry_agg.head(30).copy()

for _, row in top_geometry_candidates.iterrows():
    product_name = row["product"]
    window = int(row["window"])

    for invert in [False, True]:
        for entry_z in ENTRY_ZS:
            summary, day_df = backtest_geometry_residual(
                product_name=product_name,
                window=window,
                entry_z=entry_z,
                invert=invert,
            )

            summary["source_score"] = row["geometry_score"]
            geometry_bt_summaries.append(summary)

            day_df["source_score"] = row["geometry_score"]
            geometry_bt_days.append(day_df)

geometry_bt_summary = pd.DataFrame(geometry_bt_summaries)
geometry_bt_day = pd.concat(geometry_bt_days, ignore_index=True)

geometry_bt_summary = geometry_bt_summary.sort_values("total_liquidated_pnl", ascending=False)

display(geometry_bt_summary.head(80))

geometry_bt_summary.to_csv(OUT_DIR / "geometry_backtest_summary.csv", index=False)
geometry_bt_day.to_csv(OUT_DIR / "geometry_backtest_by_day.csv", index=False)

KeyboardInterrupt: 

In [17]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 17 — FINAL RANKING + REPORT EXPORT
# Safe version: works even if Cell 16 / geometry backtest was skipped.
# ════════════════════════════════════════════════════════════════════════════

from pathlib import Path
import pandas as pd
import numpy as np

OUT_DIR = Path("outputs_panels")
OUT_DIR.mkdir(exist_ok=True)

# Collect only backtest summaries that actually exist.
bt_sources = []

possible_bt_names = [
    "single_bt_summary",
    "leadlag_bt_summary",
    "pair_bt_summary",
    "combo_bt_summary",
    "geometry_bt_summary",  # may not exist if Cell 16 skipped
]

for name in possible_bt_names:
    if name in globals():
        df = globals()[name]
        if isinstance(df, pd.DataFrame) and len(df) > 0:
            temp = df.copy()
            temp["source_table"] = name
            bt_sources.append(temp)
            print(f"Included {name}: {temp.shape}")
        else:
            print(f"Skipped {name}: empty or not DataFrame")
    else:
        print(f"Skipped {name}: not defined")

if bt_sources:
    all_bt_summary = pd.concat(bt_sources, ignore_index=True)
    all_bt_summary = all_bt_summary.sort_values("total_liquidated_pnl", ascending=False)
else:
    all_bt_summary = pd.DataFrame()
    print("No backtest summaries found.")

display(all_bt_summary.head(100))

all_bt_summary.to_csv(OUT_DIR / "ALL_panel_backtest_summary.csv", index=False)

# Robust candidates only if backtests exist.
if len(all_bt_summary) > 0 and "positive_days" in all_bt_summary.columns:
    robust = all_bt_summary[
        (all_bt_summary["positive_days"] == 3)
        & (all_bt_summary["total_liquidated_pnl"] > 0)
    ].copy()

    if len(robust) > 0:
        robust["pnl_per_turnover"] = (
            robust["total_liquidated_pnl"]
            / robust["total_turnover"].replace(0, np.nan)
        )

        robust = robust.sort_values(
            ["total_liquidated_pnl", "pnl_per_turnover"],
            ascending=[False, False],
        )
else:
    robust = pd.DataFrame()

display(robust.head(50))

robust.to_csv(OUT_DIR / "ROBUST_panel_candidates.csv", index=False)


# Helper for report.
report = []

report.append("# Panels Deep Research Report\n")
report.append("Products: " + ", ".join(PANEL_PRODUCTS) + "\n")
report.append("\nCell 16 / geometry residual backtest may have been skipped. This report includes all available scan and backtest outputs.\n")

def add_df(title, df_name, n=40):
    report.append(f"\n## {title}\n")

    if df_name not in globals():
        report.append(f"`{df_name}` not available / not run.\n")
        return

    df = globals()[df_name]

    if df is None or not isinstance(df, pd.DataFrame) or len(df) == 0:
        report.append("No rows.\n")
        return

    report.append("```text")
    report.append(df.head(n).to_string(index=False))
    report.append("```\n")


# Pure scans
add_df("Product Diagnostics", "diag", 30)
add_df("Trade Diagnostics", "trade_diag", 30)
add_df("Area Normalized Scan", "area_norm_df", 30)
add_df("Formula Scan", "formula_agg", 50)
add_df("Integer Combo Scan", "combo_agg", 50)
add_df("Pair Spread Scan", "pair_agg", 50)
add_df("Geometry Residual Scan", "geometry_agg", 50)
add_df("Single Signal Scan", "single_agg", 80)
add_df("Lead-Lag Scan", "leadlag_agg", 80)

# Backtests, only if available
if len(all_bt_summary) > 0:
    report.append("\n## All Available Backtests\n")
    report.append("```text")
    report.append(all_bt_summary.head(100).to_string(index=False))
    report.append("```\n")
else:
    report.append("\n## All Available Backtests\nNo backtests available.\n")

if len(robust) > 0:
    report.append("\n## Robust Positive-All-Day Candidates\n")
    report.append("```text")
    report.append(robust.head(100).to_string(index=False))
    report.append("```\n")
else:
    report.append("\n## Robust Positive-All-Day Candidates\nNo robust positive-all-day candidates available.\n")


report_path = OUT_DIR / "PANELS_deep_research_report.md"
report_path.write_text("\n".join(report), encoding="utf-8")

print(f"Saved report to: {report_path.resolve()}")
print(f"Saved all outputs to: {OUT_DIR.resolve()}")

Included single_bt_summary: (480, 15)
Included leadlag_bt_summary: (400, 16)
Included pair_bt_summary: (120, 15)
Included combo_bt_summary: (120, 15)
Skipped geometry_bt_summary: not defined


,strategy,product,signal,entry_z,invert,total_marked_pnl,total_liquidated_pnl,avg_day_liquidated_pnl,worst_day_liquidated_pnl,best_day_liquidated_pnl,...,source_score,source_table,leader,follower,past_h,a,b,window,coefs,max_units
880,pair_spread,NaN,NaN,2.0,NaN,59216.0,58911.0,19637.000000,4351.0,28470.0,...,14.770684,pair_bt_summary,NaN,NaN,NaN,PANEL_1X2,PANEL_1X4,500.0,NaN,NaN
881,pair_spread,NaN,NaN,1.5,NaN,59005.0,58700.0,19566.666667,4170.0,28440.0,...,14.770684,pair_bt_summary,NaN,NaN,NaN,PANEL_1X2,PANEL_1X4,500.0,NaN,NaN
1000,integer_combo,NaN,NaN,1.5,NaN,55274.0,55014.0,18338.000000,7170.0,32720.0,...,9.267030,combo_bt_summary,NaN,NaN,NaN,NaN,NaN,1000.0,"[0, 1, 1, 0, 0]",10.0
882,pair_spread,NaN,NaN,2.0,NaN,53965.0,53660.0,17886.666667,-2370.0,28470.0,...,13.518394,pair_bt_summary,NaN,NaN,NaN,PANEL_1X2,PANEL_1X4,1000.0,NaN,NaN
883,pair_spread,NaN,NaN,1.5,NaN,53439.0,53134.0,17711.333333,-2816.0,28390.0,...,13.518394,pair_bt_summary,NaN,NaN,NaN,PANEL_1X2,PANEL_1X4,1000.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50,single_signal,PANEL_1X4,mom_ret_500,1.0,False,22818.0,22683.0,7561.000000,-1152.0,23658.0,...,54.406471,single_bt_summary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,single_signal,PANEL_1X4,mom_ret_500,1.0,False,22818.0,22683.0,7561.000000,-1152.0,23658.0,...,52.142762,single_bt_summary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,single_signal,PANEL_1X4,rev_ret_500,1.0,True,22818.0,22683.0,7561.000000,-1152.0,23658.0,...,54.406471,single_bt_summary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1018,integer_combo,NaN,NaN,1.0,NaN,22450.0,22155.0,7385.000000,-3865.0,14840.0,...,9.723979,combo_bt_summary,NaN,NaN,NaN,NaN,NaN,1000.0,"[1, 1, 0, 0, 0]",10.0


,strategy,product,signal,entry_z,invert,total_marked_pnl,total_liquidated_pnl,avg_day_liquidated_pnl,worst_day_liquidated_pnl,best_day_liquidated_pnl,...,source_table,leader,follower,past_h,a,b,window,coefs,max_units,pnl_per_turnover
880,pair_spread,NaN,NaN,2.00,NaN,59216.0,58911.0,19637.000000,4351.0,28470.0,...,pair_bt_summary,NaN,NaN,NaN,PANEL_1X2,PANEL_1X4,500.0,NaN,NaN,981.850000
881,pair_spread,NaN,NaN,1.50,NaN,59005.0,58700.0,19566.666667,4170.0,28440.0,...,pair_bt_summary,NaN,NaN,NaN,PANEL_1X2,PANEL_1X4,500.0,NaN,NaN,978.333333
1000,integer_combo,NaN,NaN,1.50,NaN,55274.0,55014.0,18338.000000,7170.0,32720.0,...,combo_bt_summary,NaN,NaN,NaN,NaN,NaN,1000.0,"[0, 1, 1, 0, 0]",10.0,916.900000
886,pair_spread,NaN,NaN,1.00,NaN,41290.0,41000.0,13666.666667,5990.0,28210.0,...,pair_bt_summary,NaN,NaN,NaN,PANEL_1X4,PANEL_2X4,1000.0,NaN,NaN,683.333333
887,pair_spread,NaN,NaN,0.75,NaN,41112.0,40822.0,13607.333333,5812.0,28210.0,...,pair_bt_summary,NaN,NaN,NaN,PANEL_1X4,PANEL_2X4,1000.0,NaN,NaN,680.366667
1001,integer_combo,NaN,NaN,2.00,NaN,41095.0,40790.0,13596.666667,5870.0,21430.0,...,combo_bt_summary,NaN,NaN,NaN,NaN,NaN,1000.0,"[1, 0, 0, 0, 1]",10.0,679.833333
1002,integer_combo,NaN,NaN,1.50,NaN,40005.0,39700.0,13233.333333,5740.0,21430.0,...,combo_bt_summary,NaN,NaN,NaN,NaN,NaN,1000.0,"[1, 0, 0, 0, 1]",10.0,661.666667
1003,integer_combo,NaN,NaN,2.00,NaN,39725.0,39420.0,13140.000000,5670.0,20260.0,...,combo_bt_summary,NaN,NaN,NaN,NaN,NaN,500.0,"[1, 0, 0, 0, 1]",10.0,657.000000
1004,integer_combo,NaN,NaN,1.00,NaN,39534.0,39229.0,13076.333333,5740.0,21430.0,...,combo_bt_summary,NaN,NaN,NaN,NaN,NaN,1000.0,"[1, 0, 0, 0, 1]",10.0,653.816667
1005,integer_combo,NaN,NaN,1.50,NaN,36865.0,36560.0,12186.666667,4810.0,20070.0,...,combo_bt_summary,NaN,NaN,NaN,NaN,NaN,500.0,"[1, 0, 0, 0, 1]",10.0,609.333333


Saved report to: /Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels/PANELS_deep_research_report.md
Saved all outputs to: /Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels


In [18]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 18 — FOCUSED CORRECTED BACKTEST HELPERS
# ════════════════════════════════════════════════════════════════════════════

from pathlib import Path
import numpy as np
import pandas as pd

FOCUSED_OUT = Path("outputs_panels_focused")
FOCUSED_OUT.mkdir(exist_ok=True)

POS_LIMIT = 10

required = ["panel_prices", "panel_trades", "mid_wide", "PANEL_PRODUCTS", "DAYS"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing variables: {missing}. Run the earlier Panels scan setup cells first.")


def zscore_series(s: pd.Series, window=500, min_periods=100):
    mean = s.rolling(window, min_periods=min_periods).mean()
    std = s.rolling(window, min_periods=min_periods).std()
    return (s - mean) / std.replace(0, np.nan)


def get_panel_day(product_name: str, day: int) -> pd.DataFrame:
    return (
        panel_prices[
            (panel_prices["product"] == product_name)
            & (panel_prices["file_day"] == day)
        ]
        .sort_values("timestamp")
        .copy()
        .reset_index(drop=True)
    )


def fit_ols_y_on_x(y: pd.Series, x: pd.Series):
    valid = y.notna() & x.notna()
    yv = y[valid].values
    xv = x[valid].values

    if len(yv) < 200 or np.var(xv) == 0:
        return np.nan, np.nan

    beta = np.cov(yv, xv, ddof=0)[0, 1] / np.var(xv)
    alpha = np.mean(yv) - beta * np.mean(xv)

    return alpha, beta


def fit_global_pair_ols(a: str, b: str):
    y = mid_wide[a]
    x = mid_wide[b]
    return fit_ols_y_on_x(y, x)


def make_state_targets_from_signal(
    signal: pd.Series,
    entry_z: float,
    exit_z: float,
    pos_limit: int = 10,
    allow_flip: bool = True,
    max_hold_steps=None,
):
    """
    Converts a continuous signal into target position:
    +pos_limit when signal is high,
    -pos_limit when signal is low,
    flat when signal exits back inside exit_z.

    Corrected logic:
    - enter when |signal| > entry_z
    - exit when |signal| <= exit_z
    - optional flip if opposite extreme appears
    - optional max holding period
    """
    signal = signal.fillna(0).reset_index(drop=True)

    target = pd.Series(0, index=signal.index, dtype=float)
    state = 0
    hold_steps = 0
    entries = 0
    exits = 0
    flips = 0
    holds = []

    for i, s in enumerate(signal):
        old_state = state

        if state == 0:
            hold_steps = 0

            if s > entry_z:
                state = 1
                entries += 1
                hold_steps = 1

            elif s < -entry_z:
                state = -1
                entries += 1
                hold_steps = 1

        else:
            hold_steps += 1

            # Exit on convergence.
            if abs(s) <= exit_z:
                holds.append(hold_steps)
                state = 0
                exits += 1
                hold_steps = 0

            # Optional flip on opposite extreme.
            elif allow_flip and state == 1 and s < -entry_z:
                holds.append(hold_steps)
                state = -1
                flips += 1
                entries += 1
                hold_steps = 1

            elif allow_flip and state == -1 and s > entry_z:
                holds.append(hold_steps)
                state = 1
                flips += 1
                entries += 1
                hold_steps = 1

            # Optional time stop.
            elif max_hold_steps is not None and hold_steps >= max_hold_steps:
                holds.append(hold_steps)
                state = 0
                exits += 1
                hold_steps = 0

        target.iloc[i] = state * pos_limit

    if state != 0 and hold_steps > 0:
        holds.append(hold_steps)

    stats = {
        "entries": entries,
        "exits": exits,
        "flips": flips,
        "avg_hold_steps": float(np.mean(holds)) if holds else 0.0,
        "max_hold_steps": int(np.max(holds)) if holds else 0,
    }

    return target, stats


def execute_single_product(product_name: str, day: int, target: pd.Series):
    p = get_panel_day(product_name, day)
    target = target.shift(1).fillna(0).reset_index(drop=True)

    cash = 0.0
    pos = 0
    turnover = 0
    fills = 0
    pnl_path = []

    for i, row in p.iterrows():
        desired = int(max(-POS_LIMIT, min(POS_LIMIT, target.iloc[i])))
        delta = desired - pos

        if delta > 0:
            qty = min(delta, int(row["ask_volume_1"]))
            if qty > 0:
                cash -= qty * row["ask_price_1"]
                pos += qty
                turnover += qty
                fills += 1

        elif delta < 0:
            qty = min(-delta, int(row["bid_volume_1"]))
            if qty > 0:
                cash += qty * row["bid_price_1"]
                pos -= qty
                turnover += qty
                fills += 1

        pnl_path.append(cash + pos * row["mid"])

    final = p.iloc[-1]
    marked_pnl = cash + pos * final["mid"]

    liquidated_pnl = cash
    if pos > 0:
        liquidated_pnl += pos * final["bid_price_1"]
    elif pos < 0:
        liquidated_pnl -= (-pos) * final["ask_price_1"]

    return {
        "marked_pnl": marked_pnl,
        "liquidated_pnl": liquidated_pnl,
        "final_pos": pos,
        "turnover": turnover,
        "fills": fills,
        "max_pnl": max(pnl_path) if pnl_path else 0,
        "min_pnl": min(pnl_path) if pnl_path else 0,
    }


def execute_pair(a: str, b: str, day: int, target_a: pd.Series, target_b: pd.Series):
    pa = get_panel_day(a, day)
    pb = get_panel_day(b, day)

    target_a = target_a.shift(1).fillna(0).reset_index(drop=True)
    target_b = target_b.shift(1).fillna(0).reset_index(drop=True)

    cash = 0.0
    pos_a = 0
    pos_b = 0
    turnover = 0
    fills = 0
    pnl_path = []

    n = min(len(pa), len(pb))

    for i in range(n):
        ra = pa.iloc[i]
        rb = pb.iloc[i]

        desired_a = int(max(-POS_LIMIT, min(POS_LIMIT, target_a.iloc[i])))
        desired_b = int(max(-POS_LIMIT, min(POS_LIMIT, target_b.iloc[i])))

        # Execute A.
        delta_a = desired_a - pos_a
        if delta_a > 0:
            qty = min(delta_a, int(ra["ask_volume_1"]))
            if qty > 0:
                cash -= qty * ra["ask_price_1"]
                pos_a += qty
                turnover += qty
                fills += 1

        elif delta_a < 0:
            qty = min(-delta_a, int(ra["bid_volume_1"]))
            if qty > 0:
                cash += qty * ra["bid_price_1"]
                pos_a -= qty
                turnover += qty
                fills += 1

        # Execute B.
        delta_b = desired_b - pos_b
        if delta_b > 0:
            qty = min(delta_b, int(rb["ask_volume_1"]))
            if qty > 0:
                cash -= qty * rb["ask_price_1"]
                pos_b += qty
                turnover += qty
                fills += 1

        elif delta_b < 0:
            qty = min(-delta_b, int(rb["bid_volume_1"]))
            if qty > 0:
                cash += qty * rb["bid_price_1"]
                pos_b -= qty
                turnover += qty
                fills += 1

        pnl_path.append(cash + pos_a * ra["mid"] + pos_b * rb["mid"])

    fa = pa.iloc[-1]
    fb = pb.iloc[-1]

    marked_pnl = cash + pos_a * fa["mid"] + pos_b * fb["mid"]

    liquidated_pnl = cash

    if pos_a > 0:
        liquidated_pnl += pos_a * fa["bid_price_1"]
    elif pos_a < 0:
        liquidated_pnl -= (-pos_a) * fa["ask_price_1"]

    if pos_b > 0:
        liquidated_pnl += pos_b * fb["bid_price_1"]
    elif pos_b < 0:
        liquidated_pnl -= (-pos_b) * fb["ask_price_1"]

    return {
        "marked_pnl": marked_pnl,
        "liquidated_pnl": liquidated_pnl,
        "final_pos_a": pos_a,
        "final_pos_b": pos_b,
        "turnover": turnover,
        "fills": fills,
        "max_pnl": max(pnl_path) if pnl_path else 0,
        "min_pnl": min(pnl_path) if pnl_path else 0,
    }

In [20]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 19 — FAST VECTORISED PANELS SCAN
# Stage 1: broad fast vectorised scan over shortlisted panel baskets.
# No iterrows. No timestamp loops. No repeated filtering inside loops.
# ════════════════════════════════════════════════════════════════════════════

import time
import ast
from pathlib import Path
import numpy as np
import pandas as pd

FAST_OUT = Path("outputs_panels_fast")
FAST_OUT.mkdir(exist_ok=True)

# Required from previous setup cells
required = ["panel_prices", "PANEL_PRODUCTS", "DAYS"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing variables: {missing}. Run Panels setup cells first.")

POS_LIMIT = 10

FAST_WINDOWS = [250, 500, 1000, 2500]
FAST_HORIZONS = [100, 250, 500, 1000]
FAST_THRESHOLDS = [1.0, 1.5, 2.0, 2.5]
FAST_MODES = ["meanrev", "breakout"]

start_time = time.perf_counter()

# ───────────────────────────────────────────────────────────────────────────
# 1. Pivot once
# ───────────────────────────────────────────────────────────────────────────

mid_fast = (
    panel_prices
    .pivot_table(index=["file_day", "timestamp"], columns="product", values="mid_price", aggfunc="last")
    .sort_index()
    .reindex(columns=PANEL_PRODUCTS)
)

bid_fast = (
    panel_prices
    .pivot_table(index=["file_day", "timestamp"], columns="product", values="bid_price_1", aggfunc="last")
    .sort_index()
    .reindex(columns=PANEL_PRODUCTS)
)

ask_fast = (
    panel_prices
    .pivot_table(index=["file_day", "timestamp"], columns="product", values="ask_price_1", aggfunc="last")
    .sort_index()
    .reindex(columns=PANEL_PRODUCTS)
)

# Drop rows where any required product is missing.
valid_rows = ~(mid_fast.isna().any(axis=1) | bid_fast.isna().any(axis=1) | ask_fast.isna().any(axis=1))
mid_fast = mid_fast.loc[valid_rows]
bid_fast = bid_fast.loc[valid_rows]
ask_fast = ask_fast.loc[valid_rows]

assert list(mid_fast.columns) == PANEL_PRODUCTS
assert list(bid_fast.columns) == PANEL_PRODUCTS
assert list(ask_fast.columns) == PANEL_PRODUCTS

print("Aligned wide shape:", mid_fast.shape)
print("Products:", PANEL_PRODUCTS)

# ───────────────────────────────────────────────────────────────────────────
# 2. Convert each day to NumPy arrays once
# ───────────────────────────────────────────────────────────────────────────

FAST_DAY_BASE = {}

for day in DAYS:
    X_df = mid_fast.loc[day]
    B_df = bid_fast.loc[day]
    A_df = ask_fast.loc[day]

    # Sanity: each day handled separately, no cross-day horizon leakage possible.
    assert X_df.index.min() >= 0
    assert X_df.index.max() <= 999900
    assert X_df.shape == B_df.shape == A_df.shape

    FAST_DAY_BASE[day] = {
        "timestamps": X_df.index.to_numpy(),
        "X": X_df.to_numpy(float),
        "B": B_df.to_numpy(float),
        "A": A_df.to_numpy(float),
    }

    print(f"Day {day}: X shape={FAST_DAY_BASE[day]['X'].shape}")

product_to_idx = {p: i for i, p in enumerate(PANEL_PRODUCTS)}

# ───────────────────────────────────────────────────────────────────────────
# 3. Candidate basket construction
# q_signal: used to compute spread = X @ q_signal
# q_trade:  used to compute execution PnL. Scaled to position limit.
# ───────────────────────────────────────────────────────────────────────────

def make_q(coefs_dict):
    q = np.zeros(len(PANEL_PRODUCTS), dtype=float)
    for p, c in coefs_dict.items():
        q[product_to_idx[p]] = float(c)
    return q


def scale_trade_q(q):
    """
    Convert signal coefficients into trade quantities respecting max abs position.
    Integer combos get max integer units.
    Float OLS combos get proportional diagnostic sizing.
    """
    q = np.asarray(q, dtype=float)

    nonzero = np.abs(q[np.abs(q) > 1e-12])
    if len(nonzero) == 0:
        return q.copy()

    # If q is effectively integer, use integer unit scaling.
    if np.allclose(q, np.round(q), atol=1e-9):
        q_int = np.round(q).astype(int)
        max_abs = np.max(np.abs(q_int[np.abs(q_int) > 0]))
        units = max(1, POS_LIMIT // max_abs)
        return q_int.astype(float) * units

    # Otherwise, proportional diagnostic sizing.
    return q / np.max(np.abs(q)) * POS_LIMIT


def fit_ols_arrays(y, x):
    valid = np.isfinite(y) & np.isfinite(x)
    yv = y[valid]
    xv = x[valid]

    if len(yv) < 200 or np.var(xv) == 0:
        return 0.0, 1.0

    beta = np.cov(yv, xv, ddof=0)[0, 1] / np.var(xv)
    alpha = np.mean(yv) - beta * np.mean(xv)
    return alpha, beta


def add_candidate(candidates, name, family, q_signal_by_day, note=""):
    q_trade_by_day = {}

    for day, q in q_signal_by_day.items():
        q_trade_by_day[day] = scale_trade_q(q)

    candidates.append({
        "name": name,
        "family": family,
        "q_signal_by_day": q_signal_by_day,
        "q_trade_by_day": q_trade_by_day,
        "note": note,
    })


candidates = []

# A. Standalone unit baskets: each product alone.
for p in PANEL_PRODUCTS:
    q = make_q({p: 1})
    add_candidate(
        candidates,
        name=f"single::{p}",
        family="single",
        q_signal_by_day={day: q for day in DAYS},
        note="single product",
    )

# B. Simple pair differences: A - B for all pairs.
for a, b in combinations(PANEL_PRODUCTS, 2):
    q = make_q({a: 1, b: -1})
    add_candidate(
        candidates,
        name=f"pair_simple::{a}-{b}",
        family="pair_simple",
        q_signal_by_day={day: q for day in DAYS},
        note="simple pair difference",
    )

# C. Pair sums: A + B. Useful only as directional/basket diagnostics.
for a, b in combinations(PANEL_PRODUCTS, 2):
    q = make_q({a: 1, b: 1})
    add_candidate(
        candidates,
        name=f"pair_sum::{a}+{b}",
        family="pair_sum",
        q_signal_by_day={day: q for day in DAYS},
        note="all-positive pair basket, not market neutral",
    )

# D. OLS pair residuals, global beta.
for a, b in combinations(PANEL_PRODUCTS, 2):
    y = mid_fast[a]
    x = mid_fast[b]
    alpha, beta = fit_ols_arrays(y.to_numpy(float), x.to_numpy(float))

    # alpha does not affect rolling z-score materially, so q is enough for signal.
    q = make_q({a: 1, b: -beta})
    add_candidate(
        candidates,
        name=f"pair_global_ols::{a}~{b}",
        family="pair_global_ols",
        q_signal_by_day={day: q for day in DAYS},
        note=f"global beta={beta:.6f}",
    )

# E. OLS pair residuals, day-specific beta.
for a, b in combinations(PANEL_PRODUCTS, 2):
    q_by_day = {}

    for day in DAYS:
        X_day = FAST_DAY_BASE[day]["X"]
        ia = product_to_idx[a]
        ib = product_to_idx[b]
        alpha, beta = fit_ols_arrays(X_day[:, ia], X_day[:, ib])
        q_by_day[day] = make_q({a: 1, b: -beta})

    add_candidate(
        candidates,
        name=f"pair_day_ols::{a}~{b}",
        family="pair_day_ols",
        q_signal_by_day=q_by_day,
        note="day-specific beta; diagnostic/in-sample",
    )

# F. Hand-designed geometry formulas.
FORMULA_CANDIDATES = {
    "formula::same_area_2x2_minus_1x4": {"PANEL_2X2": 1, "PANEL_1X4": -1},
    "formula::same_aspect_2x4_minus_1x2": {"PANEL_2X4": 1, "PANEL_1X2": -1},
    "formula::same_aspect_4x4_minus_2x2": {"PANEL_4X4": 1, "PANEL_2X2": -1},
    "formula::2x_2x2_minus_2x4": {"PANEL_2X2": 2, "PANEL_2X4": -1},
    "formula::2x_1x4_minus_2x4": {"PANEL_1X4": 2, "PANEL_2X4": -1},
    "formula::2x_2x4_minus_4x4": {"PANEL_2X4": 2, "PANEL_4X4": -1},
    "formula::2x4_minus_2x2_plus_1x4": {"PANEL_2X4": 1, "PANEL_2X2": -1, "PANEL_1X4": -1},
}

for name, coefs in FORMULA_CANDIDATES.items():
    q = make_q(coefs)
    add_candidate(
        candidates,
        name=name,
        family="formula",
        q_signal_by_day={day: q for day in DAYS},
        note=str(coefs),
    )

# G. Add top integer combos from previous scan if available.
if "combo_agg" in globals() and isinstance(combo_agg, pd.DataFrame) and "coefs" in combo_agg.columns:
    added = 0

    for coef_str in combo_agg.head(30)["coefs"]:
        try:
            coefs_list = ast.literal_eval(coef_str)
        except Exception:
            continue

        if len(coefs_list) != len(PANEL_PRODUCTS):
            continue

        q = np.array(coefs_list, dtype=float)

        if np.allclose(q, 0):
            continue

        add_candidate(
            candidates,
            name=f"combo_from_scan::{coefs_list}",
            family="combo_from_scan",
            q_signal_by_day={day: q for day in DAYS},
            note="from previous integer combo scan top 30",
        )

        added += 1

    print(f"Added {added} candidates from combo_agg.")

print("Number of candidate baskets:", len(candidates))

# ───────────────────────────────────────────────────────────────────────────
# 4. Vectorised PnL calculation
# ───────────────────────────────────────────────────────────────────────────

def rolling_z_np(spread, window):
    s = pd.Series(spread)
    mean = s.rolling(window, min_periods=max(50, window // 5)).mean().to_numpy(float)
    std = s.rolling(window, min_periods=max(50, window // 5)).std().to_numpy(float)
    z = (spread - mean) / std
    z[~np.isfinite(z)] = 0.0
    return z


def vectorised_exec_arrays(X, B, A, q_trade, h):
    """
    For one unit long basket q_trade:
        entry uses ask where q > 0, bid where q < 0
        exit uses bid where q > 0, ask where q < 0

    Returns arrays for every valid timestamp t where exit is t+h.
    """
    assert h > 0
    assert X.shape == B.shape == A.shape
    assert X.shape[0] > h

    X0 = X[:-h]
    X1 = X[h:]
    B0 = B[:-h]
    A0 = A[:-h]
    B1 = B[h:]
    A1 = A[h:]

    q = q_trade.reshape(1, -1)

    entry_px = np.where(q > 0, A0, B0)
    exit_px = np.where(q > 0, B1, A1)

    exec_long = ((exit_px - entry_px) * q).sum(axis=1)
    mid_long = ((X1 - X0) * q).sum(axis=1)

    # Cost identity for long basket.
    cost_long = mid_long - exec_long

    # Should be non-negative apart from tiny floating noise.
    if np.nanmin(cost_long) < -1e-7:
        raise AssertionError(f"Cost identity failed: min cost_long={np.nanmin(cost_long)}")

    return exec_long, mid_long, cost_long


day_rows = []
runtime_checkpoint = time.perf_counter()

for ci, cand in enumerate(candidates, start=1):
    if ci == 1 or ci % 10 == 0:
        elapsed = time.perf_counter() - start_time
        print(f"[{elapsed:8.2f}s] Processing candidate {ci}/{len(candidates)}: {cand['name']}")

    for day in DAYS:
        base = FAST_DAY_BASE[day]
        X = base["X"]
        B = base["B"]
        A = base["A"]

        q_signal = np.asarray(cand["q_signal_by_day"][day], dtype=float)
        q_trade = np.asarray(cand["q_trade_by_day"][day], dtype=float)

        assert q_signal.shape == (len(PANEL_PRODUCTS),)
        assert q_trade.shape == (len(PANEL_PRODUCTS),)

        spread = X @ q_signal

        for window in FAST_WINDOWS:
            z_full = rolling_z_np(spread, window=window)

            for h in FAST_HORIZONS:
                if X.shape[0] <= h + window:
                    continue

                exec_long, mid_long, cost_long = vectorised_exec_arrays(X, B, A, q_trade, h)

                z = z_full[:-h]

                assert len(z) == len(exec_long)
                assert len(exec_long) == X.shape[0] - h

                for threshold in FAST_THRESHOLDS:
                    active = np.abs(z) >= threshold

                    if not np.any(active):
                        for mode in FAST_MODES:
                            day_rows.append({
                                "candidate": cand["name"],
                                "family": cand["family"],
                                "note": cand["note"],
                                "day": day,
                                "window": window,
                                "horizon": h,
                                "threshold": threshold,
                                "mode": mode,
                                "event_count": 0,
                                "win_events": 0,
                                "total_exec_pnl": 0.0,
                                "total_mid_pnl": 0.0,
                                "total_cost": 0.0,
                                "avg_exec_pnl": np.nan,
                                "median_exec_pnl": np.nan,
                                "hit_rate": np.nan,
                                "avg_cost": np.nan,
                                "avg_abs_z": np.nan,
                                "q_signal": str(np.round(q_signal, 6).tolist()),
                                "q_trade": str(np.round(q_trade, 6).tolist()),
                            })
                        continue

                    z_active = z[active]
                    exec_active_long = exec_long[active]
                    mid_active_long = mid_long[active]
                    cost_active_long = cost_long[active]

                    # Sanity: cost identity active.
                    if len(exec_active_long) > 0:
                        assert np.all(cost_active_long >= -1e-7)

                    for mode in FAST_MODES:
                        if mode == "meanrev":
                            # z low -> long basket; z high -> short basket
                            side = -np.sign(z_active)
                        elif mode == "breakout":
                            # z high -> long basket; z low -> short basket
                            side = np.sign(z_active)
                        else:
                            raise ValueError(mode)

                        exec_pnl = side * exec_active_long
                        mid_pnl = side * mid_active_long

                        # Cost identity:
                        # abs(mid_pnl - exec_pnl) should equal long-side spread cost.
                        identity_err = np.nanmax(np.abs(np.abs(mid_pnl - exec_pnl) - cost_active_long))
                        assert identity_err < 1e-6, f"Cost identity failed: {identity_err}"

                        event_count = len(exec_pnl)
                        win_events = int((exec_pnl > 0).sum())

                        day_rows.append({
                            "candidate": cand["name"],
                            "family": cand["family"],
                            "note": cand["note"],
                            "day": day,
                            "window": window,
                            "horizon": h,
                            "threshold": threshold,
                            "mode": mode,
                            "event_count": int(event_count),
                            "win_events": win_events,
                            "total_exec_pnl": float(np.nansum(exec_pnl)),
                            "total_mid_pnl": float(np.nansum(mid_pnl)),
                            "total_cost": float(np.nansum(cost_active_long)),
                            "avg_exec_pnl": float(np.nanmean(exec_pnl)) if event_count else np.nan,
                            "median_exec_pnl": float(np.nanmedian(exec_pnl)) if event_count else np.nan,
                            "hit_rate": float(win_events / event_count) if event_count else np.nan,
                            "avg_cost": float(np.nanmean(cost_active_long)) if event_count else np.nan,
                            "avg_abs_z": float(np.nanmean(np.abs(z_active))) if event_count else np.nan,
                            "q_signal": str(np.round(q_signal, 6).tolist()),
                            "q_trade": str(np.round(q_trade, 6).tolist()),
                        })

FAST_DAY_RESULTS = pd.DataFrame(day_rows)

print("FAST_DAY_RESULTS shape:", FAST_DAY_RESULTS.shape)
display(FAST_DAY_RESULTS.head())

# ───────────────────────────────────────────────────────────────────────────
# 5. Aggregate summary
# ───────────────────────────────────────────────────────────────────────────

FAST_BASE_SUMMARY = (
    FAST_DAY_RESULTS
    .groupby(["candidate", "family", "note", "window", "horizon", "threshold", "mode", "q_signal", "q_trade"])
    .agg(
        days=("day", "count"),
        active_days=("event_count", lambda x: int((x > 0).sum())),
        event_count=("event_count", "sum"),
        win_events=("win_events", "sum"),
        total_exec_pnl=("total_exec_pnl", "sum"),
        total_mid_pnl=("total_mid_pnl", "sum"),
        total_cost=("total_cost", "sum"),
        avg_day_exec_pnl=("total_exec_pnl", "mean"),
        worst_day_exec_pnl=("total_exec_pnl", "min"),
        best_day_exec_pnl=("total_exec_pnl", "max"),
        positive_days=("total_exec_pnl", lambda x: int((x > 0).sum())),
        avg_exec_pnl=("avg_exec_pnl", "mean"),
        median_exec_pnl=("median_exec_pnl", "mean"),
        avg_cost=("avg_cost", "mean"),
        avg_abs_z=("avg_abs_z", "mean"),
    )
    .reset_index()
)

FAST_BASE_SUMMARY["hit_rate"] = (
    FAST_BASE_SUMMARY["win_events"]
    / FAST_BASE_SUMMARY["event_count"].replace(0, np.nan)
)

FAST_BASE_SUMMARY["exec_minus_cost_ratio"] = (
    FAST_BASE_SUMMARY["total_exec_pnl"]
    / FAST_BASE_SUMMARY["total_cost"].replace(0, np.nan)
)

# Prefer stable signals:
# - active all days
# - positive all days
# - enough events
# - high total exec PnL
# - good avg event PnL
FAST_BASE_SUMMARY["stability_score"] = (
    FAST_BASE_SUMMARY["positive_days"] * 1000
    + FAST_BASE_SUMMARY["active_days"] * 100
    + np.sign(FAST_BASE_SUMMARY["total_exec_pnl"]) * np.log1p(np.abs(FAST_BASE_SUMMARY["total_exec_pnl"]))
    + FAST_BASE_SUMMARY["avg_exec_pnl"].fillna(0)
)

FAST_BASE_SUMMARY = FAST_BASE_SUMMARY.sort_values(
    ["positive_days", "total_exec_pnl", "avg_exec_pnl"],
    ascending=[False, False, False],
)

FAST_BEST_PER_BASKET = (
    FAST_BASE_SUMMARY
    .sort_values(["positive_days", "total_exec_pnl", "avg_exec_pnl"], ascending=[False, False, False])
    .groupby("candidate")
    .head(1)
    .reset_index(drop=True)
)

top_keys = FAST_BASE_SUMMARY.head(20)[
    ["candidate", "window", "horizon", "threshold", "mode"]
]

FAST_SIGNALS_TOP_ONLY = FAST_DAY_RESULTS.merge(
    top_keys,
    on=["candidate", "window", "horizon", "threshold", "mode"],
    how="inner"
).sort_values(["candidate", "window", "horizon", "threshold", "mode", "day"])

print("\nTop FAST_BASE_SUMMARY:")
display(FAST_BASE_SUMMARY.head(80))

print("\nFAST_BEST_PER_BASKET:")
display(FAST_BEST_PER_BASKET.head(80))

print("\nSignal counts per day for top configs:")
signal_counts = (
    FAST_SIGNALS_TOP_ONLY
    .groupby(["candidate", "window", "horizon", "threshold", "mode", "day"])
    .agg(
        event_count=("event_count", "sum"),
        total_exec_pnl=("total_exec_pnl", "sum"),
        hit_rate=("hit_rate", "mean"),
        avg_exec_pnl=("avg_exec_pnl", "mean"),
    )
    .reset_index()
)
display(signal_counts.head(100))

# ───────────────────────────────────────────────────────────────────────────
# 6. Save outputs
# ───────────────────────────────────────────────────────────────────────────

FAST_DAY_RESULTS.to_csv(FAST_OUT / "FAST_DAY_RESULTS.csv", index=False)
FAST_BASE_SUMMARY.to_csv(FAST_OUT / "FAST_BASE_SUMMARY.csv", index=False)
FAST_BEST_PER_BASKET.to_csv(FAST_OUT / "FAST_BEST_PER_BASKET.csv", index=False)
FAST_SIGNALS_TOP_ONLY.to_csv(FAST_OUT / "FAST_SIGNALS_TOP_ONLY.csv", index=False)

report = []
report.append("# Fast Vectorised Panels Scan\n")
report.append(f"Candidate baskets: {len(candidates)}\n")
report.append(f"Aligned shape: {mid_fast.shape}\n")
report.append(f"Windows: {FAST_WINDOWS}\n")
report.append(f"Horizons: {FAST_HORIZONS}\n")
report.append(f"Thresholds: {FAST_THRESHOLDS}\n")
report.append(f"Modes: {FAST_MODES}\n")

def add_table(title, df, n=50):
    report.append(f"\n## {title}\n")
    report.append("```text")
    report.append(df.head(n).to_string(index=False))
    report.append("```\n")

add_table("Top FAST_BASE_SUMMARY", FAST_BASE_SUMMARY, 80)
add_table("FAST_BEST_PER_BASKET", FAST_BEST_PER_BASKET, 80)
add_table("FAST_SIGNALS_TOP_ONLY", FAST_SIGNALS_TOP_ONLY, 120)

report_path = FAST_OUT / "FAST_panels_report.md"
report_path.write_text("\n".join(report), encoding="utf-8")

runtime = time.perf_counter() - start_time

print("\nSaved:")
print((FAST_OUT / "FAST_DAY_RESULTS.csv").resolve())
print((FAST_OUT / "FAST_BASE_SUMMARY.csv").resolve())
print((FAST_OUT / "FAST_BEST_PER_BASKET.csv").resolve())
print((FAST_OUT / "FAST_SIGNALS_TOP_ONLY.csv").resolve())
print(report_path.resolve())
print(f"\nFinal runtime: {runtime:.2f}s")

Aligned wide shape: (30000, 5)
Products: ['PANEL_1X2', 'PANEL_2X2', 'PANEL_1X4', 'PANEL_2X4', 'PANEL_4X4']
Day 2: X shape=(10000, 5)
Day 3: X shape=(10000, 5)
Day 4: X shape=(10000, 5)
Added 30 candidates from combo_agg.
Number of candidate baskets: 82
[    0.11s] Processing candidate 1/82: single::PANEL_1X2
[    0.64s] Processing candidate 10/82: pair_simple::PANEL_2X2-PANEL_1X4
[    1.19s] Processing candidate 20/82: pair_sum::PANEL_2X2+PANEL_1X4
[    1.73s] Processing candidate 30/82: pair_global_ols::PANEL_2X2~PANEL_1X4
[    2.40s] Processing candidate 40/82: pair_day_ols::PANEL_2X2~PANEL_1X4
[    2.93s] Processing candidate 50/82: formula::2x_1x4_minus_2x4
[    3.44s] Processing candidate 60/82: combo_from_scan::[1, 0, 0, -1, 0]
[    3.95s] Processing candidate 70/82: combo_from_scan::[0, 1, 0, 1, 1]
[    4.45s] Processing candidate 80/82: combo_from_scan::[0, 1, 1, 0, 0]
FAST_DAY_RESULTS shape: (31488, 20)


,candidate,family,note,day,window,horizon,threshold,mode,event_count,win_events,total_exec_pnl,total_mid_pnl,total_cost,avg_exec_pnl,median_exec_pnl,hit_rate,avg_cost,avg_abs_z,q_signal,q_trade
0,single::PANEL_1X2,single,single product,2,250,100,1.0,meanrev,5714,2794,-185840.0,-77205.0,679355.0,-32.523626,-30.0,0.488974,118.893070,1.714715,"[1.0, 0.0, 0.0, 0.0, 0.0]","[10.0, 0.0, 0.0, 0.0, 0.0]"
1,single::PANEL_1X2,single,single product,2,250,100,1.0,breakout,5714,2896,185840.0,77205.0,679355.0,32.523626,30.0,0.506825,118.893070,1.714715,"[1.0, 0.0, 0.0, 0.0, 0.0]","[10.0, 0.0, 0.0, 0.0, 0.0]"
2,single::PANEL_1X2,single,single product,2,250,100,1.5,meanrev,3404,1688,-260.0,67500.0,404600.0,-0.076381,0.0,0.495887,118.860165,2.031124,"[1.0, 0.0, 0.0, 0.0, 0.0]","[10.0, 0.0, 0.0, 0.0, 0.0]"
3,single::PANEL_1X2,single,single product,2,250,100,1.5,breakout,3404,1700,260.0,-67500.0,404600.0,0.076381,0.0,0.499412,118.860165,2.031124,"[1.0, 0.0, 0.0, 0.0, 0.0]","[10.0, 0.0, 0.0, 0.0, 0.0]"
4,single::PANEL_1X2,single,single product,2,250,100,2.0,meanrev,1553,793,32610.0,45930.0,184880.0,20.998068,50.0,0.510625,119.047006,2.390538,"[1.0, 0.0, 0.0, 0.0, 0.0]","[10.0, 0.0, 0.0, 0.0, 0.0]"



Top FAST_BASE_SUMMARY:


,candidate,family,note,window,horizon,threshold,mode,q_signal,q_trade,days,...,worst_day_exec_pnl,best_day_exec_pnl,positive_days,avg_exec_pnl,median_exec_pnl,avg_cost,avg_abs_z,hit_rate,exec_minus_cost_ratio,stability_score
89,"combo_from_scan::[0, 0, 0, 1, 1]",combo_from_scan,from previous integer combo scan top 30,1000,1000,1.0,meanrev,"[0.0, 0.0, 0.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 10.0, 10.0]",12,...,5577420.0,8409090.0,12,1584.835710,1243.333333,186.183861,1.691595,0.615936,8.503660,14803.093524
121,"combo_from_scan::[0, 0, 0, 1, 1]",combo_from_scan,from previous integer combo scan top 30,2500,1000,1.0,meanrev,"[0.0, 0.0, 0.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 10.0, 10.0]",12,...,1151650.0,9818530.0,12,1479.837802,1353.333333,186.660967,1.711632,0.594291,8.124249,14697.970816
57,"combo_from_scan::[0, 0, 0, 1, 1]",combo_from_scan,from previous integer combo scan top 30,500,1000,1.0,meanrev,"[0.0, 0.0, 0.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 10.0, 10.0]",12,...,4943440.0,6166200.0,12,1196.443410,911.666667,186.036005,1.689437,0.608690,6.355875,14414.472057
91,"combo_from_scan::[0, 0, 0, 1, 1]",combo_from_scan,from previous integer combo scan top 30,1000,1000,1.5,meanrev,"[0.0, 0.0, 0.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 10.0, 10.0]",12,...,3006220.0,5420830.0,12,1916.422373,1926.666667,186.306562,2.110540,0.669500,10.326327,15134.234223
25,"combo_from_scan::[0, 0, 0, 1, 1]",combo_from_scan,from previous integer combo scan top 30,250,1000,1.0,meanrev,"[0.0, 0.0, 0.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 10.0, 10.0]",12,...,3872110.0,5148500.0,12,876.342706,613.333333,185.951509,1.723551,0.575160,4.693435,14094.144979
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,"combo_from_scan::[0, 0, 0, 1, 1]",combo_from_scan,from previous integer combo scan top 30,2500,100,2.0,meanrev,"[0.0, 0.0, 0.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 10.0, 10.0]",12,...,207430.0,530630.0,12,318.444735,368.333333,186.936085,2.518738,0.569890,1.665291,13533.841103
389,"combo_from_scan::[0, 1, 0, 0, 1]",combo_from_scan,from previous integer combo scan top 30,250,100,2.0,meanrev,"[0.0, 1.0, 0.0, 0.0, 1.0]","[0.0, 10.0, 0.0, 0.0, 10.0]",12,...,84020.0,539140.0,12,281.136761,295.000000,172.295401,2.420139,0.592987,1.578533,13496.478137
1411,"combo_from_scan::[1, 1, 0, 0, 0]",combo_from_scan,from previous integer combo scan top 30,250,100,1.5,meanrev,"[1.0, 1.0, 0.0, 0.0, 0.0]","[10.0, 10.0, 0.0, 0.0, 0.0]",12,...,117150.0,775810.0,12,120.482577,143.333333,200.322439,2.067950,0.544545,0.585891,13335.792133
421,"combo_from_scan::[0, 1, 0, 0, 1]",combo_from_scan,from previous integer combo scan top 30,500,100,2.0,meanrev,"[0.0, 1.0, 0.0, 0.0, 1.0]","[0.0, 10.0, 0.0, 0.0, 10.0]",12,...,322750.0,438830.0,12,276.261018,251.666667,172.032447,2.454548,0.571744,1.575148,13491.563934



FAST_BEST_PER_BASKET:


,candidate,family,note,window,horizon,threshold,mode,q_signal,q_trade,days,...,worst_day_exec_pnl,best_day_exec_pnl,positive_days,avg_exec_pnl,median_exec_pnl,avg_cost,avg_abs_z,hit_rate,exec_minus_cost_ratio,stability_score
0,"combo_from_scan::[0, 0, 0, 1, 1]",combo_from_scan,from previous integer combo scan top 30,1000,1000,1.0,meanrev,"[0.0, 0.0, 0.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 10.0, 10.0]",12,...,5.577420e+06,8.409090e+06,12,1584.835710,1243.333333,186.183861,1.691595,0.615936,8.503660,14803.093524
1,"combo_from_scan::[1, 1, 0, 0, 0]",combo_from_scan,from previous integer combo scan top 30,2500,1000,1.0,meanrev,"[1.0, 1.0, 0.0, 0.0, 0.0]","[10.0, 10.0, 0.0, 0.0, 0.0]",12,...,1.040350e+06,5.532840e+06,12,829.363989,1291.666667,200.697212,1.754839,0.573596,3.706973,14046.856345
2,"combo_from_scan::[0, 1, 0, 0, 1]",combo_from_scan,from previous integer combo scan top 30,500,1000,1.0,meanrev,"[0.0, 1.0, 0.0, 0.0, 1.0]","[0.0, 10.0, 0.0, 0.0, 10.0]",12,...,1.134000e+04,5.756180e+06,12,511.953941,316.666667,172.340778,1.716234,0.526049,2.970075,13729.139229
3,"combo_from_scan::[1, 0, 0, 0, 1]",combo_from_scan,from previous integer combo scan top 30,2500,500,2.0,meanrev,"[1.0, 0.0, 0.0, 0.0, 1.0]","[10.0, 0.0, 0.0, 0.0, 10.0]",12,...,3.061400e+05,1.226490e+06,12,697.531478,456.666667,202.280067,2.359783,0.554907,2.819106,13913.500568
4,"combo_from_scan::[1, 1, 0, 0, 1]",combo_from_scan,from previous integer combo scan top 30,2500,1000,1.0,meanrev,"[1.0, 1.0, 0.0, 0.0, 1.0]","[10.0, 10.0, 0.0, 0.0, 10.0]",9,...,4.490590e+06,8.575180e+06,9,1515.073186,1623.333333,287.274276,1.664930,0.637903,5.313609,11433.026457
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,pair_day_ols::PANEL_1X2~PANEL_1X4,pair_day_ols,day-specific beta; diagnostic/in-sample,2500,1000,1.0,meanrev,"[1.0, 0.0, -0.621707, 0.0, 0.0]","[10.0, 0.0, -6.217075, 0.0, 0.0]",1,...,7.505748e+06,7.505748e+06,1,1792.631570,1403.233583,166.076170,1.863255,0.774779,10.794032,2908.462750
61,pair_day_ols::PANEL_2X2~PANEL_2X4,pair_day_ols,day-specific beta; diagnostic/in-sample,2500,1000,1.0,meanrev,"[0.0, 1.0, 0.0, -0.093687, 0.0]","[0.0, 10.0, 0.0, -0.936873, 0.0]",1,...,7.497409e+06,7.497409e+06,1,1729.506129,1437.961075,95.502884,1.660889,0.705190,18.109465,2845.336197
62,pair_day_ols::PANEL_2X2~PANEL_1X4,pair_day_ols,day-specific beta; diagnostic/in-sample,2500,1000,1.0,meanrev,"[0.0, 1.0, -0.159259, 0.0, 0.0]","[0.0, 10.0, -1.592588, 0.0, 0.0]",1,...,7.345910e+06,7.345910e+06,1,1914.493041,1777.926587,100.536497,1.703940,0.816784,19.042766,3030.302696
63,pair_day_ols::PANEL_1X2~PANEL_2X2,pair_day_ols,day-specific beta; diagnostic/in-sample,1000,1000,1.0,meanrev,"[1.0, -0.551062, 0.0, 0.0, 0.0]","[10.0, -5.510619, 0.0, 0.0, 0.0]",1,...,6.853174e+06,6.853174e+06,1,1336.683111,1409.484750,162.168654,1.756313,0.660425,8.242549,2452.423334



Signal counts per day for top configs:


,candidate,window,horizon,threshold,mode,day,event_count,total_exec_pnl,hit_rate,avg_exec_pnl
0,"combo_from_scan::[0, 0, 0, 1, 1]",250,1000,1.0,meanrev,2,20904,15488440.0,0.604095,740.931879
1,"combo_from_scan::[0, 0, 0, 1, 1]",250,1000,1.0,meanrev,3,19748,20594000.0,0.613328,1042.839781
2,"combo_from_scan::[0, 0, 0, 1, 1]",250,1000,1.0,meanrev,4,21056,17797720.0,0.510638,845.256459
3,"combo_from_scan::[0, 0, 0, 1, 1]",250,1000,1.5,meanrev,2,12736,8615440.0,0.624058,676.463568
4,"combo_from_scan::[0, 0, 0, 1, 1]",250,1000,1.5,meanrev,3,11508,14395080.0,0.654501,1250.875912
5,"combo_from_scan::[0, 0, 0, 1, 1]",250,1000,1.5,meanrev,4,12436,9061400.0,0.512705,728.642650
6,"combo_from_scan::[0, 0, 0, 1, 1]",500,1000,1.0,meanrev,2,19768,23129600.0,0.645690,1170.052610
7,"combo_from_scan::[0, 0, 0, 1, 1]",500,1000,1.0,meanrev,3,17140,24664800.0,0.673512,1439.019837
8,"combo_from_scan::[0, 0, 0, 1, 1]",500,1000,1.0,meanrev,4,20172,19773760.0,0.517351,980.257783
9,"combo_from_scan::[0, 0, 0, 1, 1]",500,1000,1.5,meanrev,2,9644,13617440.0,0.674824,1412.011613



Saved:
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_fast/FAST_DAY_RESULTS.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_fast/FAST_BASE_SUMMARY.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_fast/FAST_BEST_PER_BASKET.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_fast/FAST_SIGNALS_TOP_ONLY.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_fast/FAST_panels_report.md

Final runtime: 5.47s


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 20 — CORRECTED PAIR-SPREAD TESTS
# ════════════════════════════════════════════════════════════════════════════

PAIR_CANDIDATES = [
    ("PANEL_1X2", "PANEL_1X4"),
    ("PANEL_1X4", "PANEL_2X4"),
    ("PANEL_2X2", "PANEL_4X4"),
    ("PANEL_1X2", "PANEL_2X4"),
    ("PANEL_2X2", "PANEL_1X4"),
]


def get_pair_params(a: str, b: str, day: int, beta_mode: str):
    dm = mid_wide.loc[day]

    if beta_mode == "day_ols":
        # Diagnostic / in-sample. Useful for relationship discovery but less trustworthy.
        alpha, beta = fit_ols_y_on_x(dm[a], dm[b])
        return alpha, beta

    if beta_mode == "global_ols":
        # Same alpha/beta used across all days.
        alpha, beta = fit_global_pair_ols(a, b)
        return alpha, beta

    if beta_mode == "simple_diff":
        return 0.0, 1.0

    raise ValueError(f"Unknown beta_mode: {beta_mode}")


def backtest_pair_spread_corrected(
    a: str,
    b: str,
    window: int,
    entry_z: float,
    exit_z: float,
    beta_mode: str,
    allow_flip: bool,
    max_hold_steps=None,
):
    day_rows = []

    for day in DAYS:
        dm = mid_wide.loc[day].copy()

        alpha, beta = get_pair_params(a, b, day, beta_mode)

        spread = dm[a] - (alpha + beta * dm[b])
        z = zscore_series(spread, window=window, min_periods=max(50, window // 5)).fillna(0)

        # state = +1 means long spread = buy A, sell B.
        # state = -1 means short spread = sell A, buy B.
        spread_target, state_stats = make_state_targets_from_signal(
            signal=-z,  # signal positive when spread cheap, so buy spread
            entry_z=entry_z,
            exit_z=exit_z,
            pos_limit=POS_LIMIT,
            allow_flip=allow_flip,
            max_hold_steps=max_hold_steps,
        )

        target_a = spread_target.copy()
        target_b = -spread_target.copy()

        result = execute_pair(a, b, day, target_a, target_b)

        result.update({
            "day": day,
            "strategy": "pair_spread",
            "a": a,
            "b": b,
            "window": window,
            "entry_z": entry_z,
            "exit_z": exit_z,
            "beta_mode": beta_mode,
            "alpha": alpha,
            "beta": beta,
            "allow_flip": allow_flip,
            "max_hold_steps_param": max_hold_steps,
            **state_stats,
        })

        day_rows.append(result)

    day_df = pd.DataFrame(day_rows)

    summary = {
        "strategy": "pair_spread",
        "a": a,
        "b": b,
        "window": window,
        "entry_z": entry_z,
        "exit_z": exit_z,
        "beta_mode": beta_mode,
        "allow_flip": allow_flip,
        "max_hold_steps_param": max_hold_steps,
        "total_marked_pnl": day_df["marked_pnl"].sum(),
        "total_liquidated_pnl": day_df["liquidated_pnl"].sum(),
        "avg_day_liquidated_pnl": day_df["liquidated_pnl"].mean(),
        "worst_day_liquidated_pnl": day_df["liquidated_pnl"].min(),
        "best_day_liquidated_pnl": day_df["liquidated_pnl"].max(),
        "positive_days": int((day_df["liquidated_pnl"] > 0).sum()),
        "total_turnover": day_df["turnover"].sum(),
        "total_fills": day_df["fills"].sum(),
        "total_entries": day_df["entries"].sum(),
        "avg_hold_steps": day_df["avg_hold_steps"].mean(),
        "max_hold_steps_seen": day_df["max_hold_steps"].max(),
    }

    return summary, day_df


pair_summaries = []
pair_days = []

for a, b in PAIR_CANDIDATES:
    for beta_mode in ["simple_diff", "global_ols", "day_ols"]:
        for window in [250, 500, 1000, 2500]:
            for entry_z in [1.0, 1.5, 2.0, 2.5]:
                for exit_z in [0.25, 0.5, 0.75]:
                    for allow_flip in [False, True]:
                        for max_hold_steps in [None, 500, 1000, 2500]:
                            summary, day_df = backtest_pair_spread_corrected(
                                a=a,
                                b=b,
                                window=window,
                                entry_z=entry_z,
                                exit_z=exit_z,
                                beta_mode=beta_mode,
                                allow_flip=allow_flip,
                                max_hold_steps=max_hold_steps,
                            )

                            pair_summaries.append(summary)
                            pair_days.append(day_df)

pair_summary_df = pd.DataFrame(pair_summaries)
pair_day_df = pd.concat(pair_days, ignore_index=True)

pair_summary_df = pair_summary_df.sort_values(
    ["positive_days", "total_liquidated_pnl"],
    ascending=[False, False],
)

display(pair_summary_df.head(100))

pair_summary_df.to_csv(FOCUSED_OUT / "pair_focused_summary.csv", index=False)
pair_day_df.to_csv(FOCUSED_OUT / "pair_focused_by_day.csv", index=False)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 21 — ROBUSTNESS FILTER + REPORT
# ════════════════════════════════════════════════════════════════════════════

all_focused = []

s1 = standalone_summary_df.copy()
s1["family"] = "standalone"
all_focused.append(s1)

s2 = pair_summary_df.copy()
s2["family"] = "pair_spread"
all_focused.append(s2)

all_focused_df = pd.concat(all_focused, ignore_index=True, sort=False)

all_focused_df["pnl_per_turnover"] = (
    all_focused_df["total_liquidated_pnl"]
    / all_focused_df["total_turnover"].replace(0, np.nan)
)

# Strict robust set.
robust_focused = all_focused_df[
    (all_focused_df["positive_days"] == 3)
    & (all_focused_df["total_liquidated_pnl"] > 0)
    & (all_focused_df["worst_day_liquidated_pnl"] > 0)
].copy()

robust_focused = robust_focused.sort_values(
    ["total_liquidated_pnl", "pnl_per_turnover"],
    ascending=[False, False],
)

# Less strict set: positive total, not catastrophic day.
candidate_focused = all_focused_df[
    (all_focused_df["total_liquidated_pnl"] > 0)
    & (all_focused_df["positive_days"] >= 2)
].copy()

candidate_focused = candidate_focused.sort_values(
    ["positive_days", "total_liquidated_pnl"],
    ascending=[False, False],
)

display(robust_focused.head(100))
display(candidate_focused.head(100))

all_focused_df.to_csv(FOCUSED_OUT / "ALL_focused_panel_results.csv", index=False)
robust_focused.to_csv(FOCUSED_OUT / "ROBUST_focused_panel_results.csv", index=False)
candidate_focused.to_csv(FOCUSED_OUT / "CANDIDATE_focused_panel_results.csv", index=False)

# Markdown report
report = []

report.append("# Focused Panels Corrected Backtest Report\n")

def add_df(title, df, n=60):
    report.append(f"\n## {title}\n")
    if df is None or len(df) == 0:
        report.append("No rows.\n")
        return

    report.append("```text")
    report.append(df.head(n).to_string(index=False))
    report.append("```\n")

add_df("Robust Focused Results", robust_focused, 80)
add_df("Candidate Focused Results", candidate_focused, 80)
add_df("All Focused Results", all_focused_df.sort_values("total_liquidated_pnl", ascending=False), 100)

report_path = FOCUSED_OUT / "FOCUSED_panels_corrected_report.md"
report_path.write_text("\n".join(report), encoding="utf-8")

print(f"Saved report to: {report_path.resolve()}")
print(f"Saved outputs to: {FOCUSED_OUT.resolve()}")

In [21]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 20 — FAST SCAN CORRECTION / SANITY SUMMARY
# Fixes duplicate-candidate inflation and ranks by per-event edge, not fake total PnL.
# ════════════════════════════════════════════════════════════════════════════

from pathlib import Path
import numpy as np
import pandas as pd

FAST_OUT = Path("outputs_panels_fast")
FAST_OUT.mkdir(exist_ok=True)

required = ["FAST_DAY_RESULTS"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing variables: {missing}. Run the fast scan cell first.")

print("Original FAST_DAY_RESULTS:", FAST_DAY_RESULTS.shape)

# 1) Drop exact duplicate rows caused by repeated candidate coefficient vectors.
dedupe_subset = [
    "candidate", "family", "note", "day", "window", "horizon", "threshold", "mode",
    "q_signal", "q_trade",
]

FAST_DAY_RESULTS_DEDUPED = FAST_DAY_RESULTS.drop_duplicates(subset=dedupe_subset).copy()

print("Deduped FAST_DAY_RESULTS:", FAST_DAY_RESULTS_DEDUPED.shape)
print("Rows removed:", len(FAST_DAY_RESULTS) - len(FAST_DAY_RESULTS_DEDUPED))

# 2) Sanity check: each candidate/config should have at most one row per day.
dupe_check = (
    FAST_DAY_RESULTS_DEDUPED
    .groupby(["candidate", "window", "horizon", "threshold", "mode", "day"])
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)

display(dupe_check.head(20))

if dupe_check["n"].max() > 1:
    print("WARNING: Still duplicated rows exist.")
else:
    print("OK: no duplicate candidate/config/day rows remain.")

# 3) Re-aggregate with signal-discovery metrics, not inflated overlapping total PnL.
FAST_SIGNAL_SUMMARY_CORRECTED = (
    FAST_DAY_RESULTS_DEDUPED
    .groupby(["candidate", "family", "note", "window", "horizon", "threshold", "mode", "q_signal", "q_trade"])
    .agg(
        rows=("day", "count"),
        unique_days=("day", "nunique"),
        active_days=("event_count", lambda x: int((x > 0).sum())),
        event_count=("event_count", "sum"),
        total_exec_pnl=("total_exec_pnl", "sum"),
        total_mid_pnl=("total_mid_pnl", "sum"),
        total_cost=("total_cost", "sum"),
        day_avg_exec_pnl_mean=("avg_exec_pnl", "mean"),
        day_avg_exec_pnl_median=("avg_exec_pnl", "median"),
        worst_day_avg_exec_pnl=("avg_exec_pnl", "min"),
        best_day_avg_exec_pnl=("avg_exec_pnl", "max"),
        positive_avg_edge_days=("avg_exec_pnl", lambda x: int((x > 0).sum())),
        mean_hit_rate=("hit_rate", "mean"),
        min_hit_rate=("hit_rate", "min"),
        mean_avg_cost=("avg_cost", "mean"),
        mean_abs_z=("avg_abs_z", "mean"),
        worst_day_total_exec_pnl=("total_exec_pnl", "min"),
        best_day_total_exec_pnl=("total_exec_pnl", "max"),
        positive_total_pnl_days=("total_exec_pnl", lambda x: int((x > 0).sum())),
    )
    .reset_index()
)

FAST_SIGNAL_SUMMARY_CORRECTED["overall_hit_rate"] = (
    FAST_SIGNAL_SUMMARY_CORRECTED["event_count"]
    .where(FAST_SIGNAL_SUMMARY_CORRECTED["event_count"] > 0)
)

# Better signal score:
# - must be present on all 3 days
# - per-event average should be positive all days
# - hit rate above 50
# - avoid relying purely on huge event count
FAST_SIGNAL_SUMMARY_CORRECTED["signal_quality_score"] = (
    FAST_SIGNAL_SUMMARY_CORRECTED["positive_avg_edge_days"] * 1000
    + FAST_SIGNAL_SUMMARY_CORRECTED["unique_days"] * 100
    + FAST_SIGNAL_SUMMARY_CORRECTED["day_avg_exec_pnl_mean"].fillna(0)
    + (FAST_SIGNAL_SUMMARY_CORRECTED["mean_hit_rate"].fillna(0.5) - 0.5) * 1000
)

# Main ranking: stable positive per-event edge, then stronger avg edge.
FAST_SIGNAL_SUMMARY_CORRECTED = FAST_SIGNAL_SUMMARY_CORRECTED.sort_values(
    [
        "positive_avg_edge_days",
        "unique_days",
        "day_avg_exec_pnl_mean",
        "mean_hit_rate",
        "event_count",
    ],
    ascending=[False, False, False, False, False],
)

# 4) Best per basket under corrected ranking.
FAST_BEST_PER_BASKET_CORRECTED = (
    FAST_SIGNAL_SUMMARY_CORRECTED
    .groupby("candidate", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

# 5) Filter out day_ols as diagnostic-only / in-sample.
FAST_TRADABLE_CORRECTED = FAST_SIGNAL_SUMMARY_CORRECTED[
    ~FAST_SIGNAL_SUMMARY_CORRECTED["family"].eq("pair_day_ols")
].copy()

# 6) More conservative top list:
# all 3 days, positive average edge each day, enough events.
FAST_ROBUST_SIGNAL_CANDIDATES = FAST_TRADABLE_CORRECTED[
    (FAST_TRADABLE_CORRECTED["unique_days"] == 3)
    & (FAST_TRADABLE_CORRECTED["positive_avg_edge_days"] == 3)
    & (FAST_TRADABLE_CORRECTED["event_count"] >= 500)
    & (FAST_TRADABLE_CORRECTED["day_avg_exec_pnl_mean"] > 0)
].copy()

FAST_ROBUST_SIGNAL_CANDIDATES = FAST_ROBUST_SIGNAL_CANDIDATES.sort_values(
    ["day_avg_exec_pnl_mean", "mean_hit_rate", "event_count"],
    ascending=[False, False, False],
)

print("\nTop corrected signal summary:")
display(FAST_SIGNAL_SUMMARY_CORRECTED.head(80))

print("\nTop tradable corrected signals, excluding day_ols:")
display(FAST_TRADABLE_CORRECTED.head(80))

print("\nRobust signal candidates:")
display(FAST_ROBUST_SIGNAL_CANDIDATES.head(80))

print("\nBest per basket corrected:")
display(FAST_BEST_PER_BASKET_CORRECTED.head(80))

# 7) Save outputs.
FAST_DAY_RESULTS_DEDUPED.to_csv(FAST_OUT / "FAST_DAY_RESULTS_DEDUPED.csv", index=False)
FAST_SIGNAL_SUMMARY_CORRECTED.to_csv(FAST_OUT / "FAST_SIGNAL_SUMMARY_CORRECTED.csv", index=False)
FAST_TRADABLE_CORRECTED.to_csv(FAST_OUT / "FAST_TRADABLE_CORRECTED.csv", index=False)
FAST_ROBUST_SIGNAL_CANDIDATES.to_csv(FAST_OUT / "FAST_ROBUST_SIGNAL_CANDIDATES.csv", index=False)
FAST_BEST_PER_BASKET_CORRECTED.to_csv(FAST_OUT / "FAST_BEST_PER_BASKET_CORRECTED.csv", index=False)

report = []
report.append("# Corrected Fast Panels Signal Summary\n")
report.append(f"Original rows: {len(FAST_DAY_RESULTS)}\n")
report.append(f"Deduped rows: {len(FAST_DAY_RESULTS_DEDUPED)}\n")
report.append(f"Rows removed: {len(FAST_DAY_RESULTS) - len(FAST_DAY_RESULTS_DEDUPED)}\n")

def add_table(title, df, n=60):
    report.append(f"\n## {title}\n")
    report.append("```text")
    report.append(df.head(n).to_string(index=False))
    report.append("```\n")

add_table("Robust Signal Candidates", FAST_ROBUST_SIGNAL_CANDIDATES, 80)
add_table("Tradable Corrected Signals", FAST_TRADABLE_CORRECTED, 80)
add_table("Best Per Basket Corrected", FAST_BEST_PER_BASKET_CORRECTED, 80)

report_path = FAST_OUT / "FAST_corrected_panels_report.md"
report_path.write_text("\n".join(report), encoding="utf-8")

print("\nSaved:")
print((FAST_OUT / "FAST_DAY_RESULTS_DEDUPED.csv").resolve())
print((FAST_OUT / "FAST_SIGNAL_SUMMARY_CORRECTED.csv").resolve())
print((FAST_OUT / "FAST_TRADABLE_CORRECTED.csv").resolve())
print((FAST_OUT / "FAST_ROBUST_SIGNAL_CANDIDATES.csv").resolve())
print((FAST_OUT / "FAST_BEST_PER_BASKET_CORRECTED.csv").resolve())
print(report_path.resolve())

Original FAST_DAY_RESULTS: (31488, 20)
Deduped FAST_DAY_RESULTS: (24960, 20)
Rows removed: 6528


,candidate,window,horizon,threshold,mode,day,n
0,"combo_from_scan::[0, 0, 0, 1, 1]",250,100,1.0,breakout,2,1
16624,pair_simple::PANEL_1X2-PANEL_4X4,500,100,2.0,meanrev,3,1
16646,pair_simple::PANEL_1X2-PANEL_4X4,500,250,2.0,breakout,4,1
16645,pair_simple::PANEL_1X2-PANEL_4X4,500,250,2.0,breakout,3,1
16644,pair_simple::PANEL_1X2-PANEL_4X4,500,250,2.0,breakout,2,1
16643,pair_simple::PANEL_1X2-PANEL_4X4,500,250,1.5,meanrev,4,1
16642,pair_simple::PANEL_1X2-PANEL_4X4,500,250,1.5,meanrev,3,1
16641,pair_simple::PANEL_1X2-PANEL_4X4,500,250,1.5,meanrev,2,1
16640,pair_simple::PANEL_1X2-PANEL_4X4,500,250,1.5,breakout,4,1
16639,pair_simple::PANEL_1X2-PANEL_4X4,500,250,1.5,breakout,3,1


OK: no duplicate candidate/config/day rows remain.

Top corrected signal summary:


,candidate,family,note,window,horizon,threshold,mode,q_signal,q_trade,rows,...,positive_avg_edge_days,mean_hit_rate,min_hit_rate,mean_avg_cost,mean_abs_z,worst_day_total_exec_pnl,best_day_total_exec_pnl,positive_total_pnl_days,overall_hit_rate,signal_quality_score
639,"combo_from_scan::[0, 1, 0, 1, 1]",combo_from_scan,from previous integer combo scan top 30,2500,1000,2.5,meanrev,"[0.0, 1.0, 0.0, 1.0, 1.0]","[0.0, 10.0, 0.0, 10.0, 10.0]",3,...,3,0.839979,0.716763,269.996731,2.893756,5.748800e+05,1.069920e+06,3,742,8748.761178
637,"combo_from_scan::[0, 1, 0, 1, 1]",combo_from_scan,from previous integer combo scan top 30,2500,1000,2.0,meanrev,"[0.0, 1.0, 0.0, 1.0, 1.0]","[0.0, 10.0, 0.0, 10.0, 10.0]",3,...,3,0.835540,0.773992,270.820495,2.351532,3.061800e+06,5.607090e+06,3,3239,7806.215157
9343,pair_sum::PANEL_1X2+PANEL_2X4,pair_sum,"all-positive pair basket, not market neutral",2500,1000,2.5,meanrev,"[1.0, 0.0, 0.0, 1.0, 0.0]","[10.0, 0.0, 0.0, 10.0, 0.0]",3,...,3,0.747168,0.521649,216.535677,2.801301,7.438600e+05,1.751600e+06,3,1437,6416.917594
635,"combo_from_scan::[0, 1, 0, 1, 1]",combo_from_scan,from previous integer combo scan top 30,2500,1000,1.5,meanrev,"[0.0, 1.0, 0.0, 1.0, 1.0]","[0.0, 10.0, 0.0, 10.0, 10.0]",3,...,3,0.731000,0.698886,271.468617,1.980881,5.380610e+06,1.155554e+07,3,8271,6270.827832
7039,pair_global_ols::PANEL_1X4~PANEL_2X4,pair_global_ols,global beta=-0.803361,2500,1000,2.5,meanrev,"[0.0, 0.0, 1.0, 0.803361, 0.0]","[0.0, 0.0, 10.0, 8.033612, 0.0]",3,...,3,0.660236,0.525424,162.882738,2.752447,1.375458e+03,1.172880e+06,3,660,6180.930146
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1647,"combo_from_scan::[1, 1, 0, 0, 1]",combo_from_scan,from previous integer combo scan top 30,2500,250,2.5,meanrev,"[1.0, 1.0, 0.0, 0.0, 1.0]","[10.0, 10.0, 0.0, 0.0, 10.0]",3,...,3,0.752689,0.526627,286.821648,2.795805,3.994000e+04,6.272500e+05,3,816,4825.777215
7158,pair_global_ols::PANEL_1X4~PANEL_4X4,pair_global_ols,global beta=-0.415900,2500,500,2.5,breakout,"[0.0, 0.0, 1.0, 0.0, 0.4159]","[0.0, 0.0, 10.0, 0.0, 4.159001]",3,...,3,0.657761,0.513444,119.799023,3.051907,2.915462e+05,1.337878e+06,3,2148,4722.147370
7541,pair_global_ols::PANEL_2X2~PANEL_4X4,pair_global_ols,global beta=-0.729471,2500,500,2.0,meanrev,"[0.0, 1.0, 0.0, 0.0, 0.729471]","[0.0, 10.0, 0.0, 0.0, 7.294713]",3,...,3,0.690378,0.418764,148.148089,2.379112,2.594850e+05,2.942037e+06,3,3753,4719.978331
1591,"combo_from_scan::[1, 1, 0, 0, 1]",combo_from_scan,from previous integer combo scan top 30,500,500,2.5,meanrev,"[1.0, 1.0, 0.0, 0.0, 1.0]","[10.0, 10.0, 0.0, 0.0, 10.0]",3,...,3,0.589286,0.572289,287.169938,2.862426,2.365700e+05,8.375900e+05,3,1294,4590.299721



Top tradable corrected signals, excluding day_ols:


...


Robust signal candidates:


,candidate,family,note,window,horizon,threshold,mode,q_signal,q_trade,rows,...,positive_avg_edge_days,mean_hit_rate,min_hit_rate,mean_avg_cost,mean_abs_z,worst_day_total_exec_pnl,best_day_total_exec_pnl,positive_total_pnl_days,overall_hit_rate,signal_quality_score
639,"combo_from_scan::[0, 1, 0, 1, 1]",combo_from_scan,from previous integer combo scan top 30,2500,1000,2.5,meanrev,"[0.0, 1.0, 0.0, 1.0, 1.0]","[0.0, 10.0, 0.0, 10.0, 10.0]",3,...,3,0.839979,0.716763,269.996731,2.893756,5.748800e+05,1.069920e+06,3,742,8748.761178
637,"combo_from_scan::[0, 1, 0, 1, 1]",combo_from_scan,from previous integer combo scan top 30,2500,1000,2.0,meanrev,"[0.0, 1.0, 0.0, 1.0, 1.0]","[0.0, 10.0, 0.0, 10.0, 10.0]",3,...,3,0.835540,0.773992,270.820495,2.351532,3.061800e+06,5.607090e+06,3,3239,7806.215157
9343,pair_sum::PANEL_1X2+PANEL_2X4,pair_sum,"all-positive pair basket, not market neutral",2500,1000,2.5,meanrev,"[1.0, 0.0, 0.0, 1.0, 0.0]","[10.0, 0.0, 0.0, 10.0, 0.0]",3,...,3,0.747168,0.521649,216.535677,2.801301,7.438600e+05,1.751600e+06,3,1437,6416.917594
635,"combo_from_scan::[0, 1, 0, 1, 1]",combo_from_scan,from previous integer combo scan top 30,2500,1000,1.5,meanrev,"[0.0, 1.0, 0.0, 1.0, 1.0]","[0.0, 10.0, 0.0, 10.0, 10.0]",3,...,3,0.731000,0.698886,271.468617,1.980881,5.380610e+06,1.155554e+07,3,8271,6270.827832
7039,pair_global_ols::PANEL_1X4~PANEL_2X4,pair_global_ols,global beta=-0.803361,2500,1000,2.5,meanrev,"[0.0, 0.0, 1.0, 0.803361, 0.0]","[0.0, 0.0, 10.0, 8.033612, 0.0]",3,...,3,0.660236,0.525424,162.882738,2.752447,1.375458e+03,1.172880e+06,3,660,6180.930146
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1647,"combo_from_scan::[1, 1, 0, 0, 1]",combo_from_scan,from previous integer combo scan top 30,2500,250,2.5,meanrev,"[1.0, 1.0, 0.0, 0.0, 1.0]","[10.0, 10.0, 0.0, 0.0, 10.0]",3,...,3,0.752689,0.526627,286.821648,2.795805,3.994000e+04,6.272500e+05,3,816,4825.777215
7158,pair_global_ols::PANEL_1X4~PANEL_4X4,pair_global_ols,global beta=-0.415900,2500,500,2.5,breakout,"[0.0, 0.0, 1.0, 0.0, 0.4159]","[0.0, 0.0, 10.0, 0.0, 4.159001]",3,...,3,0.657761,0.513444,119.799023,3.051907,2.915462e+05,1.337878e+06,3,2148,4722.147370
7541,pair_global_ols::PANEL_2X2~PANEL_4X4,pair_global_ols,global beta=-0.729471,2500,500,2.0,meanrev,"[0.0, 1.0, 0.0, 0.0, 0.729471]","[0.0, 10.0, 0.0, 0.0, 7.294713]",3,...,3,0.690378,0.418764,148.148089,2.379112,2.594850e+05,2.942037e+06,3,3753,4719.978331
1591,"combo_from_scan::[1, 1, 0, 0, 1]",combo_from_scan,from previous integer combo scan top 30,500,500,2.5,meanrev,"[1.0, 1.0, 0.0, 0.0, 1.0]","[10.0, 10.0, 0.0, 0.0, 10.0]",3,...,3,0.589286,0.572289,287.169938,2.862426,2.365700e+05,8.375900e+05,3,1294,4590.299721



Best per basket corrected:


,candidate,family,note,window,horizon,threshold,mode,q_signal,q_trade,rows,...,positive_avg_edge_days,mean_hit_rate,min_hit_rate,mean_avg_cost,mean_abs_z,worst_day_total_exec_pnl,best_day_total_exec_pnl,positive_total_pnl_days,overall_hit_rate,signal_quality_score
0,"combo_from_scan::[0, 1, 0, 1, 1]",combo_from_scan,from previous integer combo scan top 30,2500,1000,2.5,meanrev,"[0.0, 1.0, 0.0, 1.0, 1.0]","[0.0, 10.0, 0.0, 10.0, 10.0]",3,...,3,0.839979,0.716763,269.996731,2.893756,5.748800e+05,1.069920e+06,3,742,8748.761178
1,pair_sum::PANEL_1X2+PANEL_2X4,pair_sum,"all-positive pair basket, not market neutral",2500,1000,2.5,meanrev,"[1.0, 0.0, 0.0, 1.0, 0.0]","[10.0, 0.0, 0.0, 10.0, 0.0]",3,...,3,0.747168,0.521649,216.535677,2.801301,7.438600e+05,1.751600e+06,3,1437,6416.917594
2,pair_global_ols::PANEL_1X4~PANEL_2X4,pair_global_ols,global beta=-0.803361,2500,1000,2.5,meanrev,"[0.0, 0.0, 1.0, 0.803361, 0.0]","[0.0, 0.0, 10.0, 8.033612, 0.0]",3,...,3,0.660236,0.525424,162.882738,2.752447,1.375458e+03,1.172880e+06,3,660,6180.930146
3,pair_simple::PANEL_1X4-PANEL_2X4,pair_simple,simple pair difference,2500,1000,2.5,breakout,"[0.0, 0.0, 1.0, -1.0, 0.0]","[0.0, 0.0, 10.0, -10.0, 0.0]",3,...,3,0.676556,0.405882,183.495993,2.819516,1.107400e+05,3.186860e+06,3,1609,6148.945378
4,single::PANEL_4X4,single,single product,2500,1000,2.5,meanrev,"[0.0, 0.0, 0.0, 0.0, 1.0]","[0.0, 0.0, 0.0, 0.0, 10.0]",3,...,3,0.859745,0.804878,85.873917,2.788425,2.658000e+05,9.273500e+05,3,917,5735.339877
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,pair_day_ols::PANEL_2X2~PANEL_1X4,pair_day_ols,day-specific beta; diagnostic/in-sample,2500,1000,2.5,breakout,"[0.0, 1.0, -0.046815, 0.0, 0.0]","[0.0, 10.0, -0.468152, 0.0, 0.0]",1,...,1,0.696872,0.696872,90.936464,2.825525,2.931489e+06,2.931489e+06,1,927,4459.210921
61,pair_day_ols::PANEL_1X2~PANEL_1X4,pair_day_ols,day-specific beta; diagnostic/in-sample,1000,1000,2.5,meanrev,"[1.0, 0.0, 0.45531, 0.0, 0.0]","[10.0, 0.0, 4.553096, 0.0, 0.0]",1,...,1,0.525000,0.525000,162.067142,2.778110,1.256645e+06,1.256645e+06,1,400,4266.613578
62,pair_day_ols::PANEL_1X2~PANEL_4X4,pair_day_ols,day-specific beta; diagnostic/in-sample,2500,1000,2.5,meanrev,"[1.0, 0.0, 0.0, 0.0, 0.609118]","[10.0, 0.0, 0.0, 0.0, 6.091185]",1,...,1,0.894472,0.894472,171.486975,2.773407,1.239970e+06,1.239970e+06,1,398,4609.974994
63,pair_day_ols::PANEL_1X4~PANEL_4X4,pair_day_ols,day-specific beta; diagnostic/in-sample,2500,1000,2.0,breakout,"[0.0, 0.0, 1.0, 0.0, 0.962038]","[0.0, 0.0, 10.0, 0.0, 9.62038]",1,...,1,0.761040,0.761040,165.271876,2.568661,5.615340e+06,5.615340e+06,1,2038,4116.359172



Saved:
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_fast/FAST_DAY_RESULTS_DEDUPED.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_fast/FAST_SIGNAL_SUMMARY_CORRECTED.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_fast/FAST_TRADABLE_CORRECTED.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_fast/FAST_ROBUST_SIGNAL_CANDIDATES.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_fast/FAST_BEST_PER_BASKET_CORRECTED.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_fast/FAST_corrected_panels_report.md


In [22]:
# ════════════════════════════════════════════════════════════════════════════
# STAGE 2 — FOCUSED PANELS STRATEGY OPTIMISER
# Non-overlapping entries, bid/ask execution, realistic exit logic.
# ════════════════════════════════════════════════════════════════════════════

import time
from pathlib import Path
import numpy as np
import pandas as pd

STAGE2_OUT = Path("outputs_panels_stage2")
STAGE2_OUT.mkdir(exist_ok=True)

required = ["panel_prices", "PANEL_PRODUCTS", "DAYS"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing variables: {missing}. Run Panels setup cells first.")

POS_LIMIT = 10

STAGE2_WINDOWS = [250, 500, 1000, 2500]
STAGE2_ENTRY_ZS = [1.5, 2.0, 2.5, 3.0]
STAGE2_EXIT_ZS = [None, 0.25, 0.5, 0.75]  # None = fixed max-hold exit only
STAGE2_MAX_HOLDS = [250, 500, 1000, 1500, 2500]

start = time.perf_counter()

# ───────────────────────────────────────────────────────────────────────────
# 1. Pivot once
# ───────────────────────────────────────────────────────────────────────────

mid_stage2 = (
    panel_prices
    .pivot_table(index=["file_day", "timestamp"], columns="product", values="mid_price", aggfunc="last")
    .sort_index()
    .reindex(columns=PANEL_PRODUCTS)
)

bid_stage2 = (
    panel_prices
    .pivot_table(index=["file_day", "timestamp"], columns="product", values="bid_price_1", aggfunc="last")
    .sort_index()
    .reindex(columns=PANEL_PRODUCTS)
)

ask_stage2 = (
    panel_prices
    .pivot_table(index=["file_day", "timestamp"], columns="product", values="ask_price_1", aggfunc="last")
    .sort_index()
    .reindex(columns=PANEL_PRODUCTS)
)

valid = ~(mid_stage2.isna().any(axis=1) | bid_stage2.isna().any(axis=1) | ask_stage2.isna().any(axis=1))

mid_stage2 = mid_stage2.loc[valid]
bid_stage2 = bid_stage2.loc[valid]
ask_stage2 = ask_stage2.loc[valid]

assert list(mid_stage2.columns) == PANEL_PRODUCTS
assert list(bid_stage2.columns) == PANEL_PRODUCTS
assert list(ask_stage2.columns) == PANEL_PRODUCTS

print("Aligned shape:", mid_stage2.shape)
print("Products:", PANEL_PRODUCTS)

product_to_idx = {p: i for i, p in enumerate(PANEL_PRODUCTS)}

# ───────────────────────────────────────────────────────────────────────────
# 2. Convert each day to NumPy once
# ───────────────────────────────────────────────────────────────────────────

STAGE2_DAY_BASE = {}

for day in DAYS:
    X_df = mid_stage2.loc[day]
    B_df = bid_stage2.loc[day]
    A_df = ask_stage2.loc[day]

    assert X_df.index.min() >= 0
    assert X_df.index.max() <= 999900
    assert X_df.shape == B_df.shape == A_df.shape

    STAGE2_DAY_BASE[day] = {
        "timestamps": X_df.index.to_numpy(),
        "X": X_df.to_numpy(float),
        "B": B_df.to_numpy(float),
        "A": A_df.to_numpy(float),
    }

    print(f"Day {day}: {STAGE2_DAY_BASE[day]['X'].shape}")


# ───────────────────────────────────────────────────────────────────────────
# 3. Helpers
# ───────────────────────────────────────────────────────────────────────────

def make_q(coefs):
    q = np.zeros(len(PANEL_PRODUCTS), dtype=float)
    for p, c in coefs.items():
        q[product_to_idx[p]] = float(c)
    return q


def scale_q_to_position_limit(q):
    q = np.asarray(q, dtype=float)

    nz = np.abs(q[np.abs(q) > 1e-12])
    if len(nz) == 0:
        return q.copy()

    if np.allclose(q, np.round(q), atol=1e-9):
        q_int = np.round(q).astype(int)
        max_abs = np.max(np.abs(q_int[q_int != 0]))
        units = max(1, POS_LIMIT // max_abs)
        return q_int.astype(float) * units

    return q / np.max(np.abs(q)) * POS_LIMIT


def rolling_z_np(spread, window):
    s = pd.Series(spread)
    min_periods = max(50, window // 5)
    mean = s.rolling(window, min_periods=min_periods).mean().to_numpy(float)
    std = s.rolling(window, min_periods=min_periods).std().to_numpy(float)
    z = (spread - mean) / std
    z[~np.isfinite(z)] = 0.0
    return z


def pnl_for_trades(X, B, A, q_trade, entry_idx, exit_idx, side):
    """
    Vectorised PnL for selected non-overlapping trades.

    q_trade is the base basket.
    side = +1 means long basket.
    side = -1 means short basket.
    """
    entry_idx = np.asarray(entry_idx, dtype=int)
    exit_idx = np.asarray(exit_idx, dtype=int)
    side = np.asarray(side, dtype=float)

    assert len(entry_idx) == len(exit_idx) == len(side)

    if len(entry_idx) == 0:
        return {
            "exec_pnl": np.array([]),
            "mid_pnl": np.array([]),
            "cost": np.array([]),
            "target": np.empty((0, len(q_trade))),
        }

    targets = side[:, None] * q_trade.reshape(1, -1)

    X0 = X[entry_idx]
    X1 = X[exit_idx]
    B0 = B[entry_idx]
    A0 = A[entry_idx]
    B1 = B[exit_idx]
    A1 = A[exit_idx]

    entry_px = np.where(targets > 0, A0, B0)
    exit_px = np.where(targets > 0, B1, A1)

    exec_pnl = ((exit_px - entry_px) * targets).sum(axis=1)
    mid_pnl = ((X1 - X0) * targets).sum(axis=1)
    cost = mid_pnl - exec_pnl

    if len(cost):
        assert np.nanmin(cost) >= -1e-7, f"Cost identity failed. min cost={np.nanmin(cost)}"

    return {
        "exec_pnl": exec_pnl,
        "mid_pnl": mid_pnl,
        "cost": cost,
        "target": targets,
    }


def select_non_overlapping_trades(
    z,
    mode,
    entry_z,
    exit_z,
    max_hold,
):
    """
    Greedy non-overlapping trade selector.

    mode:
      meanrev  -> z high short, z low long
      breakout -> z high long, z low short

    exit_z:
      None -> fixed max_hold exit
      number -> exit earlier if abs(z) <= exit_z
    """
    n = len(z)

    entry_candidates = np.flatnonzero(np.abs(z) >= entry_z)
    entry_candidates = entry_candidates[entry_candidates + max_hold < n]

    entries = []
    exits = []
    sides = []
    holds = []

    last_exit = -1

    for i in entry_candidates:
        if i <= last_exit:
            continue

        zi = z[i]
        if zi == 0:
            continue

        if mode == "meanrev":
            side = -np.sign(zi)
        elif mode == "breakout":
            side = np.sign(zi)
        else:
            raise ValueError(mode)

        hard_exit = min(i + max_hold, n - 1)
        exit_i = hard_exit

        if exit_z is not None:
            search_region = np.abs(z[i + 1:hard_exit + 1])
            found = np.flatnonzero(search_region <= exit_z)
            if len(found) > 0:
                exit_i = i + 1 + int(found[0])

        if exit_i <= i:
            continue

        entries.append(i)
        exits.append(exit_i)
        sides.append(side)
        holds.append(exit_i - i)

        last_exit = exit_i

    return (
        np.asarray(entries, dtype=int),
        np.asarray(exits, dtype=int),
        np.asarray(sides, dtype=float),
        np.asarray(holds, dtype=int),
    )


# ───────────────────────────────────────────────────────────────────────────
# 4. Focus candidates
# ───────────────────────────────────────────────────────────────────────────

STAGE2_CANDIDATES = [
    {
        "name": "single_PANEL_4X4_meanrev",
        "family": "single",
        "q_signal": make_q({"PANEL_4X4": 1}),
        "q_trade": scale_q_to_position_limit(make_q({"PANEL_4X4": 1})),
        "mode": "meanrev",
        "note": "Best clean single-product signal from corrected fast scan.",
    },
    {
        "name": "basket_2X2_2X4_4X4_meanrev",
        "family": "positive_basket",
        "q_signal": make_q({"PANEL_2X2": 1, "PANEL_2X4": 1, "PANEL_4X4": 1}),
        "q_trade": scale_q_to_position_limit(make_q({"PANEL_2X2": 1, "PANEL_2X4": 1, "PANEL_4X4": 1})),
        "mode": "meanrev",
        "note": "Strongest overall corrected basket signal.",
    },
    {
        "name": "basket_2X4_4X4_meanrev",
        "family": "positive_basket",
        "q_signal": make_q({"PANEL_2X4": 1, "PANEL_4X4": 1}),
        "q_trade": scale_q_to_position_limit(make_q({"PANEL_2X4": 1, "PANEL_4X4": 1})),
        "mode": "meanrev",
        "note": "Cleaner two-product large-panel basket.",
    },
    {
        "name": "spread_2x_2X2_minus_2X4_meanrev",
        "family": "geometry_spread",
        "q_signal": make_q({"PANEL_2X2": 2, "PANEL_2X4": -1}),
        "q_trade": scale_q_to_position_limit(make_q({"PANEL_2X2": 2, "PANEL_2X4": -1})),
        "mode": "meanrev",
        "note": "Explainable geometry relation: 2×2X2 vs 2X4.",
    },
    {
        "name": "spread_2X2_minus_2X4_meanrev",
        "family": "relative_spread",
        "q_signal": make_q({"PANEL_2X2": 1, "PANEL_2X4": -1}),
        "q_trade": scale_q_to_position_limit(make_q({"PANEL_2X2": 1, "PANEL_2X4": -1})),
        "mode": "meanrev",
        "note": "Simpler 2X2 vs 2X4 relative spread.",
    },
    {
        "name": "spread_1X4_minus_2X4_breakout",
        "family": "relative_spread",
        "q_signal": make_q({"PANEL_1X4": 1, "PANEL_2X4": -1}),
        "q_trade": scale_q_to_position_limit(make_q({"PANEL_1X4": 1, "PANEL_2X4": -1})),
        "mode": "breakout",
        "note": "Higher-risk continuation spread.",
    },
    {
        "name": "single_PANEL_1X4_breakout",
        "family": "single",
        "q_signal": make_q({"PANEL_1X4": 1}),
        "q_trade": scale_q_to_position_limit(make_q({"PANEL_1X4": 1})),
        "mode": "breakout",
        "note": "Earlier stateful scan favourite.",
    },
    {
        "name": "single_PANEL_2X4_meanrev",
        "family": "single",
        "q_signal": make_q({"PANEL_2X4": 1}),
        "q_trade": scale_q_to_position_limit(make_q({"PANEL_2X4": 1})),
        "mode": "meanrev",
        "note": "Earlier predictive scan favourite.",
    },
]

print("Stage 2 candidates:")
for c in STAGE2_CANDIDATES:
    print(c["name"], "q_signal=", c["q_signal"], "q_trade=", c["q_trade"], "mode=", c["mode"])


# ───────────────────────────────────────────────────────────────────────────
# 5. Run optimiser
# ───────────────────────────────────────────────────────────────────────────

day_rows = []
trade_rows = []

total_configs = (
    len(STAGE2_CANDIDATES)
    * len(STAGE2_WINDOWS)
    * len(STAGE2_ENTRY_ZS)
    * len(STAGE2_EXIT_ZS)
    * len(STAGE2_MAX_HOLDS)
)

print(f"Total candidate/config combinations: {total_configs}")

config_count = 0

for cand_i, cand in enumerate(STAGE2_CANDIDATES, start=1):
    print(f"\n[{time.perf_counter() - start:.2f}s] Candidate {cand_i}/{len(STAGE2_CANDIDATES)}: {cand['name']}")

    q_signal = cand["q_signal"]
    q_trade = cand["q_trade"]
    mode = cand["mode"]

    for window in STAGE2_WINDOWS:
        for entry_z in STAGE2_ENTRY_ZS:
            for exit_z in STAGE2_EXIT_ZS:
                for max_hold in STAGE2_MAX_HOLDS:
                    config_count += 1

                    for day in DAYS:
                        base = STAGE2_DAY_BASE[day]
                        ts = base["timestamps"]
                        X = base["X"]
                        B = base["B"]
                        A = base["A"]

                        spread = X @ q_signal
                        z = rolling_z_np(spread, window)

                        entry_idx, exit_idx, side, holds = select_non_overlapping_trades(
                            z=z,
                            mode=mode,
                            entry_z=entry_z,
                            exit_z=exit_z,
                            max_hold=max_hold,
                        )

                        pnl = pnl_for_trades(
                            X=X,
                            B=B,
                            A=A,
                            q_trade=q_trade,
                            entry_idx=entry_idx,
                            exit_idx=exit_idx,
                            side=side,
                        )

                        exec_pnl = pnl["exec_pnl"]
                        mid_pnl = pnl["mid_pnl"]
                        cost = pnl["cost"]

                        trade_count = len(exec_pnl)

                        row = {
                            "candidate": cand["name"],
                            "family": cand["family"],
                            "note": cand["note"],
                            "mode": mode,
                            "day": day,
                            "window": window,
                            "entry_z": entry_z,
                            "exit_z": "fixed" if exit_z is None else exit_z,
                            "max_hold": max_hold,
                            "trade_count": trade_count,
                            "total_exec_pnl": float(np.sum(exec_pnl)) if trade_count else 0.0,
                            "total_mid_pnl": float(np.sum(mid_pnl)) if trade_count else 0.0,
                            "total_cost": float(np.sum(cost)) if trade_count else 0.0,
                            "avg_exec_pnl": float(np.mean(exec_pnl)) if trade_count else np.nan,
                            "median_exec_pnl": float(np.median(exec_pnl)) if trade_count else np.nan,
                            "hit_rate": float(np.mean(exec_pnl > 0)) if trade_count else np.nan,
                            "avg_hold": float(np.mean(holds)) if trade_count else 0.0,
                            "max_hold_seen": int(np.max(holds)) if trade_count else 0,
                            "q_signal": str(np.round(q_signal, 6).tolist()),
                            "q_trade": str(np.round(q_trade, 6).tolist()),
                        }

                        day_rows.append(row)

                        if trade_count:
                            tr = pd.DataFrame({
                                "candidate": cand["name"],
                                "family": cand["family"],
                                "mode": mode,
                                "day": day,
                                "window": window,
                                "entry_z": entry_z,
                                "exit_z": "fixed" if exit_z is None else exit_z,
                                "max_hold": max_hold,
                                "entry_idx": entry_idx,
                                "exit_idx": exit_idx,
                                "entry_ts": ts[entry_idx],
                                "exit_ts": ts[exit_idx],
                                "side": side,
                                "hold": holds,
                                "exec_pnl": exec_pnl,
                                "mid_pnl": mid_pnl,
                                "cost": cost,
                            })
                            trade_rows.append(tr)

    print(f"Finished {cand['name']} at {time.perf_counter() - start:.2f}s")

STAGE2_DAY_RESULTS = pd.DataFrame(day_rows)
STAGE2_TRADE_LOG = pd.concat(trade_rows, ignore_index=True) if trade_rows else pd.DataFrame()

print("\nSTAGE2_DAY_RESULTS:", STAGE2_DAY_RESULTS.shape)
print("STAGE2_TRADE_LOG:", STAGE2_TRADE_LOG.shape)

display(STAGE2_DAY_RESULTS.head())
display(STAGE2_TRADE_LOG.head())


# ───────────────────────────────────────────────────────────────────────────
# 6. Summaries
# ───────────────────────────────────────────────────────────────────────────

STAGE2_SUMMARY = (
    STAGE2_DAY_RESULTS
    .groupby([
        "candidate", "family", "note", "mode", "window", "entry_z", "exit_z", "max_hold", "q_signal", "q_trade"
    ])
    .agg(
        days=("day", "count"),
        active_days=("trade_count", lambda x: int((x > 0).sum())),
        trade_count=("trade_count", "sum"),
        total_exec_pnl=("total_exec_pnl", "sum"),
        total_mid_pnl=("total_mid_pnl", "sum"),
        total_cost=("total_cost", "sum"),
        avg_day_exec_pnl=("total_exec_pnl", "mean"),
        worst_day_exec_pnl=("total_exec_pnl", "min"),
        best_day_exec_pnl=("total_exec_pnl", "max"),
        positive_days=("total_exec_pnl", lambda x: int((x > 0).sum())),
        avg_trade_pnl=("avg_exec_pnl", "mean"),
        median_trade_pnl=("median_exec_pnl", "mean"),
        mean_hit_rate=("hit_rate", "mean"),
        min_hit_rate=("hit_rate", "min"),
        avg_hold=("avg_hold", "mean"),
        max_hold_seen=("max_hold_seen", "max"),
    )
    .reset_index()
)

STAGE2_SUMMARY["pnl_per_trade"] = (
    STAGE2_SUMMARY["total_exec_pnl"] / STAGE2_SUMMARY["trade_count"].replace(0, np.nan)
)

STAGE2_SUMMARY["pnl_per_cost"] = (
    STAGE2_SUMMARY["total_exec_pnl"] / STAGE2_SUMMARY["total_cost"].replace(0, np.nan)
)

STAGE2_SUMMARY["one_day_dependency"] = (
    STAGE2_SUMMARY["best_day_exec_pnl"] / STAGE2_SUMMARY["total_exec_pnl"].replace(0, np.nan)
)

# Robust filter:
# - at least one trade on all 3 days
# - positive PnL on all 3 days
# - not solely one-day dependent
# - enough total trades, but not too many
STAGE2_ROBUST = STAGE2_SUMMARY[
    (STAGE2_SUMMARY["days"] == len(DAYS))
    & (STAGE2_SUMMARY["active_days"] == len(DAYS))
    & (STAGE2_SUMMARY["positive_days"] == len(DAYS))
    & (STAGE2_SUMMARY["total_exec_pnl"] > 0)
    & (STAGE2_SUMMARY["trade_count"] >= 3)
    & (STAGE2_SUMMARY["one_day_dependency"] <= 0.80)
].copy()

STAGE2_ROBUST = STAGE2_ROBUST.sort_values(
    ["total_exec_pnl", "pnl_per_trade", "mean_hit_rate"],
    ascending=[False, False, False],
)

STAGE2_SUMMARY = STAGE2_SUMMARY.sort_values(
    ["positive_days", "total_exec_pnl", "pnl_per_trade"],
    ascending=[False, False, False],
)

STAGE2_BEST_BY_CANDIDATE = (
    STAGE2_ROBUST
    .groupby("candidate", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

print("\nTop STAGE2_SUMMARY:")
display(STAGE2_SUMMARY.head(100))

print("\nSTAGE2_ROBUST:")
display(STAGE2_ROBUST.head(100))

print("\nSTAGE2_BEST_BY_CANDIDATE:")
display(STAGE2_BEST_BY_CANDIDATE)


# ───────────────────────────────────────────────────────────────────────────
# 7. Save outputs
# ───────────────────────────────────────────────────────────────────────────

STAGE2_DAY_RESULTS.to_csv(STAGE2_OUT / "STAGE2_DAY_RESULTS.csv", index=False)
STAGE2_SUMMARY.to_csv(STAGE2_OUT / "STAGE2_SUMMARY.csv", index=False)
STAGE2_ROBUST.to_csv(STAGE2_OUT / "STAGE2_ROBUST.csv", index=False)
STAGE2_BEST_BY_CANDIDATE.to_csv(STAGE2_OUT / "STAGE2_BEST_BY_CANDIDATE.csv", index=False)
STAGE2_TRADE_LOG.to_csv(STAGE2_OUT / "STAGE2_TRADE_LOG.csv", index=False)

report = []
report.append("# Stage 2 Focused Panels Strategy Optimiser\n")
report.append(f"Candidates tested: {len(STAGE2_CANDIDATES)}\n")
report.append(f"Windows: {STAGE2_WINDOWS}\n")
report.append(f"Entry z: {STAGE2_ENTRY_ZS}\n")
report.append(f"Exit z: {STAGE2_EXIT_ZS}\n")
report.append(f"Max holds: {STAGE2_MAX_HOLDS}\n")
report.append(f"Runtime: {time.perf_counter() - start:.2f}s\n")

def add_table(title, df, n=60):
    report.append(f"\n## {title}\n")
    if df is None or len(df) == 0:
        report.append("No rows.\n")
        return
    report.append("```text")
    report.append(df.head(n).to_string(index=False))
    report.append("```\n")

add_table("STAGE2_ROBUST", STAGE2_ROBUST, 80)
add_table("STAGE2_BEST_BY_CANDIDATE", STAGE2_BEST_BY_CANDIDATE, 80)
add_table("Top STAGE2_SUMMARY", STAGE2_SUMMARY, 100)

report_path = STAGE2_OUT / "STAGE2_panels_report.md"
report_path.write_text("\n".join(report), encoding="utf-8")

print("\nSaved:")
print((STAGE2_OUT / "STAGE2_DAY_RESULTS.csv").resolve())
print((STAGE2_OUT / "STAGE2_SUMMARY.csv").resolve())
print((STAGE2_OUT / "STAGE2_ROBUST.csv").resolve())
print((STAGE2_OUT / "STAGE2_BEST_BY_CANDIDATE.csv").resolve())
print((STAGE2_OUT / "STAGE2_TRADE_LOG.csv").resolve())
print(report_path.resolve())
print(f"\nFinal runtime: {time.perf_counter() - start:.2f}s")

Aligned shape: (30000, 5)
Products: ['PANEL_1X2', 'PANEL_2X2', 'PANEL_1X4', 'PANEL_2X4', 'PANEL_4X4']
Day 2: (10000, 5)
Day 3: (10000, 5)
Day 4: (10000, 5)
Stage 2 candidates:
single_PANEL_4X4_meanrev q_signal= [0. 0. 0. 0. 1.] q_trade= [ 0.  0.  0.  0. 10.] mode= meanrev
basket_2X2_2X4_4X4_meanrev q_signal= [0. 1. 0. 1. 1.] q_trade= [ 0. 10.  0. 10. 10.] mode= meanrev
basket_2X4_4X4_meanrev q_signal= [0. 0. 0. 1. 1.] q_trade= [ 0.  0.  0. 10. 10.] mode= meanrev
spread_2x_2X2_minus_2X4_meanrev q_signal= [ 0.  2.  0. -1.  0.] q_trade= [ 0. 10.  0. -5.  0.] mode= meanrev
spread_2X2_minus_2X4_meanrev q_signal= [ 0.  1.  0. -1.  0.] q_trade= [  0.  10.   0. -10.   0.] mode= meanrev
spread_1X4_minus_2X4_breakout q_signal= [ 0.  0.  1. -1.  0.] q_trade= [  0.   0.  10. -10.   0.] mode= breakout
single_PANEL_1X4_breakout q_signal= [0. 0. 1. 0. 0.] q_trade= [ 0.  0. 10.  0.  0.] mode= breakout
single_PANEL_2X4_meanrev q_signal= [0. 0. 0. 1. 0.] q_trade= [ 0.  0.  0. 10.  0.] mode= meanrev
Tota

,candidate,family,note,mode,day,window,entry_z,exit_z,max_hold,trade_count,total_exec_pnl,total_mid_pnl,total_cost,avg_exec_pnl,median_exec_pnl,hit_rate,avg_hold,max_hold_seen,q_signal,q_trade
0,single_PANEL_4X4_meanrev,single,Best clean single-product signal from correcte...,meanrev,2,250,1.5,fixed,250,35,-6550.0,-3525.0,3025.0,-187.142857,-300.0,0.400000,250.0,250,"[0.0, 0.0, 0.0, 0.0, 1.0]","[0.0, 0.0, 0.0, 0.0, 10.0]"
1,single_PANEL_4X4_meanrev,single,Best clean single-product signal from correcte...,meanrev,3,250,1.5,fixed,250,34,3140.0,6010.0,2870.0,92.352941,290.0,0.588235,250.0,250,"[0.0, 0.0, 0.0, 0.0, 1.0]","[0.0, 0.0, 0.0, 0.0, 10.0]"
2,single_PANEL_4X4_meanrev,single,Best clean single-product signal from correcte...,meanrev,4,250,1.5,fixed,250,35,-27110.0,-24055.0,3055.0,-774.571429,-840.0,0.200000,250.0,250,"[0.0, 0.0, 0.0, 0.0, 1.0]","[0.0, 0.0, 0.0, 0.0, 10.0]"
3,single_PANEL_4X4_meanrev,single,Best clean single-product signal from correcte...,meanrev,2,250,1.5,fixed,500,18,10380.0,11980.0,1600.0,576.666667,535.0,0.666667,500.0,500,"[0.0, 0.0, 0.0, 0.0, 1.0]","[0.0, 0.0, 0.0, 0.0, 10.0]"
4,single_PANEL_4X4_meanrev,single,Best clean single-product signal from correcte...,meanrev,3,250,1.5,fixed,500,18,-5400.0,-3855.0,1545.0,-300.000000,-505.0,0.444444,500.0,500,"[0.0, 0.0, 0.0, 0.0, 1.0]","[0.0, 0.0, 0.0, 0.0, 10.0]"


,candidate,family,mode,day,window,entry_z,exit_z,max_hold,entry_idx,exit_idx,entry_ts,exit_ts,side,hold,exec_pnl,mid_pnl,cost
0,single_PANEL_4X4_meanrev,single,meanrev,2,250,1.5,fixed,250,49,299,4900,29900,-1.0,250,-300.0,-205.0,95.0
1,single_PANEL_4X4_meanrev,single,meanrev,2,250,1.5,fixed,250,309,559,30900,55900,1.0,250,-440.0,-350.0,90.0
2,single_PANEL_4X4_meanrev,single,meanrev,2,250,1.5,fixed,250,560,810,56000,81000,1.0,250,1250.0,1345.0,95.0
3,single_PANEL_4X4_meanrev,single,meanrev,2,250,1.5,fixed,250,844,1094,84400,109400,-1.0,250,580.0,670.0,90.0
4,single_PANEL_4X4_meanrev,single,meanrev,2,250,1.5,fixed,250,1101,1351,110100,135100,1.0,250,1050.0,1140.0,90.0



Top STAGE2_SUMMARY:


,candidate,family,note,mode,window,entry_z,exit_z,max_hold,q_signal,q_trade,...,positive_days,avg_trade_pnl,median_trade_pnl,mean_hit_rate,min_hit_rate,avg_hold,max_hold_seen,pnl_per_trade,pnl_per_cost,one_day_dependency
177,basket_2X2_2X4_4X4_meanrev,positive_basket,Strongest overall corrected basket signal.,meanrev,1000,1.5,fixed,1000,"[0.0, 1.0, 0.0, 1.0, 1.0]","[0.0, 10.0, 0.0, 10.0, 10.0]",...,3,2215.185185,3636.666667,0.666667,0.555556,1000.000000,1000,2215.185185,8.330084,0.534693
178,basket_2X2_2X4_4X4_meanrev,positive_basket,Strongest overall corrected basket signal.,meanrev,1000,1.5,fixed,1500,"[0.0, 1.0, 0.0, 1.0, 1.0]","[0.0, 10.0, 0.0, 10.0, 10.0]",...,3,2888.333333,3813.333333,0.833333,0.666667,1500.000000,1500,2888.333333,10.599388,0.630698
517,basket_2X4_4X4_meanrev,positive_basket,Cleaner two-product large-panel basket.,meanrev,1000,2.0,fixed,1000,"[0.0, 0.0, 0.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 10.0, 10.0]",...,3,2475.238095,1226.666667,0.714286,0.428571,1000.000000,1000,2475.238095,13.396907,0.756252
498,basket_2X4_4X4_meanrev,positive_basket,Cleaner two-product large-panel basket.,meanrev,1000,1.5,fixed,1500,"[0.0, 0.0, 0.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 10.0, 10.0]",...,3,3177.000000,4108.333333,0.744444,0.600000,1500.000000,1500,3233.125000,17.301003,0.472646
816,single_PANEL_1X4_breakout,single,Earlier stateful scan favourite.,breakout,1000,1.5,fixed,500,"[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 10.0, 0.0, 0.0]",...,3,1003.907204,918.333333,0.592186,0.538462,500.000000,500,1032.619048,12.303546,0.604104
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254,basket_2X2_2X4_4X4_meanrev,positive_basket,Strongest overall corrected basket signal.,meanrev,2500,1.5,0.75,2500,"[0.0, 1.0, 0.0, 1.0, 1.0]","[0.0, 10.0, 0.0, 10.0, 10.0]",...,3,558.263889,955.000000,0.789352,0.666667,258.326389,1023,600.930233,2.226626,0.501161
567,basket_2X4_4X4_meanrev,positive_basket,Cleaner two-product large-panel basket.,meanrev,2500,1.5,0.5,1000,"[0.0, 0.0, 0.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 10.0, 10.0]",...,3,648.656566,1185.000000,0.793218,0.727273,333.999134,1000,645.000000,3.512594,0.632171
573,basket_2X4_4X4_meanrev,positive_basket,Cleaner two-product large-panel basket.,meanrev,2500,1.5,0.75,1500,"[0.0, 0.0, 0.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 10.0, 10.0]",...,3,573.843137,995.000000,0.833007,0.750000,237.763725,1110,586.363636,3.165644,0.509690
274,basket_2X2_2X4_4X4_meanrev,positive_basket,Strongest overall corrected basket signal.,meanrev,2500,2.0,0.75,2500,"[0.0, 1.0, 0.0, 1.0, 1.0]","[0.0, 10.0, 0.0, 10.0, 10.0]",...,3,1191.726190,1578.333333,0.849206,0.714286,298.361111,864,1223.809524,4.577026,0.474319



STAGE2_ROBUST:


,candidate,family,note,mode,window,entry_z,exit_z,max_hold,q_signal,q_trade,...,positive_days,avg_trade_pnl,median_trade_pnl,mean_hit_rate,min_hit_rate,avg_hold,max_hold_seen,pnl_per_trade,pnl_per_cost,one_day_dependency
177,basket_2X2_2X4_4X4_meanrev,positive_basket,Strongest overall corrected basket signal.,meanrev,1000,1.5,fixed,1000,"[0.0, 1.0, 0.0, 1.0, 1.0]","[0.0, 10.0, 0.0, 10.0, 10.0]",...,3,2215.185185,3636.666667,0.666667,0.555556,1000.000000,1000,2215.185185,8.330084,0.534693
178,basket_2X2_2X4_4X4_meanrev,positive_basket,Strongest overall corrected basket signal.,meanrev,1000,1.5,fixed,1500,"[0.0, 1.0, 0.0, 1.0, 1.0]","[0.0, 10.0, 0.0, 10.0, 10.0]",...,3,2888.333333,3813.333333,0.833333,0.666667,1500.000000,1500,2888.333333,10.599388,0.630698
517,basket_2X4_4X4_meanrev,positive_basket,Cleaner two-product large-panel basket.,meanrev,1000,2.0,fixed,1000,"[0.0, 0.0, 0.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 10.0, 10.0]",...,3,2475.238095,1226.666667,0.714286,0.428571,1000.000000,1000,2475.238095,13.396907,0.756252
498,basket_2X4_4X4_meanrev,positive_basket,Cleaner two-product large-panel basket.,meanrev,1000,1.5,fixed,1500,"[0.0, 0.0, 0.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 10.0, 10.0]",...,3,3177.000000,4108.333333,0.744444,0.600000,1500.000000,1500,3233.125000,17.301003,0.472646
816,single_PANEL_1X4_breakout,single,Earlier stateful scan favourite.,breakout,1000,1.5,fixed,500,"[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 10.0, 0.0, 0.0]",...,3,1003.907204,918.333333,0.592186,0.538462,500.000000,500,1032.619048,12.303546,0.604104
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
807,single_PANEL_1X4_breakout,single,Earlier stateful scan favourite.,breakout,1000,1.5,0.5,1000,"[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 10.0, 0.0, 0.0]",...,3,507.725932,-508.333333,0.339894,0.294118,337.761757,1000,512.600000,6.009379,0.639485
289,basket_2X2_2X4_4X4_meanrev,positive_basket,Strongest overall corrected basket signal.,meanrev,2500,2.5,0.5,2500,"[0.0, 1.0, 0.0, 1.0, 1.0]","[0.0, 10.0, 0.0, 10.0, 10.0]",...,3,2401.666667,2400.000000,1.000000,1.000000,400.550000,829,2320.000000,8.650847,0.525078
2157,spread_2X2_minus_2X4_meanrev,relative_spread,Simpler 2X2 vs 2X4 relative spread.,meanrev,1000,3.0,fixed,1000,"[0.0, 1.0, 0.0, -1.0, 0.0]","[0.0, 10.0, 0.0, -10.0, 0.0]",...,3,2613.888889,4161.666667,0.805556,0.666667,1000.000000,1000,2550.000000,14.088398,0.368627
564,basket_2X4_4X4_meanrev,positive_basket,Cleaner two-product large-panel basket.,meanrev,2500,1.5,0.25,2500,"[0.0, 0.0, 0.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 10.0, 10.0]",...,3,890.444444,1345.000000,0.788889,0.666667,444.485185,2500,875.172414,4.904348,0.472813



STAGE2_BEST_BY_CANDIDATE:


,candidate,family,note,mode,window,entry_z,exit_z,max_hold,q_signal,q_trade,...,positive_days,avg_trade_pnl,median_trade_pnl,mean_hit_rate,min_hit_rate,avg_hold,max_hold_seen,pnl_per_trade,pnl_per_cost,one_day_dependency
0,basket_2X2_2X4_4X4_meanrev,positive_basket,Strongest overall corrected basket signal.,meanrev,1000,1.5,fixed,1000,"[0.0, 1.0, 0.0, 1.0, 1.0]","[0.0, 10.0, 0.0, 10.0, 10.0]",...,3,2215.185185,3636.666667,0.666667,0.555556,1000.0,1000,2215.185185,8.330084,0.534693
1,basket_2X4_4X4_meanrev,positive_basket,Cleaner two-product large-panel basket.,meanrev,1000,2.0,fixed,1000,"[0.0, 0.0, 0.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 10.0, 10.0]",...,3,2475.238095,1226.666667,0.714286,0.428571,1000.0,1000,2475.238095,13.396907,0.756252
2,single_PANEL_1X4_breakout,single,Earlier stateful scan favourite.,breakout,1000,1.5,fixed,500,"[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 10.0, 0.0, 0.0]",...,3,1003.907204,918.333333,0.592186,0.538462,500.0,500,1032.619048,12.303546,0.604104
3,single_PANEL_2X4_meanrev,single,Earlier predictive scan favourite.,meanrev,2500,2.0,fixed,1000,"[0.0, 0.0, 0.0, 1.0, 0.0]","[0.0, 0.0, 0.0, 10.0, 0.0]",...,3,2116.666667,2523.333333,0.666667,0.500000,1000.0,1000,2356.875000,24.095847,0.646513
4,spread_2X2_minus_2X4_meanrev,relative_spread,Simpler 2X2 vs 2X4 relative spread.,meanrev,1000,2.0,fixed,2500,"[0.0, 1.0, 0.0, -1.0, 0.0]","[0.0, 10.0, 0.0, -10.0, 0.0]",...,3,4107.777778,5776.666667,0.777778,0.666667,2500.0,2500,4107.777778,21.747059,0.546118
5,spread_2x_2X2_minus_2X4_meanrev,geometry_spread,Explainable geometry relation: 2×2X2 vs 2X4.,meanrev,2500,1.5,fixed,1000,"[0.0, 2.0, 0.0, -1.0, 0.0]","[0.0, 10.0, 0.0, -5.0, 0.0]",...,3,1495.208333,2283.333333,0.666667,0.625000,1000.0,1000,1495.208333,11.196568,0.409085
6,spread_1X4_minus_2X4_breakout,relative_spread,Higher-risk continuation spread.,breakout,250,3.0,fixed,2500,"[0.0, 0.0, 1.0, -1.0, 0.0]","[0.0, 0.0, 10.0, -10.0, 0.0]",...,3,3744.444444,3903.333333,0.888889,0.666667,2500.0,2500,3744.444444,20.240240,0.529080
7,single_PANEL_4X4_meanrev,single,Best clean single-product signal from correcte...,meanrev,500,2.5,fixed,500,"[0.0, 0.0, 0.0, 0.0, 1.0]","[0.0, 0.0, 0.0, 0.0, 10.0]",...,3,713.168498,618.333333,0.564103,0.500000,500.0,500,703.414634,8.193182,0.501734



Saved:
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_stage2/STAGE2_DAY_RESULTS.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_stage2/STAGE2_SUMMARY.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_stage2/STAGE2_ROBUST.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_stage2/STAGE2_BEST_BY_CANDIDATE.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_stage2/STAGE2_TRADE_LOG.csv
/Users/artem/Desktop/Prosperity/prosperityGoldmanSnacks/outputs_panels_stage2/STAGE2_panels_report.md

Final runtime: 5.68s
